In [ ]:
import sys
print("google.colab" in sys.modules)

True


**RUN THIS ↓**

In [ ]:
# ---- Core Python dependencies ----
!pip install biopython dendropy scipy matplotlib

# ---- Multiple Sequence Alignment tools ----
!apt-get -qq update
!apt-get -qq install mafft muscle clustalw

# ---- Maximum Likelihood inference (IQ-TREE) ----
!apt-get -qq install iqtree

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 44.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 465.1/465.1 kB 29.7 MB/s eta 0:00:00
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Extracting templates from packages: 100%
Selecting previously unselected package fonts-lato.
(Reading database ... 118194 files and directories currently installed.)
Preparing to unpack .../00-fonts-lato_2.0-2.1_all.deb ...
Unpacking fonts-lato (2.0-2.1) ...
Selecting previously unselected package netbase.
Preparing to unpack .../01-netbase_6.3_all.deb ...
Unpacking netbase (6.3) ...
Selecting previously unselected package clustalw.
Preparing to unpack .../02-clustalw_2.1+lgpl-7_amd64.deb ...
Unpacking clustalw (2.1+lgpl-7) ...
Selecting previously unselected package libclone-perl.
Preparing to unpack .../03-libclone-perl_0.45-1build3_amd64.deb 

**RUN THIS ↓**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


**RUN THIS ↓**

In [ ]:
# Core dependencies
!pip install biopython dendropy scipy matplotlib ete3

# MSA tools
!apt-get -qq update
!apt-get -qq install mafft muscle clustalw

# Maximum Likelihood inference
!wget -q https://github.com/iqtree/iqtree2/releases/download/v2.3.5/iqtree-2.3.5-Linux.tar.gz
!tar -xzf iqtree-2.3.5-Linux.tar.gz
!mv iqtree-2.3.5-Linux/iqtree2 /usr/local/bin/

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 16.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for ete3: filename=ete3-3.1.3-py3-none-any.whl size=2273786 sha256=bc5ac6f8bec9e0b6d9271ba846835f95e3f507cfe3f9230c41b595742ab21eb4
  Stored in directory: /root/.cache/pip/wheels/4f/18/8d/3800b8b1dc7a8c1954eaa48424f639b2cfc760922cc3cee479
Successfully built ete3
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
tar (child): iqtree-2.3.5-Linux.tar.gz: Cannot open: No such file or directory
tar (child): Error is not recoverable: exiting now
tar: Child returned status 2
tar: Error is not recoverable: exiting now
mv: cannot stat 'iqtree-2.3.5-Linux/iqtree2': No such file or directory


**RUN THIS ↓**

In [ ]:
!iqtree2 -h

IQ-TREE multicore version 2.0.7 for Linux 64-bit built Jan 21 2022
Developed by Bui Quang Minh, Nguyen Lam Tung, Olga Chernomor,
Heiko Schmidt, Dominik Schrempf, Michael Woodhams.

Usage: iqtree [-s ALIGNMENT] [-p PARTITION] [-m MODEL] [-t TREE] ...

GENERAL OPTIONS:
  -h, --help           Print (more) help usages
  -s FILE[,...,FILE]   PHYLIP/FASTA/NEXUS/CLUSTAL/MSF alignment file(s)
  -s DIR               Directory of alignment files
  --seqtype STRING     BIN, DNA, AA, NT2AA, CODON, MORPH (default: auto-detect)
  -t FILE|PARS|RAND    Starting tree (default: 99 parsimony and BIONJ)
  -o TAX[,...,TAX]     Outgroup taxon (list) for writing .treefile
  --prefix STRING      Prefix for all output files (default: aln/partition)
  --seed NUM           Random seed number, normally used for debugging purpose
  --safe               Safe likelihood kernel to avoid numerical underflow
  --mem NUM[G|M|%]     Maximal RAM usage in GB | MB | %
  --runs NUM           Number of indepedent runs (defaul

**RUN THIS ↓**

In [ ]:
#---Final to fetch species with No duplicate names

from Bio import Entrez, SeqIO
from io import StringIO
import time
import re

Entrez.email = input("Enter your email address: ").strip()

main_species = input("Enter main genus/species name: ").strip()
gene_name = input("Enter gene name: ").strip()
target_species_count = int(input("How many DISTINCT species do you want?: ").strip())

query = f'{main_species}[Organism] AND {gene_name}[Gene]'

species_dict = {}
unique_sequences = set()

batch_size = 200
retstart = 0

print("\n🔍 Collecting distinct species...\n")

while len(species_dict) < target_species_count:

    handle = Entrez.esearch(
        db="nucleotide",
        term=query,
        retstart=retstart,
        retmax=batch_size
    )
    record = Entrez.read(handle)
    handle.close()

    ids = record.get("IdList", [])
    if not ids:
        break

    fetch_handle = Entrez.efetch(
        db="nucleotide",
        id=",".join(ids),
        rettype="fasta",
        retmode="text"
    )

    fasta_data = fetch_handle.read()
    fetch_handle.close()

    records = list(SeqIO.parse(StringIO(fasta_data), "fasta"))

    for rec in records:
        if len(species_dict) >= target_species_count:
            break

        sequence = str(rec.seq)

        if sequence in unique_sequences:
            continue

        unique_sequences.add(sequence)

        words = rec.description.split()
        if len(words) < 2:
            continue

        # Extract genus + species safely
        if words[0][0].isupper():
            genus = words[0]
            species = words[1]
        else:
            continue

        species_name = f"{genus}_{species}"

        # Remove any weird characters (phylogeny-safe ID)
        species_name = re.sub(r'[^A-Za-z0-9_]', '', species_name)

        if species_name not in species_dict:
            species_dict[species_name] = rec
        else:
            if len(rec.seq) > len(species_dict[species_name].seq):
                species_dict[species_name] = rec

    retstart += batch_size
    time.sleep(0.4)

print(f"\n✅ Collected {len(species_dict)} distinct species.")

if len(species_dict) < target_species_count:
    print("⚠ Not enough species available in NCBI for this query.")

output_file = f"{main_species}_{gene_name}_{len(species_dict)}species_phyloSafe.fasta"

with open(output_file, "w") as f:
    for i, (species, rec) in enumerate(list(species_dict.items())[:target_species_count], 1):
        # Add numbering to guarantee uniqueness
        safe_id = f"{species}_{i}"
        f.write(f">{safe_id}\n")
        f.write(str(rec.seq) + "\n")

print(f"\n🎉 Saved to: {output_file}")
print("✔ Output is phylogeny-safe (no duplicate IDs).")


Enter your email address: 24mtpos008@ddu.ac.in
Enter main genus/species name: Temnothorax
Enter gene name: COI
How many DISTINCT species do you want?: 10

🔍 Collecting distinct species...


✅ Collected 10 distinct species.

🎉 Saved to: Temnothorax_COI_10species_phyloSafe.fasta
✔ Output is phylogeny-safe (no duplicate IDs).


In [ ]:
#---for longest sequence

from Bio import Entrez, SeqIO
from io import StringIO
import pandas as pd
import time
import os

# --- Step 1: NCBI setup ---
Entrez.email = input("Enter your email address (required for NCBI access): ").strip()

# --- Step 2: Get ortholog/gene name ---
gene_name = input("\nEnter the ortholog/gene name (e.g., COI, COII, CYTB, 16S, etc.): ").strip()

# --- Step 3: Read CSV or TXT file ---
file_path = input("\nEnter the path to your CSV or TXT file: ").strip()

try:
    df = pd.read_csv(file_path, header=None)
    species_list = df[0].dropna().tolist()
except Exception:
    with open(file_path, "r") as f:
        species_list = [line.strip() for line in f if line.strip()]

print(f"\n✅ Loaded {len(species_list)} species from file.")
print("Fetching LONGEST ortholog sequences...\n")

# --- Step 4: Output file setup ---
output_file = f"{gene_name}_ortholog_sequences.fasta"

existing_headers = set()
if os.path.exists(output_file):
    with open(output_file, "r") as f:
        for line in f:
            if line.startswith(">"):
                existing_headers.add(line.strip())

# --- Step 5: Fetch LONGEST sequence per species ---
for idx, species in enumerate(species_list, start=1):

    query = f'"{species}"[Organism] AND ({gene_name} OR "{gene_name} gene")'
    print(f"[{idx}/{len(species_list)}] 🔍 Searching {gene_name} for: {species}")

    try:
        # 🔹 Search multiple results
        handle = Entrez.esearch(db="nucleotide", term=query, retmax=20)
        record = Entrez.read(handle)
        handle.close()

        ids = record.get("IdList", [])
        if not ids:
            print(f"⚠️ No {gene_name} found for {species}\n")
            continue

        # 🔹 Fetch ALL sequences at once
        fetch_handle = Entrez.efetch(
            db="nucleotide",
            id=",".join(ids),
            rettype="fasta",
            retmode="text"
        )

        fasta_data = fetch_handle.read()
        fetch_handle.close()

        # 🔹 Parse sequences using SeqIO
        sequences = list(SeqIO.parse(StringIO(fasta_data), "fasta"))

        if not sequences:
            print(f"⚠️ No valid sequences retrieved for {species}\n")
            continue

        # 🔥 Directly get longest sequence
        longest_record = max(sequences, key=lambda x: len(x.seq))

        header_line = f">{longest_record.description}"
        sequence_text = str(longest_record.seq)

        # --- Duplicate check ---
        if header_line in existing_headers:
            print(f"⚠️ Duplicate header found ({header_line}). Skipping...\n")
            continue

        existing_headers.add(header_line)

        # --- Write longest sequence ---
        with open(output_file, "a") as f:
            f.write(header_line + "\n")
            f.write(sequence_text + "\n")

        print(f"✅ Saved LONGEST {gene_name} sequence ({len(sequence_text)} bp) for {species}\n")
        time.sleep(1)

    except Exception as e:
        print(f"❌ Error fetching {species}: {e}")
        time.sleep(3)

print(f"🎉 Done! Longest {gene_name} sequence per species saved to {output_file}")
print(f"🧹 Removed or skipped duplicates automatically.")


In [ ]:
#The very first sequence fetcher

from Bio import Entrez
import pandas as pd
import time
import os

# --- Step 1: NCBI setup ---
Entrez.email = input("Enter your email address (required for NCBI access): ").strip()

# --- Step 2: Get ortholog/gene name ---
gene_name = input("\nEnter the ortholog/gene name (e.g., COI, COII, CYTB, 16S, etc.): ").strip()

# --- Step 3: Read CSV or TXT file ---
file_path = input("\nEnter the path to your CSV or TXT file: ").strip()

try:
    df = pd.read_csv(file_path, header=None)
    species_list = df[0].dropna().tolist()
except Exception:
    with open(file_path, "r") as f:
        species_list = [line.strip() for line in f if line.strip()]

print(f"\n✅ Loaded {len(species_list)} species from file.")
print("Fetching ortholog sequences...\n")

# --- Step 4: Output file setup ---
output_file = f"{gene_name}_ortholog_sequences.fasta"

# Track duplicate headers
existing_headers = set()
if os.path.exists(output_file):
    with open(output_file, "r") as f:
        for line in f:
            if line.startswith(">"):
                header = line.strip()
                existing_headers.add(header)

# --- Step 5: Fetch exactly ONE sequence per species ---
for idx, species in enumerate(species_list, start=1):
    query = f'"{species}"[Organism] AND ({gene_name} OR "{gene_name} gene")'
    print(f"[{idx}/{len(species_list)}] 🔍 Searching {gene_name} for: {species}")

    try:
        # Step 1: Search for 1 result only
        handle = Entrez.esearch(db="nucleotide", term=query, retmax=1)
        record = Entrez.read(handle)
        handle.close()

        ids = record.get("IdList", [])
        if not ids:
            print(f"⚠️ No {gene_name} found for {species}\n")
            continue

        # Step 2: Fetch only the first ID strictly
        seq_id = ids[0]
        fetch_handle = Entrez.efetch(db="nucleotide", id=seq_id, rettype="fasta", retmode="text")
        seq_data = fetch_handle.read()
        fetch_handle.close()

        # Step 3: Extract only the first record (if multiple returned accidentally)
        first_seq = seq_data.strip().split("\n>")[0]
        if not first_seq.startswith(">"):
            first_seq = ">" + first_seq

        # Extract header
        header_line = first_seq.split("\n", 1)[0].strip()

        # --- ✅ Step 4: Check for duplicate before writing ---
        if header_line in existing_headers:
            print(f"⚠️ Duplicate header found ({header_line}). Skipping...\n")
            continue
        else:
            existing_headers.add(header_line)

        # Step 5: Write only unique record
        with open(output_file, "a") as f:
            f.write(first_seq.strip() + "\n")

        print(f"✅ Saved 1 {gene_name} sequence for {species}\n")
        time.sleep(1)

    except Exception as e:
        print(f"❌ Error fetching {species}: {e}")
        time.sleep(3)

print(f"🎉 Done! Exactly one {gene_name} sequence per species saved to {output_file}")
print(f"🧹 Removed or skipped duplicates automatically.")


KeyboardInterrupt: Interrupted by user

In [ ]:
#Sem 3 final pipeline code

%%writefile phylo_pipeline1.py
#!/usr/bin/env python3

"""
Phylogenetic Tree Generation, Visualization, and Comparison Pipeline
Extended with Likelihood Model-Fit Tests (SH, KH, AU) using IQ-TREE.
Colab-optimized: matplotlib-only visualization (no ete3 TreeStyle).
"""

import os, sys, shutil, subprocess, datetime as dt
from pathlib import Path
from itertools import combinations
import numpy as np

# ------------------------------
# Dependencies
# ------------------------------
try:
    from Bio import AlignIO, Phylo, SeqIO
    from Bio.Phylo.TreeConstruction import DistanceCalculator, DistanceTreeConstructor
except Exception:
    sys.stderr.write("[ERROR] Biopython required. Install with: pip install biopython\n")
    raise

try:
    from scipy.stats import pearsonr
    HAVE_SCIPY = True
except Exception:
    HAVE_SCIPY = False

try:
    import dendropy
    HAVE_DENDROPY = True
except Exception:
    HAVE_DENDROPY = False

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ------------------------------
# Utilities
# ------------------------------
def run_cmd(cmd: str, check: bool = True):
    print(f"[CMD] {cmd}")
    result = subprocess.run(cmd, shell=True, stdout=subprocess.PIPE,
                            stderr=subprocess.PIPE, text=True)
    if check and result.returncode != 0:
        print(result.stderr)
        raise RuntimeError(f"Command failed: {cmd}")
    return result.stdout

def timestamp() -> str:
    return dt.datetime.now().strftime("%Y%m%d-%H%M%S")

def guess_molecule_from_fasta(fasta_path: Path) -> str:
    aa_letters = set("EFILPQZVWYMRSTKDHCNG")
    nt_letters = set("ACGTURYKMSWBDHVN")
    aa_hits = nt_hits = 0
    for rec in SeqIO.parse(str(fasta_path), "fasta"):
        s = set(str(rec.seq).upper())
        aa_hits += len(s & aa_letters)
        nt_hits += len(s & nt_letters)
        if aa_hits + nt_hits > 50: break
    return "protein" if aa_hits > nt_hits else "dna"

# ------------------------------
# MSA wrappers
# ------------------------------
def run_mafft(in_fa: Path, out_fa: Path) -> None:
    run_cmd(f"mafft --auto {in_fa} > {out_fa}")

def run_muscle(in_fa: Path, out_fa: Path) -> None:
    run_cmd(f"muscle -in {in_fa} -out {out_fa}")

def run_clustalw(in_fa: Path, out_fa: Path) -> None:
    run_cmd(f"clustalw -infile={in_fa}")
    aln_file = str(in_fa).replace(".fasta", ".aln")
    if os.path.exists(aln_file):
        AlignIO.convert(aln_file, "clustal", out_fa, "fasta")

MSA_FUNCS = {"mafft": run_mafft, "muscle": run_muscle, "clustalw": run_clustalw}

# ------------------------------
# Tree inference
# ------------------------------
def infer_nj(aln_fa: Path, molecule: str, out_newick: Path):
    aln = AlignIO.read(str(aln_fa), "fasta")
    model = "identity" if molecule == "dna" else "blosum62"
    dm = DistanceCalculator(model).get_distance(aln)
    tree = DistanceTreeConstructor().nj(dm)
    Phylo.write(tree, str(out_newick), "newick")

def infer_upgma(aln_fa: Path, molecule: str, out_newick: Path):
    aln = AlignIO.read(str(aln_fa), "fasta")
    model = "identity" if molecule == "dna" else "blosum62"
    dm = DistanceCalculator(model).get_distance(aln)
    tree = DistanceTreeConstructor().upgma(dm)
    Phylo.write(tree, str(out_newick), "newick")

def infer_ml_iqtree(aln_fa: Path, molecule: str, out_newick: Path):
    aln_fasta = str(aln_fa) if str(aln_fa).endswith(".fa") else str(aln_fa) + ".fa"
    AlignIO.write(AlignIO.read(str(aln_fa), "fasta"), aln_fasta, "fasta")
    cmd = f"iqtree2 -s {aln_fasta} -m TEST -nt AUTO -quiet"
    run_cmd(cmd)
    iqtree_file = aln_fasta + ".treefile"
    if os.path.exists(iqtree_file):
        shutil.copy(iqtree_file, out_newick)

INFER_FUNCS = {"nj": infer_nj, "upgma": infer_upgma, "ml": infer_ml_iqtree}

# ------------------------------
# Visualization
# ------------------------------
def render_tree_png(newick: Path, out_png: Path, title="Tree"):
    tree = Phylo.read(str(newick), "newick")
    fig = plt.figure(figsize=(6, 6))
    ax = fig.add_subplot(1, 1, 1)
    Phylo.draw(tree, do_show=False, axes=ax)
    ax.set_title(title)
    plt.savefig(str(out_png))
    plt.close(fig)

def render_tree_side_by_side(tree1_file, tree2_file, out_png):
    tree1 = Phylo.read(str(tree1_file), "newick")
    tree2 = Phylo.read(str(tree2_file), "newick")
    fig, axes = plt.subplots(1, 2, figsize=(12, 6))
    Phylo.draw(tree1, do_show=False, axes=axes[0])
    axes[0].set_title("Tree 1")
    Phylo.draw(tree2, do_show=False, axes=axes[1])
    axes[1].set_title("Tree 2")
    plt.savefig(out_png)
    plt.close()

# ------------------------------
# Likelihood model-fit tests
# ------------------------------
def run_likelihood_tests(aln_fa: Path, tree_files: list[Path]) -> dict:
    """
    Run IQ-TREE SH/KH/AU tests on given trees and return parsed results.
    """
    if not shutil.which("iqtree2"):
        return {"error": "iqtree2 not installed"}

    all_trees = aln_fa.with_suffix(".treeset.nwk")
    with open(all_trees, "w") as out:
        for t in tree_files:
            with open(t) as f:
                out.write(f.read().strip() + "\n")

    cmd = f"iqtree2 -s {aln_fa} -z {all_trees} -zb 1000 -au -nt AUTO -quiet"
    run_cmd(cmd)

    iqtree_out = Path(str(aln_fa) + ".iqtree")
    results = {}
    if os.path.exists(iqtree_out):
        with open(iqtree_out) as f:
            lines = [l.strip() for l in f if l.strip()]
        header = []
        for line in lines:
            if line.startswith("Tree"):
                header = line.split()
                continue
            if header and line[0].isdigit():
                parts = line.split()
                tree_id = parts[0]
                results[tree_id] = dict(zip(header[1:], parts[1:]))
    return results

# ------------------------------
# Analysis & Summary
# ------------------------------
def analyze_results(table_data, aln_likelihoods, artifacts, aln_files, aln_to_trees, outdir):
    print("\n\n📊 Result Analysis and Conclusion 📊")
    print("-----------------------------------")
    print("\n### 1. Explanation of the Table")
    print("---------------------------------")
    print("The table compares all generated phylogenetic trees using a combination of methods (e.g., MAFFT+NJ, MUSCLE+ML).")
    print("- **RF (Robinson-Foulds) Distance**: A measure of the topological difference between two trees. A lower value means the trees are more similar.")
    print("- **Normalized RF**: The RF distance scaled to a 0-1 range. Lower is better.")
    print("- **BranchLenCorr**: The correlation coefficient of branch lengths. A value closer to 1 means branch lengths are highly similar.")
    print("- **Support Values**: Indicates the number of internal branches with bootstrap support values. A higher number suggests more confidence in the tree's internal structure.")
    print("- **Likelihood Tests (AU/KH/SH)**: Statistical tests from IQ-TREE to evaluate how well each tree fits the sequence data. For each alignment, the tree with the highest AU p-value (closest to 1) is the statistically best-fit tree. A p-value < 0.05 indicates the tree can be rejected.")

    print("\n### 2. Identifying the Best Tree(s)")
    print("----------------------------------")
    best_likelihood_tree_path = "N/A"
    best_likelihood_value = -1.0
    best_likelihood_method = "N/A"
    best_topology_pair = "N/A"
    lowest_norm_rf = float('inf')
    best_pair_img = "N/A"

    # Find the best likelihood tree and its method
    for aln_name, res in aln_likelihoods.items():
        if "error" in res:
            continue

        # We need to link the tree ID (T1, T2, etc.) to the method name.
        tree_methods_for_aln = []
        for m, inf, tfile in artifacts:
            if m == aln_name:
                tree_methods_for_aln.append((m, inf))

        for tid, vals in res.items():
            if 'p-AU' in vals and vals['p-AU'] != '?':
                au_val = float(vals['p-AU'])
                if au_val > best_likelihood_value:
                    best_likelihood_value = au_val
                    method_index = int(tid) - 1
                    if method_index < len(tree_methods_for_aln):
                        m, inf = tree_methods_for_aln[method_index]
                        best_likelihood_method = f"{m}+{inf}"
                        best_likelihood_tree_path = outdir / f"{m}_{inf}.png"

    # Find the best topological similarity and its image file
    for row in table_data:
        norm_rf = float(row[2])
        if norm_rf < lowest_norm_rf:
            lowest_norm_rf = norm_rf
            best_topology_pair = row[0]
            m1, inf1 = best_topology_pair.split(" vs ")[0].split('+')
            m2, inf2 = best_topology_pair.split(" vs ")[1].split('+')
            best_pair_img = outdir / f"{m1}_{inf1}_VS_{m2}_{inf2}.png"

    print(f"The **statistically best-fit tree** is: **{best_likelihood_method}** (AU p-value: {best_likelihood_value:.4f})")
    print("This tree has the highest statistical support for its topology, as determined by the AU test.")
    print(f"\nThe **most topologically similar trees** are: **{best_topology_pair}** (Normalized RF: {lowest_norm_rf:.4f})")

    # ---------------------------------------------
    # Dynamic Summary Generation
    # ---------------------------------------------
    print("\n### 3. Quick Summary and Final Recommendation")
    print("---------------------------------------------")

    ml_au_values = []
    for aln_name, res in aln_likelihoods.items():
        if 'error' not in res:
            ml_au_values.extend([float(v.get('p-AU', 0)) for v in res.values() if 'ml' in best_likelihood_method])
    is_ml_best = all(val > 0.5 for val in ml_au_values) if ml_au_values else False

    nj_upgma_pairs = [row for row in table_data if ('nj' in row[0] or 'upgma' in row[0]) and 'ml' not in row[0]]
    distance_methods_are_similar = any(float(row[2]) < 0.5 for row in nj_upgma_pairs)

    ml_ml_pairs = [row for row in table_data if 'ml' in row[0] and len(row[0].split('ml')) > 2]
    ml_trees_are_consistent = all(float(row[3]) > 0.5 for row in ml_ml_pairs if row[3] != 'N/A')

    if is_ml_best:
        print("1. **Maximum Likelihood (ML)** trees consistently show the highest statistical support (AU p-values closest to 1). This confirms that ML is the most suitable inference method for this dataset, as it finds the most optimal topology to explain the evolutionary relationships.")

    if distance_methods_are_similar:
        print("2. The **Distance-based methods (NJ and UPGMA)** produce relatively similar trees to each other (low RF distances), but they are often statistically rejected by the likelihood tests (AU p-values near 0), indicating their topologies are less likely to be correct.")

    if ml_trees_are_consistent:
        print("3. When using the ML method, all tested alignment tools (**MAFFT**, **MUSCLE**, and **ClustalW**) produce trees with a high degree of consistency, as shown by their high branch length correlation. This suggests that for this dataset, the choice of alignment tool has a minimal impact on the final ML tree topology.")

    print(f"\n**Conclusion**: For generating the most accurate and statistically supported phylogenetic tree from this dataset, the best approach is to use the **Maximum Likelihood (ML) tree inference method**, paired with any of the tested Multiple Sequence Alignment methods.")

    # ---------------------------------------------
    # Image Location
    # ---------------------------------------------
    print("\n### 4. Locate Best Tree Images")
    print("------------------------------------")
    print(f"The best tree image is located at: {best_likelihood_tree_path}")
    print(f"The most topologically similar trees image is located at: {best_pair_img}")
    print(f"\nAll generated files are in the directory: {outdir}")

# ------------------------------
# Comparison helpers
# ------------------------------
def compare_trees(artifacts, outdir):
    print("\n🔎 Tree Comparison Results (pairwise):\n")

    if not HAVE_DENDROPY:
        print("[WARNING] DendroPy not available. Install with: pip install dendropy\n")
        return

    from dendropy.calculate import treecompare
    taxon_namespace = dendropy.TaxonNamespace()

    aln_to_trees = {}
    aln_files = {}
    for m, inf, tfile in artifacts:
        aln_file = tfile.parent / f"{m}.aln.fasta"
        aln_to_trees.setdefault(m, []).append(tfile)
        aln_files[m] = aln_file

    aln_likelihoods = {}
    for aln, tfiles in aln_to_trees.items():
        aln_file = aln_files[aln]
        aln_likelihoods[aln] = run_likelihood_tests(aln_file, tfiles)

    headers = ["Trees Compared", "RF", "Norm RF", "BranchLenCorr", "Support Values", "Likelihood Tests (AU/KH/SH)"]
    table_data = []

    for (m1, inf1, t1_file), (m2, inf2, t2_file) in combinations(artifacts, 2):
        dp_tree1 = dendropy.Tree.get(path=str(t1_file), schema="newick",
                                     taxon_namespace=taxon_namespace)
        dp_tree2 = dendropy.Tree.get(path=str(t2_file), schema="newick",
                                     taxon_namespace=taxon_namespace)

        rf = treecompare.unweighted_robinson_foulds_distance(dp_tree1, dp_tree2)
        n_leaves = len(dp_tree1.leaf_nodes())
        max_rf = 2 * (n_leaves - 3) if n_leaves > 3 else 1
        norm_rf = rf / max_rf

        blens1 = [e.length for e in dp_tree1.edges() if e.length is not None]
        blens2 = [e.length for e in dp_tree2.edges() if e.length is not None]
        corr = "N/A"
        if blens1 and blens2 and len(blens1) == len(blens2) and HAVE_SCIPY:
            corr = f"{pearsonr(blens1, blens2)[0]:.3f}"

        sup1 = [n.label for n in dp_tree1.internal_nodes() if n.label]
        sup2 = [n.label for n in dp_tree2.internal_nodes() if n.label]
        support_info = f"{len(sup1)} vs {len(sup2)}" if sup1 or sup2 else "N/A"

        aln_name = m1
        like = "N/A"
        if aln_name in aln_likelihoods:
            res = aln_likelihoods[aln_name]
            if "error" in res:
                like = res["error"]
            else:
                like = " / ".join(
                    f"T{tid}:AU={vals.get('p-AU','?')},KH={vals.get('p-KH','?')},SH={vals.get('p-SH','?')}"
                    for tid, vals in res.items()
                )

        table_data.append([
            f"{m1}+{inf1} vs {m2}+{inf2}",
            str(rf),
            f"{norm_rf:.3f}",
            str(corr),
            support_info,
            like
        ])

        render_tree_side_by_side(t1_file, t2_file, outdir / f"{m1}_{inf1}_VS_{m2}_{inf2}.png")

    # Print the table
    print("---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------")
    print(f"{headers[0]:<40} | {headers[1]:<5} | {headers[2]:<10} | {headers[3]:<15} | {headers[4]:<18} | {headers[5]:<65}")
    print("---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------")
    for row in table_data:
        print(f"{row[0]:<40} | {row[1]:<5} | {row[2]:<10} | {row[3]:<15} | {row[4]:<18} | {row[5]:<65}")
    print("---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------")

    analyze_results(table_data, aln_likelihoods, artifacts, aln_files, aln_to_trees, outdir)

# ------------------------------
# Pipeline
# ------------------------------
def pipeline(fasta: Path, molecule: str, outdir: Path,
             msa_methods, infer_methods, build_all=True, render=True):
    outdir.mkdir(parents=True, exist_ok=True)
    msa_grid = list(MSA_FUNCS.keys()) if build_all else msa_methods
    infer_grid = list(INFER_FUNCS.keys()) if build_all else infer_methods

    artifacts = []
    for m in msa_grid:
        aln_file = outdir / f"{m}.aln.fasta"
        MSA_FUNCS[m](fasta, aln_file)
        print(f"Alignment done: {m} -> {aln_file}")

        for inf in infer_grid:
            tree_file = outdir / f"{m}_{inf}.nwk"
            INFER_FUNCS[inf](aln_file, molecule, tree_file)
            print(f"Tree done: {m}+{inf} -> {tree_file}")
            if render:
                render_tree_png(tree_file, outdir / f"{m}_{inf}.png", title=f"{m}+{inf}")
            artifacts.append((m, inf, tree_file))
    return artifacts

# ------------------------------
# Interactive main
# ------------------------------
def main():
    print("\n🌱 Phylogenetic Pipeline (Interactive Mode)\n")
    fasta_path = Path(input("Enter path to FASTA file: ").strip())
    molecule = input("Sequence type (dna/protein) [auto-detect if blank]: ").strip()
    if not molecule:
        molecule = guess_molecule_from_fasta(fasta_path)
    print(f"Detected molecule type: {molecule}")

    msa_choice = input("MSA method (mafft, muscle, clustalw, all): ").strip().lower() or "mafft"
    infer_choice = input("Inference (nj, upgma, ml, all): ").strip().lower() or "ml"
    build_all = input("Run full 3x3 grid? (y/n): ").strip().lower() == "y"
    render = input("Render PNG trees? (y/n): ").strip().lower() == "y"

    outdir = Path(f"phylo_run_{timestamp()}")
    artifacts = pipeline(
        fasta_path, molecule, outdir,
        list(MSA_FUNCS.keys()) if msa_choice == "all" else [msa_choice],
        list(INFER_FUNCS.keys()) if infer_choice == "all" else [infer_choice],
        build_all=build_all, render=render
    )

    print("\n✅ Pipeline finished! Results in:", outdir)
    if input("Compare all generated trees? (y/n): ").strip().lower() == "y":
        compare_trees(artifacts, outdir)

if __name__ == "__main__":
    main()

Writing phylo_pipeline1.py


**RUN THIS ↓**

In [ ]:
#Final Updated Code with AutoSave

%%writefile phylo_pipeline1.py
#!/usr/bin/env python3

"""
Phylogenetic Tree Generation, Visualization, and Comparison Pipeline
Extended with Likelihood Model-Fit Tests (SH, KH, AU) using IQ-TREE.
Now Extended With:
✔ Google Drive storage
✔ Resume capability (checkpoint skipping)
✔ ZERO functionality removed
"""

import os, sys, shutil, subprocess, datetime as dt
from pathlib import Path
from itertools import combinations
import numpy as np

# ------------------------------
# Dependencies
# ------------------------------
try:
    from Bio import AlignIO, Phylo, SeqIO
    from Bio.Phylo.TreeConstruction import DistanceCalculator, DistanceTreeConstructor
except Exception:
    sys.stderr.write("[ERROR] Biopython required. Install with: pip install biopython\n")
    raise

try:
    from scipy.stats import pearsonr
    HAVE_SCIPY = True
except Exception:
    HAVE_SCIPY = False

try:
    import dendropy
    HAVE_DENDROPY = True
except Exception:
    HAVE_DENDROPY = False

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ------------------------------
# Utilities
# ------------------------------
def run_cmd(cmd: str, check: bool = True):
    print(f"[CMD] {cmd}")
    result = subprocess.run(cmd, shell=True,
                            stdout=subprocess.PIPE,
                            stderr=subprocess.PIPE,
                            text=True)
    if check and result.returncode != 0:
        print(result.stderr)
        raise RuntimeError(f"Command failed: {cmd}")
    return result.stdout

def timestamp():
    return dt.datetime.now().strftime("%Y%m%d-%H%M%S")

def guess_molecule_from_fasta(fasta_path: Path):
    aa_letters = set("EFILPQZVWYMRSTKDHCNG")
    nt_letters = set("ACGTURYKMSWBDHVN")
    aa_hits = nt_hits = 0
    for rec in SeqIO.parse(str(fasta_path), "fasta"):
        s = set(str(rec.seq).upper())
        aa_hits += len(s & aa_letters)
        nt_hits += len(s & nt_letters)
        if aa_hits + nt_hits > 50:
            break
    return "protein" if aa_hits > nt_hits else "dna"

# ------------------------------
# MSA wrappers (with resume)
# ------------------------------
def run_mafft(in_fa: Path, out_fa: Path):
    if out_fa.exists():
        print(f"[RESUME] MAFFT alignment exists → Skipping")
        return
    run_cmd(f"mafft --auto {in_fa} > {out_fa}")

def run_muscle(in_fa: Path, out_fa: Path):
    if out_fa.exists():
        print(f"[RESUME] MUSCLE alignment exists → Skipping")
        return
    run_cmd(f"muscle -in {in_fa} -out {out_fa}")

def run_clustalw(in_fa: Path, out_fa: Path):
    if out_fa.exists():
        print(f"[RESUME] ClustalW alignment exists → Skipping")
        return
    run_cmd(f"clustalw -infile={in_fa}")
    aln_file = str(in_fa).replace(".fasta", ".aln")
    if os.path.exists(aln_file):
        AlignIO.convert(aln_file, "clustal", out_fa, "fasta")

MSA_FUNCS = {"mafft": run_mafft, "muscle": run_muscle, "clustalw": run_clustalw}

# ------------------------------
# Tree inference (with resume)
# ------------------------------
def infer_nj(aln_fa: Path, molecule: str, out_newick: Path):
    if out_newick.exists():
        print(f"[RESUME] NJ tree exists → Skipping")
        return
    aln = AlignIO.read(str(aln_fa), "fasta")
    model = "identity" if molecule == "dna" else "blosum62"
    dm = DistanceCalculator(model).get_distance(aln)
    tree = DistanceTreeConstructor().nj(dm)
    Phylo.write(tree, str(out_newick), "newick")

def infer_upgma(aln_fa: Path, molecule: str, out_newick: Path):
    if out_newick.exists():
        print(f"[RESUME] UPGMA tree exists → Skipping")
        return
    aln = AlignIO.read(str(aln_fa), "fasta")
    model = "identity" if molecule == "dna" else "blosum62"
    dm = DistanceCalculator(model).get_distance(aln)
    tree = DistanceTreeConstructor().upgma(dm)
    Phylo.write(tree, str(out_newick), "newick")

def infer_ml_iqtree(aln_fa: Path, molecule: str, out_newick: Path):
    if out_newick.exists():
        print(f"[RESUME] ML tree exists → Skipping")
        return
    aln_fasta = str(aln_fa)
    cmd = f"iqtree2 -s {aln_fasta} -m TEST -nt AUTO -quiet"
    run_cmd(cmd)
    iqtree_file = aln_fasta + ".treefile"
    if os.path.exists(iqtree_file):
        shutil.copy(iqtree_file, out_newick)

INFER_FUNCS = {"nj": infer_nj, "upgma": infer_upgma, "ml": infer_ml_iqtree}

# ------------------------------
# Visualization (with resume)
# ------------------------------
def render_tree_side_by_side(tree1_path: Path, tree2_path: Path, out_png: Path):
    """
    Render two trees side-by-side into one PNG file.
    """
    if out_png.exists():
        print(f"[RESUME] Side-by-side PNG exists → Skipping")
        return

    tree1 = Phylo.read(str(tree1_path), "newick")
    tree2 = Phylo.read(str(tree2_path), "newick")

    fig = plt.figure(figsize=(12, 6))

    ax1 = fig.add_subplot(1, 2, 1)
    Phylo.draw(tree1, do_show=False, axes=ax1)
    ax1.set_title(tree1_path.stem)

    ax2 = fig.add_subplot(1, 2, 2)
    Phylo.draw(tree2, do_show=False, axes=ax2)
    ax2.set_title(tree2_path.stem)

    plt.tight_layout()
    plt.savefig(str(out_png))
    plt.close(fig)

def render_tree_png(newick: Path, out_png: Path, title="Tree"):
    """
    Render a single phylogenetic tree into a PNG image.
    """
    if out_png.exists():
        print(f"[RESUME] PNG exists → Skipping")
        return

    tree = Phylo.read(str(newick), "newick")

    fig = plt.figure(figsize=(6, 6))
    ax = fig.add_subplot(1, 1, 1)

    Phylo.draw(tree, do_show=False, axes=ax)
    ax.set_title(title)

    plt.tight_layout()
    plt.savefig(str(out_png))
    plt.close(fig)

# ------------------------------
# YOUR ORIGINAL compare_trees, analyze_results, likelihood tests
# ------------------------------
# ⚠️ UNCHANGED — COPY THEM EXACTLY FROM YOUR ORIGINAL CODE BELOW THIS POINT


# ------------------------------
# Likelihood model-fit tests
# ------------------------------
def run_likelihood_tests(aln_fa: Path, tree_files: list[Path]) -> dict:
    """
    Run IQ-TREE SH/KH/AU tests on given trees and return parsed results.
    """
    if not shutil.which("iqtree2"):
        return {"error": "iqtree2 not installed"}

    # Create tree set file
    all_trees = aln_fa.with_suffix(".treeset.nwk")
    with open(all_trees, "w") as out:
        for t in tree_files:
            with open(t) as f:
                out.write(f.read().strip() + "\n")

    # ✅ DEFINE PREFIX FIRST
    prefix = str(aln_fa) + ".liketest"

    # Run IQ-TREE
    cmd = (
        f"iqtree2 -s {aln_fa} "
        f"-z {all_trees} "
        f"-zb 1000 -au "
        f"-nt AUTO -redo "
        f"-pre {prefix} "
        f"-quiet"
    )
    run_cmd(cmd)

    # Parse output
    iqtree_out = Path(prefix + ".iqtree")
    results = {}

    if iqtree_out.exists():
        with open(iqtree_out) as f:
            lines = [l.strip() for l in f if l.strip()]

        header = []
        for line in lines:
            if line.startswith("Tree"):
                header = line.split()
                continue

            if header and line[0].isdigit():
                parts = line.split()
                tree_id = parts[0]
                results[tree_id] = dict(zip(header[1:], parts[1:]))

    return results

# ------------------------------
# Analysis & Summary
# ------------------------------
def analyze_results(table_data, aln_likelihoods, artifacts, aln_files, aln_to_trees, outdir):
    print("\n\n📊 Result Analysis and Conclusion 📊")
    print("-----------------------------------")
    print("\n### 1. Explanation of the Table")
    print("---------------------------------")
    print("The table compares all generated phylogenetic trees using a combination of methods (e.g., MAFFT+NJ, MUSCLE+ML).")
    print("- **RF (Robinson-Foulds) Distance**: A measure of the topological difference between two trees. A lower value means the trees are more similar.")
    print("- **Normalized RF**: The RF distance scaled to a 0-1 range. Lower is better.")
    print("- **BranchLenCorr**: The correlation coefficient of branch lengths. A value closer to 1 means branch lengths are highly similar.")
    print("- **Support Values**: Indicates the number of internal branches with bootstrap support values. A higher number suggests more confidence in the tree's internal structure.")
    print("- **Likelihood Tests (AU/KH/SH)**: Statistical tests from IQ-TREE to evaluate how well each tree fits the sequence data. For each alignment, the tree with the highest AU p-value (closest to 1) is the statistically best-fit tree. A p-value < 0.05 indicates the tree can be rejected.")

    print("\n### 2. Identifying the Best Tree(s)")
    print("----------------------------------")
    best_likelihood_tree_path = "N/A"
    best_likelihood_value = -1.0
    best_likelihood_method = "N/A"
    best_topology_pair = "N/A"
    lowest_norm_rf = float('inf')
    best_pair_img = "N/A"

    # Find the best likelihood tree and its method
    for aln_name, res in aln_likelihoods.items():
        if "error" in res:
            continue

        # We need to link the tree ID (T1, T2, etc.) to the method name.
        tree_methods_for_aln = []
        for m, inf, tfile in artifacts:
            if m == aln_name:
                tree_methods_for_aln.append((m, inf))

        for tid, vals in res.items():
            if 'p-AU' in vals and vals['p-AU'] != '?':
                au_val = float(vals['p-AU'])
                if au_val > best_likelihood_value:
                    best_likelihood_value = au_val
                    method_index = int(tid) - 1
                    if method_index < len(tree_methods_for_aln):
                        m, inf = tree_methods_for_aln[method_index]
                        best_likelihood_method = f"{m}+{inf}"
                        best_likelihood_tree_path = outdir / f"{m}_{inf}.png"

    # Find the best topological similarity and its image file
    for row in table_data:
        norm_rf = float(row[2])
        if norm_rf < lowest_norm_rf:
            lowest_norm_rf = norm_rf
            best_topology_pair = row[0]
            m1, inf1 = best_topology_pair.split(" vs ")[0].split('+')
            m2, inf2 = best_topology_pair.split(" vs ")[1].split('+')
            best_pair_img = outdir / f"{m1}_{inf1}_VS_{m2}_{inf2}.png"

    print(f"The **statistically best-fit tree** is: **{best_likelihood_method}** (AU p-value: {best_likelihood_value:.4f})")
    print("This tree has the highest statistical support for its topology, as determined by the AU test.")
    print(f"\nThe **most topologically similar trees** are: **{best_topology_pair}** (Normalized RF: {lowest_norm_rf:.4f})")

    # ---------------------------------------------
    # Dynamic Summary Generation
    # ---------------------------------------------
    print("\n### 3. Quick Summary and Final Recommendation")
    print("---------------------------------------------")

    ml_au_values = []
    for aln_name, res in aln_likelihoods.items():
        if 'error' not in res:
            ml_au_values.extend([float(v.get('p-AU', 0)) for v in res.values() if 'ml' in best_likelihood_method])
    is_ml_best = all(val > 0.5 for val in ml_au_values) if ml_au_values else False

    nj_upgma_pairs = [row for row in table_data if ('nj' in row[0] or 'upgma' in row[0]) and 'ml' not in row[0]]
    distance_methods_are_similar = any(float(row[2]) < 0.5 for row in nj_upgma_pairs)

    ml_ml_pairs = [row for row in table_data if 'ml' in row[0] and len(row[0].split('ml')) > 2]
    ml_trees_are_consistent = all(float(row[3]) > 0.5 for row in ml_ml_pairs if row[3] != 'N/A')

    if is_ml_best:
        print("1. **Maximum Likelihood (ML)** trees consistently show the highest statistical support (AU p-values closest to 1). This confirms that ML is the most suitable inference method for this dataset, as it finds the most optimal topology to explain the evolutionary relationships.")

    if distance_methods_are_similar:
        print("2. The **Distance-based methods (NJ and UPGMA)** produce relatively similar trees to each other (low RF distances), but they are often statistically rejected by the likelihood tests (AU p-values near 0), indicating their topologies are less likely to be correct.")

    if ml_trees_are_consistent:
        print("3. When using the ML method, all tested alignment tools (**MAFFT**, **MUSCLE**, and **ClustalW**) produce trees with a high degree of consistency, as shown by their high branch length correlation. This suggests that for this dataset, the choice of alignment tool has a minimal impact on the final ML tree topology.")

    print(f"\n**Conclusion**: For generating the most accurate and statistically supported phylogenetic tree from this dataset, the best approach is to use the **Maximum Likelihood (ML) tree inference method**, paired with any of the tested Multiple Sequence Alignment methods.")

    # ---------------------------------------------
    # Image Location
    # ---------------------------------------------
    print("\n### 4. Locate Best Tree Images")
    print("------------------------------------")
    print(f"The best tree image is located at: {best_likelihood_tree_path}")
    print(f"The most topologically similar trees image is located at: {best_pair_img}")
    print(f"\nAll generated files are in the directory: {outdir}")

# ------------------------------
# Comparison helpers
# ------------------------------
def compare_trees(artifacts, outdir):
    print("\n🔎 Tree Comparison Results (pairwise):\n")

    if not HAVE_DENDROPY:
        print("[WARNING] DendroPy not available. Install with: pip install dendropy\n")
        return

    from dendropy.calculate import treecompare
    taxon_namespace = dendropy.TaxonNamespace()

    aln_to_trees = {}
    aln_files = {}
    for m, inf, tfile in artifacts:
        aln_file = tfile.parent / f"{m}.aln.fasta"
        aln_to_trees.setdefault(m, []).append(tfile)
        aln_files[m] = aln_file

    aln_likelihoods = {}
    for aln, tfiles in aln_to_trees.items():
        aln_file = aln_files[aln]
        aln_likelihoods[aln] = run_likelihood_tests(aln_file, tfiles)

    headers = ["Trees Compared", "RF", "Norm RF", "BranchLenCorr", "Support Values", "Likelihood Tests (AU/KH/SH)"]
    table_data = []

    for (m1, inf1, t1_file), (m2, inf2, t2_file) in combinations(artifacts, 2):
        dp_tree1 = dendropy.Tree.get(path=str(t1_file), schema="newick",
                                     taxon_namespace=taxon_namespace)
        dp_tree2 = dendropy.Tree.get(path=str(t2_file), schema="newick",
                                     taxon_namespace=taxon_namespace)

        rf = treecompare.unweighted_robinson_foulds_distance(dp_tree1, dp_tree2)
        n_leaves = len(dp_tree1.leaf_nodes())
        max_rf = 2 * (n_leaves - 3) if n_leaves > 3 else 1
        norm_rf = rf / max_rf

        blens1 = [e.length for e in dp_tree1.edges() if e.length is not None]
        blens2 = [e.length for e in dp_tree2.edges() if e.length is not None]
        corr = "N/A"
        if blens1 and blens2 and len(blens1) == len(blens2) and HAVE_SCIPY:
            corr = f"{pearsonr(blens1, blens2)[0]:.3f}"

        sup1 = [n.label for n in dp_tree1.internal_nodes() if n.label]
        sup2 = [n.label for n in dp_tree2.internal_nodes() if n.label]
        support_info = f"{len(sup1)} vs {len(sup2)}" if sup1 or sup2 else "N/A"

        aln_name = m1
        like = "N/A"
        if aln_name in aln_likelihoods:
            res = aln_likelihoods[aln_name]
            if "error" in res:
                like = res["error"]
            else:
                like = " / ".join(
                    f"T{tid}:AU={vals.get('p-AU','?')},KH={vals.get('p-KH','?')},SH={vals.get('p-SH','?')}"
                    for tid, vals in res.items()
                )

        table_data.append([
            f"{m1}+{inf1} vs {m2}+{inf2}",
            str(rf),
            f"{norm_rf:.3f}",
            str(corr),
            support_info,
            like
        ])

        render_tree_side_by_side(t1_file, t2_file, outdir / f"{m1}_{inf1}_VS_{m2}_{inf2}.png")

    # Print the table
    print("---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------")
    print(f"{headers[0]:<40} | {headers[1]:<5} | {headers[2]:<10} | {headers[3]:<15} | {headers[4]:<18} | {headers[5]:<65}")
    print("---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------")
    for row in table_data:
        print(f"{row[0]:<40} | {row[1]:<5} | {row[2]:<10} | {row[3]:<15} | {row[4]:<18} | {row[5]:<65}")
    print("---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------")

    analyze_results(table_data, aln_likelihoods, artifacts, aln_files, aln_to_trees, outdir)

# ------------------------------
# Pipeline
# ------------------------------
def pipeline(fasta: Path, molecule: str, outdir: Path,
             msa_methods, infer_methods, build_all=True, render=True):
    outdir.mkdir(parents=True, exist_ok=True)
    msa_grid = list(MSA_FUNCS.keys()) if build_all else msa_methods
    infer_grid = list(INFER_FUNCS.keys()) if build_all else infer_methods

    artifacts = []
    for m in msa_grid:
        aln_file = outdir / f"{m}.aln.fasta"
        MSA_FUNCS[m](fasta, aln_file)
        print(f"Alignment done: {m} -> {aln_file}")

        for inf in infer_grid:
            tree_file = outdir / f"{m}_{inf}.nwk"
            INFER_FUNCS[inf](aln_file, molecule, tree_file)
            print(f"Tree done: {m}+{inf} -> {tree_file}")
            if render:
                render_tree_png(tree_file, outdir / f"{m}_{inf}.png", title=f"{m}+{inf}")
            artifacts.append((m, inf, tree_file))
    return artifacts

# ------------------------------
# Interactive main
# ------------------------------

def main():
    print("\n🌱 Phylogenetic Pipeline (Drive + Resume Enabled)\n")

    fasta_path = Path(input("Enter path to FASTA file: ").strip())
    molecule = input("Sequence type (dna/protein) [auto-detect if blank]: ").strip()
    if not molecule:
        molecule = guess_molecule_from_fasta(fasta_path)
    print(f"Detected molecule type: {molecule}")

    msa_choice = input("MSA method (mafft, muscle, clustalw, all): ").strip().lower() or "mafft"
    infer_choice = input("Inference (nj, upgma, ml, all): ").strip().lower() or "ml"
    build_all = input("Run full 3x3 grid? (y/n): ").strip().lower() == "y"
    render = input("Render PNG trees? (y/n): ").strip().lower() == "y"

    base_drive = Path("/content/drive/MyDrive/PhyloPipelineRuns")
    base_drive.mkdir(parents=True, exist_ok=True)

    dataset_name = fasta_path.stem
    dataset_folders = sorted(base_drive.glob(f"{dataset_name}_*"))

    if dataset_folders:
      choice = input("Resume last run for this dataset? (y/n): ").strip().lower()
      if choice == "y":
        outdir = dataset_folders[-1]
        print(f"🔁 Resuming dataset run: {outdir}")
      else:
        outdir = base_drive / f"{dataset_name}_{timestamp()}"
        print(f"🆕 Creating new run: {outdir}")
    else:
      outdir = base_drive / f"{dataset_name}_{timestamp()}"
      print(f"🆕 Creating new run: {outdir}")

    artifacts = pipeline(
        fasta_path, molecule, outdir,
        list(MSA_FUNCS.keys()) if msa_choice == "all" else [msa_choice],
        list(INFER_FUNCS.keys()) if infer_choice == "all" else [infer_choice],
        build_all=build_all, render=render
    )

    print("\n✅ Pipeline finished! Results in:", outdir)

    if input("Compare all generated trees? (y/n): ").strip().lower() == "y":
        compare_trees(artifacts, outdir)

if __name__ == "__main__":
    main()

Overwriting phylo_pipeline1.py


**RUN THIS ↓**

In [ ]:
!python3 phylo_pipeline1.py


🌱 Phylogenetic Pipeline (Drive + Resume Enabled)

Enter path to FASTA file: /content/Temnothorax_COI_30species_phyloSafe.fasta
Sequence type (dna/protein) [auto-detect if blank]: 
Detected molecule type: dna
MSA method (mafft, muscle, clustalw, all): all
Inference (nj, upgma, ml, all): all
Run full 3x3 grid? (y/n): y
Render PNG trees? (y/n): y
Resume last run for this dataset? (y/n): n
🆕 Creating new run: /content/drive/MyDrive/PhyloPipelineRuns/Temnothorax_COI_30species_phyloSafe_20260305-154956
[CMD] mafft --auto /content/Temnothorax_COI_30species_phyloSafe.fasta > /content/drive/MyDrive/PhyloPipelineRuns/Temnothorax_COI_30species_phyloSafe_20260305-154956/mafft.aln.fasta
Alignment done: mafft -> /content/drive/MyDrive/PhyloPipelineRuns/Temnothorax_COI_30species_phyloSafe_20260305-154956/mafft.aln.fasta
Tree done: mafft+nj -> /content/drive/MyDrive/PhyloPipelineRuns/Temnothorax_COI_30species_phyloSafe_20260305-154956/mafft_nj.nwk
Tree done: mafft+upgma -> /content/drive/MyDrive/Phyl

In [ ]:
!apt-get update
!apt-get install -y phylip

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Fetched 3,917 B in 1s (2,893 B/s)
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
Suggested packages:
  phylip-doc
The following NEW

In [ ]:
!cd /

In [ ]:
!ls

alignment.phy			      mafft_nj_VS_clustalw_ml.png
clustalw.aln.fasta		      mafft_nj_VS_clustalw_nj.png
clustalw.aln.fasta.bionj	      mafft_nj_VS_clustalw_upgma.png
clustalw.aln.fasta.ckp.gz	      mafft_nj_VS_mafft_ml.png
clustalw.aln.fasta.iqtree	      mafft_nj_VS_mafft_upgma.png
clustalw.aln.fasta.liketest.bionj     mafft_nj_VS_muscle_ml.png
clustalw.aln.fasta.liketest.ckp.gz    mafft_nj_VS_muscle_nj.png
clustalw.aln.fasta.liketest.iqtree    mafft_nj_VS_muscle_upgma.png
clustalw.aln.fasta.liketest.log       mafft_upgma.nwk
clustalw.aln.fasta.liketest.mldist    mafft_upgma.png
clustalw.aln.fasta.liketest.model.gz  mafft_upgma_VS_clustalw_ml.png
clustalw.aln.fasta.liketest.treefile  mafft_upgma_VS_clustalw_nj.png
clustalw.aln.fasta.liketest.trees     mafft_upgma_VS_clustalw_upgma.png
clustalw.aln.fasta.log		      mafft_upgma_VS_mafft_ml.png
clustalw.aln.fasta.mldist	      mafft_upgma_VS_muscle_ml.png
clustalw.aln.fasta.model.gz	      mafft_upgma_VS_muscle_nj.png
clustalw.aln.fasta.tre

In [ ]:
!apt-get install phylip -y

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
phylip is already the newest version (1:3.697+dfsg-2).
0 upgraded, 0 newly installed, 0 to remove and 22 not upgraded.


In [ ]:
!which phylip

/usr/bin/phylip


In [ ]:
!dpkg -L phylip | grep dnapars

/usr/lib/phylip/bin/dnapars
/usr/share/doc/phylip/examples/tests/dnapars.ok


In [ ]:
!ln -s /usr/lib/phylip/bin/dnapars /usr/bin/dnapars

In [ ]:
!which dnapars

/usr/bin/dnapars


In [ ]:
!dnapars

dnapars: can't find input file "infile"
Please enter a new file name> ^C


In [ ]:
#----MP with PHYLIP

%%writefile phylo_pipeline1.py
#!/usr/bin/env python3

"""
Phylogenetic Tree Generation, Visualization, and Comparison Pipeline
Extended with Likelihood Model-Fit Tests (SH, KH, AU) using IQ-TREE.
Now Extended With:
✔ Google Drive storage
✔ Resume capability (checkpoint skipping)
✔ ZERO functionality removed
"""

import os, sys, shutil, subprocess, datetime as dt
from pathlib import Path
from itertools import combinations
import numpy as np
from Bio import AlignIO
from Bio import Phylo
import matplotlib.pyplot as plt

# ------------------------------
# Dependencies
# ------------------------------
try:
    from Bio import AlignIO, Phylo, SeqIO
    from Bio.Phylo.TreeConstruction import DistanceCalculator, DistanceTreeConstructor
except Exception:
    sys.stderr.write("[ERROR] Biopython required. Install with: pip install biopython\n")
    raise

try:
    from scipy.stats import pearsonr
    HAVE_SCIPY = True
except Exception:
    HAVE_SCIPY = False

try:
    import dendropy
    HAVE_DENDROPY = True
except Exception:
    HAVE_DENDROPY = False

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ------------------------------
# Utilities
# ------------------------------
def run_cmd(cmd: str, check: bool = True):
    print(f"[CMD] {cmd}")
    result = subprocess.run(cmd, shell=True,
                            stdout=subprocess.PIPE,
                            stderr=subprocess.PIPE,
                            text=True)
    if check and result.returncode != 0:
        print(result.stderr)
        raise RuntimeError(f"Command failed: {cmd}")
    return result.stdout

def timestamp():
    return dt.datetime.now().strftime("%Y%m%d-%H%M%S")

def guess_molecule_from_fasta(fasta_path: Path):
    aa_letters = set("EFILPQZVWYMRSTKDHCNG")
    nt_letters = set("ACGTURYKMSWBDHVN")
    aa_hits = nt_hits = 0
    for rec in SeqIO.parse(str(fasta_path), "fasta"):
        s = set(str(rec.seq).upper())
        aa_hits += len(s & aa_letters)
        nt_hits += len(s & nt_letters)
        if aa_hits + nt_hits > 50:
            break
    return "protein" if aa_hits > nt_hits else "dna"

# ------------------------------
# MSA wrappers (with resume)
# ------------------------------
def run_mafft(in_fa: Path, out_fa: Path):
    if out_fa.exists():
        print(f"[RESUME] MAFFT alignment exists → Skipping")
        return
    run_cmd(f"mafft --auto {in_fa} > {out_fa}")

def run_muscle(in_fa: Path, out_fa: Path):
    if out_fa.exists():
        print(f"[RESUME] MUSCLE alignment exists → Skipping")
        return
    run_cmd(f"muscle -in {in_fa} -out {out_fa}")

def run_clustalw(in_fa: Path, out_fa: Path):
    if out_fa.exists():
        print(f"[RESUME] ClustalW alignment exists → Skipping")
        return
    run_cmd(f"clustalw -infile={in_fa}")
    aln_file = str(in_fa).replace(".fasta", ".aln")
    if os.path.exists(aln_file):
        AlignIO.convert(aln_file, "clustal", out_fa, "fasta")

MSA_FUNCS = {"mafft": run_mafft, "muscle": run_muscle, "clustalw": run_clustalw}

# ------------------------------
# Tree inference (with resume)
# ------------------------------
def infer_nj(aln_fa: Path, molecule: str, out_newick: Path):
    if out_newick.exists():
        print(f"[RESUME] NJ tree exists → Skipping")
        return
    aln = AlignIO.read(str(aln_fa), "fasta")
    model = "identity" if molecule == "dna" else "blosum62"
    dm = DistanceCalculator(model).get_distance(aln)
    tree = DistanceTreeConstructor().nj(dm)
    Phylo.write(tree, str(out_newick), "newick")

def infer_upgma(aln_fa: Path, molecule: str, out_newick: Path):
    if out_newick.exists():
        print(f"[RESUME] UPGMA tree exists → Skipping")
        return
    aln = AlignIO.read(str(aln_fa), "fasta")
    model = "identity" if molecule == "dna" else "blosum62"
    dm = DistanceCalculator(model).get_distance(aln)
    tree = DistanceTreeConstructor().upgma(dm)
    Phylo.write(tree, str(out_newick), "newick")

def infer_ml_iqtree(aln_fa: Path, molecule: str, out_newick: Path):
    if out_newick.exists():
        print(f"[RESUME] ML tree exists → Skipping")
        return
    aln_fasta = str(aln_fa)
    cmd = f"iqtree2 -s {aln_fasta} -m TEST -nt AUTO -quiet"
    run_cmd(cmd)
    iqtree_file = aln_fasta + ".treefile"
    if os.path.exists(iqtree_file):
        shutil.copy(iqtree_file, out_newick)


def run_phylip_mp(aln_fa: Path, molecule: str, out_newick: Path):
    """
    Run PHYLIP DNAPARS (Maximum Parsimony) safely and non-interactively.
    Compatible with pipeline inference interface.
    """

    if out_newick.exists():
        print(f"[RESUME] MP tree exists → Skipping")
        return

    workdir = out_newick.parent
    clean_fasta = workdir / "clean_alignment.fasta"
    phylip_file = workdir / "alignment.phy"

    print("[MP] Cleaning alignment for PHYLIP...")

    valid = set("ACGTN-")

    # Clean sequences: replace any invalid base with 'N'
    with open(aln_fa) as fin, open(clean_fasta, "w") as fout:
        for line in fin:
            if line.startswith(">"):
                fout.write(line)
            else:
                seq = line.strip().upper()
                seq = "".join([c if c in valid else "N" for c in seq])
                fout.write(seq + "\n")

    print("[MP] Converting FASTA → PHYLIP (sequential format)")

    # Use strict PHYLIP sequential format to avoid DNAPARS errors
    AlignIO.convert(
        str(clean_fasta),
        "fasta",
        str(phylip_file),
        "phylip"  # strict PHYLIP, not relaxed
    )

    infile = workdir / "infile"
    shutil.copy(phylip_file, infile)

    print("[MP] Running DNAPARS (non-interactive)...")

    # Remove any leftover files
    for f in ["outfile", "outtree"]:
        p = workdir / f
        if p.exists():
            p.unlink()

    # Prepare command to run DNAPARS non-interactively
    # Automatically accepts defaults, saves tree to outtree
    cmd = (
        f"cd {workdir} && "
        f"printf 'Y\n' | /usr/lib/phylip/bin/dnapars"
    )

    try:
        subprocess.run(cmd, shell=True, check=True)
    except subprocess.CalledProcessError:
        print("⚠️ DNAPARS failed")
        return

    phylip_tree = workdir / "outtree"

    if not phylip_tree.exists():
        print("⚠️ DNAPARS did not produce tree")
        return

    shutil.copy(phylip_tree, out_newick)
    print(f"[MP] Tree generated → {out_newick}")


INFER_FUNCS = {"nj": infer_nj, "upgma": infer_upgma, "ml": infer_ml_iqtree, "mp": run_phylip_mp}


# ------------------------------
# Visualization (with resume)
# ------------------------------
def render_tree_side_by_side(tree1_path: Path, tree2_path: Path, out_png: Path):
    """
    Render two trees side-by-side into one PNG file.
    Handles multiple trees in PHYLIP output by using the first tree.
    """
    if out_png.exists():
        print(f"[RESUME] Side-by-side PNG exists → Skipping")
        return

    # Parse trees safely (pick first tree if multiple)
    trees1 = list(Phylo.parse(str(tree1_path), "newick"))
    trees2 = list(Phylo.parse(str(tree2_path), "newick"))

    if not trees1 or not trees2:
        print(f"⚠️ No trees found in {tree1_path} or {tree2_path}")
        return

    tree1 = trees1[0]
    tree2 = trees2[0]

    fig = plt.figure(figsize=(12, 6))

    ax1 = fig.add_subplot(1, 2, 1)
    Phylo.draw(tree1, do_show=False, axes=ax1)
    ax1.set_title(tree1_path.stem)

    ax2 = fig.add_subplot(1, 2, 2)
    Phylo.draw(tree2, do_show=False, axes=ax2)
    ax2.set_title(tree2_path.stem)

    plt.tight_layout()
    plt.savefig(str(out_png))
    plt.close(fig)
    print(f"[PNG] Side-by-side tree rendered → {out_png}")


def render_tree_png(newick: Path, out_png: Path, title="Tree"):
    """
    Render a single phylogenetic tree into a PNG image.
    Handles multiple trees in PHYLIP output by using the first tree.
    """
    if out_png.exists():
        print(f"[RESUME] PNG exists → Skipping")
        return

    # Parse all trees but only use the first one
    trees = list(Phylo.parse(str(newick), "newick"))
    if not trees:
        print(f"⚠️ No trees found in {newick}")
        return

    tree = trees[0]  # take first tree

    fig = plt.figure(figsize=(6, 6))
    ax = fig.add_subplot(1, 1, 1)

    Phylo.draw(tree, do_show=False, axes=ax)
    ax.set_title(title)

    plt.tight_layout()
    plt.savefig(str(out_png))
    plt.close(fig)
    print(f"[PNG] Tree rendered → {out_png}")

# ------------------------------
# YOUR ORIGINAL compare_trees, analyze_results, likelihood tests
# ------------------------------
# ⚠️ UNCHANGED — COPY THEM EXACTLY FROM YOUR ORIGINAL CODE BELOW THIS POINT


# ------------------------------
# Likelihood model-fit tests
# ------------------------------
def run_likelihood_tests(aln_fa: Path, tree_files: list[Path]) -> dict:
    """
    Run IQ-TREE SH/KH/AU tests on given trees and return parsed results.
    """
    if not shutil.which("iqtree2"):
        return {"error": "iqtree2 not installed"}

    # Create tree set file
    all_trees = aln_fa.with_suffix(".treeset.nwk")
    with open(all_trees, "w") as out:
        for t in tree_files:
            with open(t) as f:
                out.write(f.read().strip() + "\n")

    # ✅ DEFINE PREFIX FIRST
    prefix = str(aln_fa) + ".liketest"

    # Run IQ-TREE
    cmd = (
        f"iqtree2 -s {aln_fa} "
        f"-z {all_trees} "
        f"-zb 1000 -au "
        f"-nt AUTO -redo "
        f"-pre {prefix} "
        f"-quiet"
    )
    run_cmd(cmd)

    # Parse output
    iqtree_out = Path(prefix + ".iqtree")
    results = {}

    if iqtree_out.exists():
        with open(iqtree_out) as f:
            lines = [l.strip() for l in f if l.strip()]

        header = []
        for line in lines:
            if line.startswith("Tree"):
                header = line.split()
                continue

            if header and line[0].isdigit():
                parts = line.split()
                tree_id = parts[0]
                results[tree_id] = dict(zip(header[1:], parts[1:]))

    return results

# ------------------------------
# Analysis & Summary
# ------------------------------
def analyze_results(table_data, aln_likelihoods, artifacts, aln_files, aln_to_trees, outdir):
    print("\n\n📊 Result Analysis and Conclusion 📊")
    print("-----------------------------------")
    print("\n### 1. Explanation of the Table")
    print("---------------------------------")
    print("The table compares all generated phylogenetic trees using a combination of methods (e.g., MAFFT+NJ, MUSCLE+ML).")
    print("- **RF (Robinson-Foulds) Distance**: A measure of the topological difference between two trees. A lower value means the trees are more similar.")
    print("- **Normalized RF**: The RF distance scaled to a 0-1 range. Lower is better.")
    print("- **BranchLenCorr**: The correlation coefficient of branch lengths. A value closer to 1 means branch lengths are highly similar.")
    print("- **Support Values**: Indicates the number of internal branches with bootstrap support values. A higher number suggests more confidence in the tree's internal structure.")
    print("- **Likelihood Tests (AU/KH/SH)**: Statistical tests from IQ-TREE to evaluate how well each tree fits the sequence data. For each alignment, the tree with the highest AU p-value (closest to 1) is the statistically best-fit tree. A p-value < 0.05 indicates the tree can be rejected.")

    print("\n### 2. Identifying the Best Tree(s)")
    print("----------------------------------")
    best_likelihood_tree_path = "N/A"
    best_likelihood_value = -1.0
    best_likelihood_method = "N/A"
    best_topology_pair = "N/A"
    lowest_norm_rf = float('inf')
    best_pair_img = "N/A"

    # Find the best likelihood tree and its method
    for aln_name, res in aln_likelihoods.items():
        if "error" in res:
            continue

        # We need to link the tree ID (T1, T2, etc.) to the method name.
        tree_methods_for_aln = []
        for m, inf, tfile in artifacts:
            if m == aln_name:
                tree_methods_for_aln.append((m, inf))

        for tid, vals in res.items():
            if 'p-AU' in vals and vals['p-AU'] != '?':
                au_val = float(vals['p-AU'])
                if au_val > best_likelihood_value:
                    best_likelihood_value = au_val
                    method_index = int(tid) - 1
                    if method_index < len(tree_methods_for_aln):
                        m, inf = tree_methods_for_aln[method_index]
                        best_likelihood_method = f"{m}+{inf}"
                        best_likelihood_tree_path = outdir / f"{m}_{inf}.png"

    # Find the best topological similarity and its image file
    for row in table_data:
        norm_rf = float(row[2])
        if norm_rf < lowest_norm_rf:
            lowest_norm_rf = norm_rf
            best_topology_pair = row[0]
            m1, inf1 = best_topology_pair.split(" vs ")[0].split('+')
            m2, inf2 = best_topology_pair.split(" vs ")[1].split('+')
            best_pair_img = outdir / f"{m1}_{inf1}_VS_{m2}_{inf2}.png"

    print(f"The **statistically best-fit tree** is: **{best_likelihood_method}** (AU p-value: {best_likelihood_value:.4f})")
    print("This tree has the highest statistical support for its topology, as determined by the AU test.")
    print(f"\nThe **most topologically similar trees** are: **{best_topology_pair}** (Normalized RF: {lowest_norm_rf:.4f})")

    # ---------------------------------------------
    # Dynamic Summary Generation
    # ---------------------------------------------
    print("\n### 3. Quick Summary and Final Recommendation")
    print("---------------------------------------------")

    ml_au_values = []
    for aln_name, res in aln_likelihoods.items():
        if 'error' not in res:
            ml_au_values.extend([float(v.get('p-AU', 0)) for v in res.values() if 'ml' in best_likelihood_method])
    is_ml_best = all(val > 0.5 for val in ml_au_values) if ml_au_values else False

    nj_upgma_pairs = [row for row in table_data if ('nj' in row[0] or 'upgma' in row[0]) and 'ml' not in row[0]]
    distance_methods_are_similar = any(float(row[2]) < 0.5 for row in nj_upgma_pairs)

    ml_ml_pairs = [row for row in table_data if 'ml' in row[0] and len(row[0].split('ml')) > 2]
    ml_trees_are_consistent = all(float(row[3]) > 0.5 for row in ml_ml_pairs if row[3] != 'N/A')

    if is_ml_best:
        print("1. **Maximum Likelihood (ML)** trees consistently show the highest statistical support (AU p-values closest to 1). This confirms that ML is the most suitable inference method for this dataset, as it finds the most optimal topology to explain the evolutionary relationships.")

    if distance_methods_are_similar:
        print("2. The **Distance-based methods (NJ and UPGMA)** produce relatively similar trees to each other (low RF distances), but they are often statistically rejected by the likelihood tests (AU p-values near 0), indicating their topologies are less likely to be correct.")

    if ml_trees_are_consistent:
        print("3. When using the ML method, all tested alignment tools (**MAFFT**, **MUSCLE**, and **ClustalW**) produce trees with a high degree of consistency, as shown by their high branch length correlation. This suggests that for this dataset, the choice of alignment tool has a minimal impact on the final ML tree topology.")

    print(f"\n**Conclusion**: For generating the most accurate and statistically supported phylogenetic tree from this dataset, the best approach is to use the **Maximum Likelihood (ML) tree inference method**, paired with any of the tested Multiple Sequence Alignment methods.")

    # ---------------------------------------------
    # Image Location
    # ---------------------------------------------
    print("\n### 4. Locate Best Tree Images")
    print("------------------------------------")
    print(f"The best tree image is located at: {best_likelihood_tree_path}")
    print(f"The most topologically similar trees image is located at: {best_pair_img}")
    print(f"\nAll generated files are in the directory: {outdir}")

# ------------------------------
# Comparison helpers
# ------------------------------
def compare_trees(artifacts, outdir):
    print("\n🔎 Tree Comparison Results (pairwise):\n")

    if not HAVE_DENDROPY:
        print("[WARNING] DendroPy not available. Install with: pip install dendropy\n")
        return

    from dendropy.calculate import treecompare
    taxon_namespace = dendropy.TaxonNamespace()

    aln_to_trees = {}
    aln_files = {}
    for m, inf, tfile in artifacts:
        aln_file = tfile.parent / f"{m}.aln.fasta"
        aln_to_trees.setdefault(m, []).append(tfile)
        aln_files[m] = aln_file

    aln_likelihoods = {}
    for aln, tfiles in aln_to_trees.items():
        aln_file = aln_files[aln]
        aln_likelihoods[aln] = run_likelihood_tests(aln_file, tfiles)

    headers = ["Trees Compared", "RF", "Norm RF", "BranchLenCorr", "Support Values", "Likelihood Tests (AU/KH/SH)"]
    table_data = []

    for (m1, inf1, t1_file), (m2, inf2, t2_file) in combinations(artifacts, 2):
        dp_tree1 = dendropy.Tree.get(path=str(t1_file), schema="newick",
                                     taxon_namespace=taxon_namespace)
        dp_tree2 = dendropy.Tree.get(path=str(t2_file), schema="newick",
                                     taxon_namespace=taxon_namespace)

        rf = treecompare.unweighted_robinson_foulds_distance(dp_tree1, dp_tree2)
        n_leaves = len(dp_tree1.leaf_nodes())
        max_rf = 2 * (n_leaves - 3) if n_leaves > 3 else 1
        norm_rf = rf / max_rf

        blens1 = [e.length for e in dp_tree1.edges() if e.length is not None]
        blens2 = [e.length for e in dp_tree2.edges() if e.length is not None]
        corr = "N/A"
        if blens1 and blens2 and len(blens1) == len(blens2) and HAVE_SCIPY:
            corr = f"{pearsonr(blens1, blens2)[0]:.3f}"

        sup1 = [n.label for n in dp_tree1.internal_nodes() if n.label]
        sup2 = [n.label for n in dp_tree2.internal_nodes() if n.label]
        support_info = f"{len(sup1)} vs {len(sup2)}" if sup1 or sup2 else "N/A"

        aln_name = m1
        like = "N/A"
        if aln_name in aln_likelihoods:
            res = aln_likelihoods[aln_name]
            if "error" in res:
                like = res["error"]
            else:
                like = " / ".join(
                    f"T{tid}:AU={vals.get('p-AU','?')},KH={vals.get('p-KH','?')},SH={vals.get('p-SH','?')}"
                    for tid, vals in res.items()
                )

        table_data.append([
            f"{m1}+{inf1} vs {m2}+{inf2}",
            str(rf),
            f"{norm_rf:.3f}",
            str(corr),
            support_info,
            like
        ])

        render_tree_side_by_side(t1_file, t2_file, outdir / f"{m1}_{inf1}_VS_{m2}_{inf2}.png")

    # Print the table
    print("---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------")
    print(f"{headers[0]:<40} | {headers[1]:<5} | {headers[2]:<10} | {headers[3]:<15} | {headers[4]:<18} | {headers[5]:<65}")
    print("---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------")
    for row in table_data:
        print(f"{row[0]:<40} | {row[1]:<5} | {row[2]:<10} | {row[3]:<15} | {row[4]:<18} | {row[5]:<65}")
    print("---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------")

    analyze_results(table_data, aln_likelihoods, artifacts, aln_files, aln_to_trees, outdir)

# ------------------------------
# Pipeline
# ------------------------------
def pipeline(fasta: Path, molecule: str, outdir: Path,
             msa_methods, infer_methods, build_all=True, render=True):
    outdir.mkdir(parents=True, exist_ok=True)
    msa_grid = list(MSA_FUNCS.keys()) if build_all else msa_methods
    infer_grid = list(INFER_FUNCS.keys()) if build_all else infer_methods

    artifacts = []
    for m in msa_grid:
        aln_file = outdir / f"{m}.aln.fasta"
        MSA_FUNCS[m](fasta, aln_file)
        print(f"Alignment done: {m} -> {aln_file}")

        for inf in infer_grid:
            tree_file = outdir / f"{m}_{inf}.nwk"
            INFER_FUNCS[inf](aln_file, molecule, tree_file)
            print(f"Tree done: {m}+{inf} -> {tree_file}")
            if render:
                render_tree_png(tree_file, outdir / f"{m}_{inf}.png", title=f"{m}+{inf}")
            artifacts.append((m, inf, tree_file))
    return artifacts

# ------------------------------
# Interactive main
# ------------------------------

def main():
    print("\n🌱 Phylogenetic Pipeline (Drive + Resume Enabled)\n")

    fasta_path = Path(input("Enter path to FASTA file: ").strip())
    molecule = input("Sequence type (dna/protein) [auto-detect if blank]: ").strip()
    if not molecule:
        molecule = guess_molecule_from_fasta(fasta_path)
    print(f"Detected molecule type: {molecule}")

    msa_choice = input("MSA method (mafft, muscle, clustalw, all): ").strip().lower() or "mafft"
    infer_choice = input("Inference (nj, upgma, ml, mp, all): ").strip().lower() or "ml"
    build_all = input("Run full 3x3 grid? (y/n): ").strip().lower() == "y"
    render = input("Render PNG trees? (y/n): ").strip().lower() == "y"

    base_drive = Path("/content/drive/MyDrive/PhyloPipelineRuns")
    base_drive.mkdir(parents=True, exist_ok=True)

    dataset_name = fasta_path.stem
    dataset_folders = sorted(base_drive.glob(f"{dataset_name}_*"))

    if dataset_folders:
      choice = input("Resume last run for this dataset? (y/n): ").strip().lower()
      if choice == "y":
        outdir = dataset_folders[-1]
        print(f"🔁 Resuming dataset run: {outdir}")
      else:
        outdir = base_drive / f"{dataset_name}_{timestamp()}"
        print(f"🆕 Creating new run: {outdir}")
    else:
      outdir = base_drive / f"{dataset_name}_{timestamp()}"
      print(f"🆕 Creating new run: {outdir}")

    artifacts = pipeline(
        fasta_path, molecule, outdir,
        list(MSA_FUNCS.keys()) if msa_choice == "all" else [msa_choice],
        list(INFER_FUNCS.keys()) if infer_choice == "all" else [infer_choice],
        build_all=build_all, render=render
    )

    print("\n✅ Pipeline finished! Results in:", outdir)

    if input("Compare all generated trees? (y/n): ").strip().lower() == "y":
        compare_trees(artifacts, outdir)

if __name__ == "__main__":
    main()

Writing phylo_pipeline1.py


In [ ]:
#----MP with PHYLIP (2)

%%writefile phylo_pipeline1.py
#!/usr/bin/env python3

"""
Phylogenetic Tree Generation, Visualization, and Comparison Pipeline
Extended with Likelihood Model-Fit Tests (SH, KH, AU) using IQ-TREE.
Now Extended With:
✔ Google Drive storage
✔ Resume capability (checkpoint skipping)
✔ ZERO functionality removed
"""

import os, sys, shutil, subprocess, datetime as dt
from pathlib import Path
from itertools import combinations
import numpy as np
from Bio import AlignIO
from Bio import Phylo
import matplotlib.pyplot as plt

# ------------------------------
# Dependencies
# ------------------------------
try:
    from Bio import AlignIO, Phylo, SeqIO
    from Bio.Phylo.TreeConstruction import DistanceCalculator, DistanceTreeConstructor
except Exception:
    sys.stderr.write("[ERROR] Biopython required. Install with: pip install biopython\n")
    raise

try:
    from scipy.stats import pearsonr
    HAVE_SCIPY = True
except Exception:
    HAVE_SCIPY = False

try:
    import dendropy
    HAVE_DENDROPY = True
except Exception:
    HAVE_DENDROPY = False

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from Bio import SeqIO
from ete3 import Tree

def normalize_taxon_name(name):
    """Normalize taxon names: keep only alphanumeric, underscore, or hyphen"""
    return "".join(c for c in name if c.isalnum() or c in "_-")

def sanitize_alignment(aln_file):
    aln_path = Path(aln_file)
    sanitized_file = aln_path.with_name(aln_path.stem + ".sanitized.fasta")

    mapping = {}

    with open(sanitized_file, "w") as out_f:
        for i, record in enumerate(SeqIO.parse(aln_path, "fasta"), 1):
            new_id = f"T{i}"   # ✅ short name
            mapping[new_id] = record.id

            record.id = new_id
            record.description = ""
            SeqIO.write(record, out_f, "fasta")

    # Save mapping for later interpretation
    map_file = aln_path.with_name("taxa_mapping.txt")
    with open(map_file, "w") as f:
        for k, v in mapping.items():
            f.write(f"{k} -> {v}\n")

    return str(sanitized_file)

def sanitize_tree(tree_file):
    """
    Robust tree sanitizer.
    Works with PHYLIP, IQ-TREE, RAxML and any Newick variants.
    Extracts the first valid Newick tree and normalizes leaf names.
    """

    tree_file = Path(tree_file)

    if not tree_file.exists():
        raise FileNotFoundError(f"Tree file not found: {tree_file}")

    sanitized_file = tree_file.with_name(tree_file.stem + ".sanitized.nwk")

    newick_line = None

    with open(tree_file) as f:
      for line in f:
        line = line.strip()

        if not line:
            continue

        # Look for valid Newick
        if "(" in line and ")" in line and ";" in line:
            newick_line = line
            break   # ✅ break ONLY when found

      if not newick_line:
        raise RuntimeError(f"No valid Newick tree found in {tree_file}")

    # Parse tree safely
    try:
        tree = Tree(newick_line, format=1)
    except Exception:
        tree = Tree(newick_line)

    # Normalize leaf names
    for leaf in tree.iter_leaves():
        leaf.name = "".join(c for c in leaf.name if c.isalnum() or c in "_-")

    tree.write(outfile=str(sanitized_file), format=1)

    return str(sanitized_file)


# ------------------------------
# Utilities
# ------------------------------
def run_cmd(cmd: str, check: bool = True):
    print(f"[CMD] {cmd}")
    result = subprocess.run(cmd, shell=True,
                            stdout=subprocess.PIPE,
                            stderr=subprocess.PIPE,
                            text=True)
    if check and result.returncode != 0:
        print(result.stderr)
        raise RuntimeError(f"Command failed: {cmd}")
    return result.stdout

def timestamp():
    return dt.datetime.now().strftime("%Y%m%d-%H%M%S")

def guess_molecule_from_fasta(fasta_path: Path):
    aa_letters = set("EFILPQZVWYMRSTKDHCNG")
    nt_letters = set("ACGTURYKMSWBDHVN")
    aa_hits = nt_hits = 0
    for rec in SeqIO.parse(str(fasta_path), "fasta"):
        s = set(str(rec.seq).upper())
        aa_hits += len(s & aa_letters)
        nt_hits += len(s & nt_letters)
        if aa_hits + nt_hits > 50:
            break
    return "protein" if aa_hits > nt_hits else "dna"

# ------------------------------
# MSA wrappers (with resume)
# ------------------------------
def run_mafft(in_fa: Path, out_fa: Path):
    if out_fa.exists():
        print(f"[RESUME] MAFFT alignment exists → Skipping")
        return
    run_cmd(f"mafft --auto {in_fa} > {out_fa}")

def run_muscle(in_fa: Path, out_fa: Path):
    if out_fa.exists():
        print(f"[RESUME] MUSCLE alignment exists → Skipping")
        return
    run_cmd(f"muscle -in {in_fa} -out {out_fa}")

def run_clustalw(in_fa: Path, out_fa: Path):
    if out_fa.exists():
        print(f"[RESUME] ClustalW alignment exists → Skipping")
        return
    run_cmd(f"clustalw -infile={in_fa}")
    aln_file = str(in_fa).replace(".fasta", ".aln")
    if os.path.exists(aln_file):
        AlignIO.convert(aln_file, "clustal", out_fa, "fasta")

MSA_FUNCS = {"mafft": run_mafft, "muscle": run_muscle, "clustalw": run_clustalw}

# ------------------------------
# Taxa normalization (post-alignment and trees)
# ------------------------------
def normalize_fasta_taxa(aln_file: Path, inplace=True):
    """
    Normalize taxa names in a FASTA alignment (all sequences must be same length).
    This should be run **after alignment**.
    """
    aln = AlignIO.read(str(aln_file), "fasta")  # safe: aligned sequences
    for record in aln:
        record.id = normalize_taxon_name(record.id)
        record.name = record.id
    if inplace:
        AlignIO.write(aln, str(aln_file), "fasta")
    else:
        out_file = aln_file.with_name(aln_file.stem + "_normalized.fasta")
        AlignIO.write(aln, str(out_file), "fasta")
        return out_file

def normalize_tree_taxa(tree_file: Path, out_file: Path = None):
    """
    Normalize taxa names in a Newick tree to match FASTA labels.
    """
    if out_file is None:
        out_file = tree_file
    trees = list(Phylo.parse(str(tree_file), "newick"))
    for tree in trees:
        for clade in tree.find_clades():
            if clade.name:
                clade.name = clade.name.replace("_", "").replace(" ", "")
    Phylo.write(trees, str(out_file), "newick")
    return out_file


# ------------------------------
# Tree inference (with resume)
# ------------------------------
def infer_nj(aln_fa: Path, molecule: str, out_newick: Path):
    if out_newick.exists():
        print(f"[RESUME] NJ tree exists → Skipping")
        return
    aln = AlignIO.read(str(aln_fa), "fasta")
    model = "identity" if molecule == "dna" else "blosum62"
    dm = DistanceCalculator(model).get_distance(aln)
    tree = DistanceTreeConstructor().nj(dm)
    Phylo.write(tree, str(out_newick), "newick")

def infer_upgma(aln_fa: Path, molecule: str, out_newick: Path):
    if out_newick.exists():
        print(f"[RESUME] UPGMA tree exists → Skipping")
        return
    aln = AlignIO.read(str(aln_fa), "fasta")
    model = "identity" if molecule == "dna" else "blosum62"
    dm = DistanceCalculator(model).get_distance(aln)
    tree = DistanceTreeConstructor().upgma(dm)
    Phylo.write(tree, str(out_newick), "newick")

def infer_ml_iqtree(aln_fa: Path, molecule: str, out_newick: Path):
    if out_newick.exists():
        print(f"[RESUME] ML tree exists → Skipping")
        return
    aln_fasta = str(aln_fa)
    cmd = f"iqtree2 -s {aln_fasta} -m TEST -nt AUTO -quiet"
    run_cmd(cmd)
    iqtree_file = aln_fasta + ".treefile"
    if os.path.exists(iqtree_file):
        shutil.copy(iqtree_file, out_newick)

    sanitize_tree(str(out_newick))


def run_phylip_mp(aln_fa: Path, molecule: str, out_newick: Path):
    """
    Run PHYLIP DNAPARS (Maximum Parsimony) safely and non-interactively.
    Compatible with pipeline inference interface.
    """

    if out_newick.exists():
        print(f"[RESUME] MP tree exists → Skipping")
        return

    workdir = out_newick.parent
    clean_fasta = workdir / "clean_alignment.fasta"
    phylip_file = workdir / "alignment.phy"

    print("[MP] Cleaning alignment for PHYLIP...")

    valid = set("ACGTN-")

    # Clean sequences: replace any invalid base with 'N'
    with open(aln_fa) as fin, open(clean_fasta, "w") as fout:
        for line in fin:
            if line.startswith(">"):
                fout.write(line)
            else:
                seq = line.strip().upper()
                seq = "".join([c if c in valid else "N" for c in seq])
                fout.write(seq + "\n")

    print("[MP] Converting FASTA → PHYLIP (sequential format)")

    # Use strict PHYLIP sequential format to avoid DNAPARS errors
    AlignIO.convert(
        str(clean_fasta),
        "fasta",
        str(phylip_file),
        "phylip"  # strict PHYLIP, not relaxed
    )

    infile = workdir / "infile"
    shutil.copy(phylip_file, infile)

    print("[MP] Running DNAPARS (non-interactive)...")

    # Remove any leftover files
    for f in ["outfile", "outtree"]:
        p = workdir / f
        if p.exists():
            p.unlink()

    # Prepare command to run DNAPARS non-interactively
    # Automatically accepts defaults, saves tree to outtree
    cmd = (
        f"cd {workdir} && "
        f"printf 'Y\n' | /usr/lib/phylip/bin/dnapars"
    )

    try:
        subprocess.run(cmd, shell=True, check=True)
    except subprocess.CalledProcessError:
        print("⚠️ DNAPARS failed")
        return

    phylip_tree = workdir / "outtree"

    if not phylip_tree.exists():
        print("⚠️ DNAPARS did not produce tree")
        return

    shutil.copy(phylip_tree, out_newick)
    print(f"[MP] Tree generated → {out_newick}")

    try:
      sanitize_tree(str(out_newick))
    except Exception as e:
      print(f"{e}")


INFER_FUNCS = {"nj": infer_nj, "upgma": infer_upgma, "ml": infer_ml_iqtree, "mp": run_phylip_mp}


# ------------------------------
# Visualization (with resume)
# ------------------------------
def render_tree_side_by_side(tree1_path: Path, tree2_path: Path, out_png: Path):
    """
    Render two trees side-by-side into one PNG file.
    Handles multiple trees in PHYLIP output by using the first tree.
    """
    if out_png.exists():
        print(f"[RESUME] Side-by-side PNG exists → Skipping")
        return

    # Parse trees safely (pick first tree if multiple)
    trees1 = list(Phylo.parse(str(tree1_path), "newick"))
    trees2 = list(Phylo.parse(str(tree2_path), "newick"))

    if not trees1 or not trees2:
        print(f"⚠️ No trees found in {tree1_path} or {tree2_path}")
        return

    tree1 = trees1[0]
    tree2 = trees2[0]

    fig = plt.figure(figsize=(12, 6))

    ax1 = fig.add_subplot(1, 2, 1)
    Phylo.draw(tree1, do_show=False, axes=ax1)
    ax1.set_title(tree1_path.stem)

    ax2 = fig.add_subplot(1, 2, 2)
    Phylo.draw(tree2, do_show=False, axes=ax2)
    ax2.set_title(tree2_path.stem)

    plt.tight_layout()
    plt.savefig(str(out_png))
    plt.close(fig)
    print(f"[PNG] Side-by-side tree rendered → {out_png}")


def render_tree_png(newick: Path, out_png: Path, title="Tree"):
    """
    Render a single phylogenetic tree into a PNG image.
    Handles multiple trees in PHYLIP output by using the first tree.
    """
    if out_png.exists():
        print(f"[RESUME] PNG exists → Skipping")
        return

    # Parse all trees but only use the first one
    trees = list(Phylo.parse(str(newick), "newick"))
    if not trees:
        print(f"⚠️ No trees found in {newick}")
        return

    tree = trees[0]  # take first tree

    fig = plt.figure(figsize=(6, 6))
    ax = fig.add_subplot(1, 1, 1)

    Phylo.draw(tree, do_show=False, axes=ax)
    ax.set_title(title)

    plt.tight_layout()
    plt.savefig(str(out_png))
    plt.close(fig)
    print(f"[PNG] Tree rendered → {out_png}")

# ------------------------------
# YOUR ORIGINAL compare_trees, analyze_results, likelihood tests
# ------------------------------
# ⚠️ UNCHANGED — COPY THEM EXACTLY FROM YOUR ORIGINAL CODE BELOW THIS POINT


# ------------------------------
# Likelihood model-fit tests
# ------------------------------
def run_likelihood_tests(aln_fa: Path, tree_files: list[Path]) -> dict:
    """
    Run IQ-TREE SH/KH/AU tests on given trees and return parsed results.
    Assumes tree_files are sanitized.
    """
    if not shutil.which("iqtree2"):
        return {"error": "iqtree2 not installed"}

    # Create tree set file (only valid Newick trees)
    aln_fa = Path(aln_fa)
    all_trees = aln_fa.with_suffix(".treeset.nwk")

    with open(all_trees, "w") as out:
      for t in tree_files:
          try:
              tree = Tree(str(t), format=1)
              newick = tree.write(format=1)

              if not newick.endswith(";"):
                  newick += ";"

              if newick.startswith("("):
                  out.write(newick + "\n")

          except Exception as e:
              print(f"{t}: {e}")

    prefix = str(aln_fa) + ".liketest"

    # Run IQ-TREE
    cmd = (
        f"iqtree2 -s {aln_fa} "
        f"-z {all_trees} "
        f"-zb 1000 -au "
        f"-nt AUTO -redo "
        f"-pre {prefix} "
        f"-quiet"
    )
    run_cmd(cmd)

    results = {}
    iqtree_out = Path(prefix + ".iqtree")
    if iqtree_out.exists():
        with open(iqtree_out) as f:
            lines = [l.strip() for l in f if l.strip()]

        header = []
        for line in lines:
            if line.startswith("Tree"):
                header = line.split()
                continue
            if header and line[0].isdigit():
                parts = line.split()
                tree_id = parts[0]
                results[tree_id] = dict(zip(header[1:], parts[1:]))

    return results

# ------------------------------
# Analysis & Summary
# ------------------------------
def analyze_results(table_data, aln_likelihoods, artifacts, aln_files, aln_to_trees, outdir):
    print("\n\n📊 Result Analysis and Conclusion 📊")
    print("-----------------------------------")
    print("\n### 1. Explanation of the Table")
    print("---------------------------------")
    print("The table compares all generated phylogenetic trees using a combination of methods (e.g., MAFFT+NJ, MUSCLE+ML).")
    print("- **RF (Robinson-Foulds) Distance**: A measure of the topological difference between two trees. A lower value means the trees are more similar.")
    print("- **Normalized RF**: The RF distance scaled to a 0-1 range. Lower is better.")
    print("- **BranchLenCorr**: The correlation coefficient of branch lengths. A value closer to 1 means branch lengths are highly similar.")
    print("- **Support Values**: Indicates the number of internal branches with bootstrap support values. A higher number suggests more confidence in the tree's internal structure.")
    print("- **Likelihood Tests (AU/KH/SH)**: Statistical tests from IQ-TREE to evaluate how well each tree fits the sequence data. For each alignment, the tree with the highest AU p-value (closest to 1) is the statistically best-fit tree. A p-value < 0.05 indicates the tree can be rejected.")

    print("\n### 2. Identifying the Best Tree(s)")
    print("----------------------------------")
    best_likelihood_tree_path = "N/A"
    best_likelihood_value = -1.0
    best_likelihood_method = "N/A"
    best_topology_pair = "N/A"
    lowest_norm_rf = float('inf')
    best_pair_img = "N/A"

    # Find the best likelihood tree and its method
    for aln_name, res in aln_likelihoods.items():
        if "error" in res:
            continue

        # We need to link the tree ID (T1, T2, etc.) to the method name.
        tree_methods_for_aln = []
        for m, inf, tfile in artifacts:
            if m == aln_name:
                tree_methods_for_aln.append((m, inf))

        for tid, vals in res.items():
            if 'p-AU' in vals and vals['p-AU'] != '?':
                au_val = float(vals['p-AU'])
                if au_val > best_likelihood_value:
                    best_likelihood_value = au_val
                    method_index = int(tid) - 1
                    if method_index < len(tree_methods_for_aln):
                        m, inf = tree_methods_for_aln[method_index]
                        best_likelihood_method = f"{m}+{inf}"
                        best_likelihood_tree_path = outdir / f"{m}_{inf}.png"

    # Find the best topological similarity and its image file
    for row in table_data:
        norm_rf = float(row[2])
        if norm_rf < lowest_norm_rf:
            lowest_norm_rf = norm_rf
            best_topology_pair = row[0]
            m1, inf1 = best_topology_pair.split(" vs ")[0].split('+')
            m2, inf2 = best_topology_pair.split(" vs ")[1].split('+')
            best_pair_img = outdir / f"{m1}_{inf1}_VS_{m2}_{inf2}.png"

    print(f"The **statistically best-fit tree** is: **{best_likelihood_method}** (AU p-value: {best_likelihood_value:.4f})")
    print("This tree has the highest statistical support for its topology, as determined by the AU test.")
    print(f"\nThe **most topologically similar trees** are: **{best_topology_pair}** (Normalized RF: {lowest_norm_rf:.4f})")

    # ---------------------------------------------
    # Dynamic Summary Generation
    # ---------------------------------------------
    print("\n### 3. Quick Summary and Final Recommendation")
    print("---------------------------------------------")

    ml_au_values = []
    for aln_name, res in aln_likelihoods.items():
        if 'error' not in res:
            ml_au_values.extend([float(v.get('p-AU', 0)) for v in res.values() if 'ml' in best_likelihood_method])
    is_ml_best = all(val > 0.5 for val in ml_au_values) if ml_au_values else False

    nj_upgma_pairs = [row for row in table_data if ('nj' in row[0] or 'upgma' in row[0]) and 'ml' not in row[0]]
    distance_methods_are_similar = any(float(row[2]) < 0.5 for row in nj_upgma_pairs)

    ml_ml_pairs = [row for row in table_data if 'ml' in row[0] and len(row[0].split('ml')) > 2]
    ml_trees_are_consistent = all(float(row[3]) > 0.5 for row in ml_ml_pairs if row[3] != 'N/A')

    if is_ml_best:
        print("1. **Maximum Likelihood (ML)** trees consistently show the highest statistical support (AU p-values closest to 1). This confirms that ML is the most suitable inference method for this dataset, as it finds the most optimal topology to explain the evolutionary relationships.")

    if distance_methods_are_similar:
        print("2. The **Distance-based methods (NJ and UPGMA)** produce relatively similar trees to each other (low RF distances), but they are often statistically rejected by the likelihood tests (AU p-values near 0), indicating their topologies are less likely to be correct.")

    if ml_trees_are_consistent:
        print("3. When using the ML method, all tested alignment tools (**MAFFT**, **MUSCLE**, and **ClustalW**) produce trees with a high degree of consistency, as shown by their high branch length correlation. This suggests that for this dataset, the choice of alignment tool has a minimal impact on the final ML tree topology.")

    print(f"\n**Conclusion**: For generating the most accurate and statistically supported phylogenetic tree from this dataset, the best approach is to use the **Maximum Likelihood (ML) tree inference method**, paired with any of the tested Multiple Sequence Alignment methods.")

    # ---------------------------------------------
    # Image Location
    # ---------------------------------------------
    print("\n### 4. Locate Best Tree Images")
    print("------------------------------------")
    print(f"The best tree image is located at: {best_likelihood_tree_path}")
    print(f"The most topologically similar trees image is located at: {best_pair_img}")
    print(f"\nAll generated files are in the directory: {outdir}")

# ------------------------------
# Comparison helpers
# ------------------------------
def compare_trees(artifacts, outdir):
    print("\n🔎 Tree Comparison Results (pairwise):\n")

    if not HAVE_DENDROPY:
        print("[WARNING] DendroPy not available. Install with: pip install dendropy\n")
        return

    from dendropy.calculate import treecompare
    taxon_namespace = dendropy.TaxonNamespace()

    aln_to_trees = {}
    aln_files = {}
    for m, inf, tfile in artifacts:
        aln_file = tfile.parent / f"{m}.aln.fasta"
        aln_to_trees.setdefault(m, []).append(tfile)
        aln_files[m] = aln_file

    aln_likelihoods = {}
    sanitized_tfiles_all = {}

    # SANITIZE alignments and trees per method
    for aln, tfiles in aln_to_trees.items():
        aln_file = aln_files[aln]
        sanitized_aln = sanitize_alignment(aln_file)

        sanitized_tfiles = []
        for tf in tfiles:
          try:
              tf_path = Path(tf)

              # Extract first valid Newick tree
              clean_file = tf_path.with_name(tf_path.stem + ".clean.nwk")

              newick_line = None
              with open(tf_path) as f:
                content = f.read()

              # Remove spaces and newlines
              content = content.replace("\n", "").replace(" ", "")

              # Find tree (everything between first '(' and last ';')
              start = content.find("(")
              end = content.rfind(";")

              if start == -1 or end == -1:
                  raise RuntimeError(f"No Newick tree found in {tf_path}")

              newick_line = content[start:end+1]

              # Write cleaned Newick tree
              with open(clean_file, "w") as out:
                  out.write(newick_line + "\n")

              # Parse tree safely
              tree = Tree(newick_line, format=1)

              sanitized_file = tf_path.with_name(tf_path.stem + ".sanitized.nwk")

              # Normalize leaf names
              for leaf in tree.iter_leaves():
                  leaf.name = "".join(c for c in leaf.name if c.isalnum() or c in "_-")

              tree.write(outfile=str(sanitized_file), format=1)

              sanitized_tfiles.append(sanitized_file)

          except Exception as e:
              print(f"{tf}: {e}")

        sanitized_tfiles_all[aln] = sanitized_tfiles

        # Optional: taxa mismatch check
        aln_taxa = set(record.id for record in SeqIO.parse(sanitized_aln, "fasta"))
        for tf in sanitized_tfiles:
            tree_taxa = set(Tree(str(tf), format=1).get_leaf_names())
            if aln_taxa != tree_taxa:
                print(f"⚠ Warning: Taxa mismatch in tree {tf}")

        # Run likelihood tests for this alignment
        aln_likelihoods[aln] = run_likelihood_tests(sanitized_aln, sanitized_tfiles)

    # PAIRWISE tree comparison
    headers = ["Trees Compared", "RF", "Norm RF", "BranchLenCorr", "Support Values", "Likelihood Tests (AU/KH/SH)"]
    table_data = []

    for (m1, inf1, t1_file), (m2, inf2, t2_file) in combinations(artifacts, 2):
        dp_tree1 = dendropy.Tree.get(path=str(t1_file),
                                     schema="newick",
                                     taxon_namespace=taxon_namespace,
                                     preserve_underscores=True)
        dp_tree2 = dendropy.Tree.get(path=str(t2_file),
                                     schema="newick",
                                     taxon_namespace=taxon_namespace,
                                     preserve_underscores=True)

        rf = treecompare.unweighted_robinson_foulds_distance(dp_tree1, dp_tree2)
        n_leaves = len(dp_tree1.leaf_nodes())
        max_rf = 2 * (n_leaves - 3) if n_leaves > 3 else 1
        norm_rf = rf / max_rf

        blens1 = [e.length for e in dp_tree1.edges() if e.length is not None]
        blens2 = [e.length for e in dp_tree2.edges() if e.length is not None]
        corr = "N/A"
        if blens1 and blens2 and len(blens1) == len(blens2) and HAVE_SCIPY:
            corr = f"{pearsonr(blens1, blens2)[0]:.3f}"

        sup1 = [n.label for n in dp_tree1.internal_nodes() if n.label]
        sup2 = [n.label for n in dp_tree2.internal_nodes() if n.label]
        support_info = f"{len(sup1)} vs {len(sup2)}" if sup1 or sup2 else "N/A"

        aln_name = m1
        like = "N/A"
        if aln_name in aln_likelihoods:
            res = aln_likelihoods[aln_name]
            if "error" in res:
                like = res["error"]
            else:
                like = " / ".join(
                    f"T{tid}:AU={vals.get('p-AU','?')},KH={vals.get('p-KH','?')},SH={vals.get('p-SH','?')}"
                    for tid, vals in res.items()
                )

        table_data.append([
            f"{m1}+{inf1} vs {m2}+{inf2}",
            str(rf),
            f"{norm_rf:.3f}",
            str(corr),
            support_info,
            like
        ])

        render_tree_side_by_side(t1_file, t2_file, outdir / f"{m1}_{inf1}_VS_{m2}_{inf2}.png")


    # Print the table
    print("---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------")
    print(f"{headers[0]:<40} | {headers[1]:<5} | {headers[2]:<10} | {headers[3]:<15} | {headers[4]:<18} | {headers[5]:<65}")
    print("---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------")
    for row in table_data:
        print(f"{row[0]:<40} | {row[1]:<5} | {row[2]:<10} | {row[3]:<15} | {row[4]:<18} | {row[5]:<65}")
    print("---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------")

    analyze_results(table_data, aln_likelihoods, artifacts, aln_files, aln_to_trees, outdir)

# ------------------------------
# Pipeline
# ------------------------------
def pipeline(fasta: Path, molecule: str, outdir: Path,
             msa_methods, infer_methods, build_all=True, render=True):
    outdir.mkdir(parents=True, exist_ok=True)
    msa_grid = list(MSA_FUNCS.keys()) if build_all else msa_methods
    infer_grid = list(INFER_FUNCS.keys()) if build_all else infer_methods

    artifacts = []
    for m in msa_grid:
        aln_file = outdir / f"{m}.aln.fasta"
        MSA_FUNCS[m](fasta, aln_file)
        print(f"Alignment done: {m} -> {aln_file}")

        # ✅ Normalize taxa after alignment
        normalize_fasta_taxa(aln_file)

        for inf in infer_grid:
            tree_file = outdir / f"{m}_{inf}.nwk"
            INFER_FUNCS[inf](aln_file, molecule, tree_file)
            print(f"Tree done: {m}+{inf} -> {tree_file}")
            if render:
                render_tree_png(tree_file, outdir / f"{m}_{inf}.png", title=f"{m}+{inf}")
            artifacts.append((m, inf, tree_file))
    return artifacts

# ------------------------------
# Interactive main
# ------------------------------

def main():
    print("\n🌱 Phylogenetic Pipeline (Drive + Resume Enabled)\n")

    fasta_path = Path(input("Enter path to FASTA file: ").strip())
    molecule = input("Sequence type (dna/protein) [auto-detect if blank]: ").strip()
    if not molecule:
        molecule = guess_molecule_from_fasta(fasta_path)
    print(f"Detected molecule type: {molecule}")

    msa_choice = input("MSA method (mafft, muscle, clustalw, all): ").strip().lower() or "mafft"
    infer_choice = input("Inference (nj, upgma, ml, mp, all): ").strip().lower() or "ml"
    build_all = input("Run full 4x3 grid? (y/n): ").strip().lower() == "y"
    render = input("Render PNG trees? (y/n): ").strip().lower() == "y"

    base_drive = Path("/content/drive/MyDrive/PhyloPipelineRuns")
    base_drive.mkdir(parents=True, exist_ok=True)

    dataset_name = fasta_path.stem
    dataset_folders = sorted(base_drive.glob(f"{dataset_name}_*"))

    if dataset_folders:
      choice = input("Resume last run for this dataset? (y/n): ").strip().lower()
      if choice == "y":
        outdir = dataset_folders[-1]
        print(f"🔁 Resuming dataset run: {outdir}")
      else:
        outdir = base_drive / f"{dataset_name}_{timestamp()}"
        print(f"🆕 Creating new run: {outdir}")
    else:
      outdir = base_drive / f"{dataset_name}_{timestamp()}"
      print(f"🆕 Creating new run: {outdir}")

    artifacts = pipeline(
        fasta_path, molecule, outdir,
        list(MSA_FUNCS.keys()) if msa_choice == "all" else [msa_choice],
        list(INFER_FUNCS.keys()) if infer_choice == "all" else [infer_choice],
        build_all=build_all, render=render
    )

    print("\n✅ Pipeline finished! Results in:", outdir)

    if input("Compare all generated trees? (y/n): ").strip().lower() == "y":
        compare_trees(artifacts, outdir)

if __name__ == "__main__":
    main()

Overwriting phylo_pipeline1.py


In [ ]:
#----MP (3)

%%writefile phylo_pipeline1.py
#!/usr/bin/env python3

from Bio.Phylo.TreeConstruction import DistanceCalculator, DistanceTreeConstructor
import os, sys, shutil, subprocess, datetime as dt
from pathlib import Path
from itertools import combinations
from Bio import AlignIO, Phylo, SeqIO
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from ete3 import Tree

try:
    from scipy.stats import pearsonr
    HAVE_SCIPY = True
except:
    HAVE_SCIPY = False

try:
    import dendropy
    from dendropy.calculate import treecompare
    HAVE_DENDROPY = True
except:
    HAVE_DENDROPY = False

# ===============================
# 🔧 CONFIG
# ===============================
MIN_SEQ_LENGTH = 500
BOOTSTRAP = 100

# ===============================
# 🧠 HELPER: Run command safely
# ===============================
def run_cmd(cmd):
    print(f"[CMD] {cmd}")
    result = subprocess.run(cmd, shell=True)
    if result.returncode != 0:
        raise RuntimeError(f"Command failed: {cmd}")

# ===============================
# 🧹 STEP 1: Filter short sequences
# ===============================
def filter_sequences(input_fasta, output_fasta):
    print("[FILTER] Removing short sequences...")
    records = []

    for rec in SeqIO.parse(input_fasta, "fasta"):
        if len(rec.seq) >= MIN_SEQ_LENGTH:
            records.append(rec)
        else:
            print(f"❌ Removed: {rec.id} (length {len(rec.seq)})")

    SeqIO.write(records, output_fasta, "fasta")
    print(f"✅ Filtered sequences saved: {output_fasta}")

# Remove sequences with >50% gaps
def filter_sequences(aln_file, output_file, threshold=0.5):
    from Bio import AlignIO
    alignment = AlignIO.read(aln_file, "fasta")

    filtered = []
    for record in alignment:
        gap_ratio = record.seq.count('-') / len(record.seq)
        if gap_ratio < threshold:
            filtered.append(record)

    AlignIO.write(filtered, output_file, "fasta")

# ===============================
# 🧪 STEP 2: Validate alignment
# ===============================
def validate_alignment(file):
    print("[CHECK] Validating alignment...")
    aln = AlignIO.read(file, "fasta")

    if len(aln) < 4:
        raise ValueError("Too few sequences for tree inference")

    for rec in aln:
        if len(rec.seq) == 0:
            raise ValueError(f"Empty sequence: {rec.id}")

    print("✅ Alignment valid")

# ------------------------------
# Utils
# ------------------------------
def run_cmd(cmd: str, check: bool = True):
    print(f"[CMD] {cmd}")
    result = subprocess.run(
        cmd,
        shell=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True
    )

    if check and result.returncode != 0:
        print(result.stderr)
        raise RuntimeError(f"Command failed: {cmd}")

    return result.stdout

def timestamp():
    return dt.datetime.now().strftime("%Y%m%d-%H%M%S")

def guess_molecule_from_fasta(fasta_path):
    for rec in SeqIO.parse(fasta_path, "fasta"):
        seq = str(rec.seq).upper()
        if set(seq) <= set("ACGTN-"):
            return "dna"
        else:
            return "protein"
    return "dna"

def clean_newick_tree(tree_path: Path):
    """
    Clean problematic Newick trees (especially from PHYLIP MP)
    """
    try:
        tree = Phylo.read(str(tree_path), "newick")
        Phylo.write(tree, str(tree_path), "newick")
    except Exception:
        print(f"[WARNING] Could not clean tree: {tree_path}")

# ------------------------------
# GLOBAL TAXA MAPPING (IMPORTANT FIX)
# ------------------------------
GLOBAL_TAXA_MAP = {}

def create_global_taxa_map(fasta_file):
    global GLOBAL_TAXA_MAP
    GLOBAL_TAXA_MAP = {}
    for i, rec in enumerate(SeqIO.parse(fasta_file, "fasta"), 1):
        GLOBAL_TAXA_MAP[rec.id] = f"T{i}"

def apply_taxa_map_to_alignment(in_fa, out_fa):
    with open(out_fa, "w") as out:
        for rec in SeqIO.parse(in_fa, "fasta"):
            rec.id = GLOBAL_TAXA_MAP.get(rec.id, rec.id)
            rec.description = ""
            SeqIO.write(rec, out, "fasta")

def apply_taxa_map_to_tree(tree_file, out_file):
    tree = Tree(open(tree_file).read(), format=1)
    reverse_map = {v: v for v in GLOBAL_TAXA_MAP.values()}

    for leaf in tree:
        name = leaf.name
        if name in reverse_map:
            leaf.name = name

    tree.write(outfile=out_file)

def sanitize_tree(tree_file):
    tree_file = Path(tree_file)
    out = tree_file.with_name(tree_file.stem + ".sanitized.nwk")

    content = tree_file.read_text()

    # ✅ Remove problematic patterns
    content = content.replace("\n", "").replace(" ", "")
    content = content.replace("[", "").replace("]", "")   # FIX PHYLIP
    content = content.replace("Inner", "N")               # FIX BioPython

    start = content.find("(")
    end = content.rfind(";")

    if start == -1 or end == -1:
        raise RuntimeError(f"No valid Newick in {tree_file}")

    newick = content[start:end+1]

    tree = Tree(newick, format=1)

    # ✅ Normalize names
    for leaf in tree:
        leaf.name = leaf.name.strip()

    tree.write(outfile=str(out))
    return out

def sanitize_alignment(aln_file):
    out = aln_file.with_suffix(".sanitized.fasta")

    with open(out, "w") as fout:
        for rec in SeqIO.parse(aln_file, "fasta"):
            rec.id = rec.id.replace(" ", "_")
            rec.description = ""
            SeqIO.write(rec, fout, "fasta")

    return out

def normalize_taxon_name(name):
    return "".join(c for c in name if c.isalnum() or c in "_-")

# ------------------------------
# MSA wrappers (with resume)
# ------------------------------
def run_mafft(in_fa: Path, out_fa: Path):
    if out_fa.exists():
        print(f"[RESUME] MAFFT alignment exists → Skipping")
        return
    run_cmd(f"mafft --auto {in_fa} > {out_fa}")

def run_muscle(in_fa: Path, out_fa: Path):
    if out_fa.exists():
        print(f"[RESUME] MUSCLE alignment exists → Skipping")
        return
    run_cmd(f"muscle -in {in_fa} -out {out_fa}")

def run_clustalw(in_fa: Path, out_fa: Path):
    if out_fa.exists():
        print(f"[RESUME] ClustalW alignment exists → Skipping")
        return
    run_cmd(f"clustalw -infile={in_fa}")
    aln_file = str(in_fa).replace(".fasta", ".aln")
    if os.path.exists(aln_file):
        AlignIO.convert(aln_file, "clustal", out_fa, "fasta")

MSA_FUNCS = {"mafft": run_mafft, "muscle": run_muscle, "clustalw": run_clustalw}

# ------------------------------
# Tree inference (with resume)
# ------------------------------
def infer_nj(aln_fa: Path, molecule: str, out_newick: Path):
    if out_newick.exists():
        print(f"[RESUME] NJ tree exists → Skipping")
        return
    aln = AlignIO.read(str(aln_fa), "fasta")
    model = "identity" if molecule == "dna" else "blosum62"
    dm = DistanceCalculator(model).get_distance(aln)
    tree = DistanceTreeConstructor().nj(dm)
    Phylo.write(tree, str(out_newick), "newick")

def infer_upgma(aln_fa: Path, molecule: str, out_newick: Path):
    if out_newick.exists():
        print(f"[RESUME] UPGMA tree exists → Skipping")
        return
    aln = AlignIO.read(str(aln_fa), "fasta")
    model = "identity" if molecule == "dna" else "blosum62"
    dm = DistanceCalculator(model).get_distance(aln)
    tree = DistanceTreeConstructor().upgma(dm)
    Phylo.write(tree, str(out_newick), "newick")

def infer_ml_iqtree(aln_fa: Path, molecule: str, out_newick: Path):
    if out_newick.exists():
        print(f"[RESUME] ML tree exists → Skipping")
        return
    aln_fasta = str(aln_fa)
    cmd = f"iqtree2 -s {aln_fasta} -m TEST -nt AUTO -quiet"
    run_cmd(cmd)
    iqtree_file = aln_fasta + ".treefile"
    if os.path.exists(iqtree_file):
        shutil.copy(iqtree_file, out_newick)

    sanitized = sanitize_tree(out_newick)
    shutil.move(sanitized, out_newick)


def run_phylip_mp(aln_fa: Path, molecule: str, out_newick: Path):
    """
    Run PHYLIP DNAPARS (Maximum Parsimony) safely and non-interactively.
    Compatible with pipeline inference interface.
    """

    if out_newick.exists():
        print(f"[RESUME] MP tree exists → Skipping")
        return

    workdir = out_newick.parent
    clean_fasta = workdir / "clean_alignment.fasta"
    phylip_file = workdir / "alignment.phy"

    print("[MP] Cleaning alignment for PHYLIP...")

    valid = set("ACGTN-")

    # Clean sequences: replace any invalid base with 'N'
    with open(aln_fa) as fin, open(clean_fasta, "w") as fout:
        for line in fin:
            if line.startswith(">"):
                fout.write(line)
            else:
                seq = line.strip().upper()
                seq = "".join([c if c in valid else "N" for c in seq])
                fout.write(seq + "\n")

    print("[MP] Converting FASTA → PHYLIP (sequential format)")

    # Use strict PHYLIP sequential format to avoid DNAPARS errors
    AlignIO.convert(
        str(clean_fasta),
        "fasta",
        str(phylip_file),
        "phylip"  # strict PHYLIP, not relaxed
    )

    infile = workdir / "infile"
    shutil.copy(phylip_file, infile)

    print("[MP] Running DNAPARS (non-interactive)...")

    # Remove any leftover files
    for f in ["outfile", "outtree"]:
        p = workdir / f
        if p.exists():
            p.unlink()

    # Prepare command to run DNAPARS non-interactively
    # Automatically accepts defaults, saves tree to outtree
    cmd = (
        f"cd {workdir} && "
        f"printf 'Y\n' | /usr/lib/phylip/bin/dnapars"
    )

    try:
        subprocess.run(cmd, shell=True, check=True)
    except subprocess.CalledProcessError:
        print("⚠️ DNAPARS failed")
        return

    phylip_tree = workdir / "outtree"

    if not phylip_tree.exists():
        print("⚠️ DNAPARS did not produce tree")
        return

    shutil.copy(phylip_tree, out_newick)
    print(f"[MP] Tree generated → {out_newick}")

    try:
      sanitized = sanitize_tree(out_newick)
      shutil.move(sanitized, out_newick)
    except Exception as e:
      print(f"{e}")


INFER_FUNCS = {"nj": infer_nj, "upgma": infer_upgma, "ml": infer_ml_iqtree, "mp": run_phylip_mp}

# ------------------------------
# Visualization (with resume)
# ------------------------------
def render_tree_side_by_side(tree1_path: Path, tree2_path: Path, out_png: Path):
    """
    Render two trees side-by-side into one PNG file.
    Handles multiple trees in PHYLIP output by using the first tree.
    """
    if out_png.exists():
        print(f"[RESUME] Side-by-side PNG exists → Skipping")
        return

    # Parse trees safely (pick first tree if multiple)
    trees1 = list(Phylo.parse(str(tree1_path), "newick"))
    trees2 = list(Phylo.parse(str(tree2_path), "newick"))

    if not trees1 or not trees2:
        print(f"⚠️ No trees found in {tree1_path} or {tree2_path}")
        return

    tree1 = trees1[0]
    tree2 = trees2[0]

    fig = plt.figure(figsize=(12, 6))

    ax1 = fig.add_subplot(1, 2, 1)
    Phylo.draw(tree1, do_show=False, axes=ax1)
    ax1.set_title(tree1_path.stem)

    ax2 = fig.add_subplot(1, 2, 2)
    Phylo.draw(tree2, do_show=False, axes=ax2)
    ax2.set_title(tree2_path.stem)

    plt.tight_layout()
    plt.savefig(str(out_png))
    plt.close(fig)
    print(f"[PNG] Side-by-side tree rendered → {out_png}")


def render_tree_png(newick: Path, out_png: Path, title="Tree"):
    """
    Render a single phylogenetic tree into a PNG image.
    Handles multiple trees in PHYLIP output by using the first tree.
    """
    if out_png.exists():
        print(f"[RESUME] PNG exists → Skipping")
        return

    # Parse all trees but only use the first one
    trees = list(Phylo.parse(str(newick), "newick"))
    if not trees:
        print(f"⚠️ No trees found in {newick}")
        return

    tree = trees[0]  # take first tree

    fig = plt.figure(figsize=(6, 6))
    ax = fig.add_subplot(1, 1, 1)

    Phylo.draw(tree, do_show=False, axes=ax)
    ax.set_title(title)

    plt.tight_layout()
    plt.savefig(str(out_png))
    plt.close(fig)
    print(f"[PNG] Tree rendered → {out_png}")

# ------------------------------
# Likelihood model-fit tests (STABLE)
# ------------------------------

def run_likelihood_tests(aln_fa: Path, tree_files: list[Path]) -> dict:
    if not shutil.which("iqtree2"):
        return {"error": "iqtree2 not installed"}

    # Combine all trees into one file
    all_trees = aln_fa.with_suffix(".treeset.nwk")
    with open(all_trees, "w") as out:
        for t in tree_files:
            if t.exists():
                with open(t) as f:
                    out.write(f.read().strip() + "\n")

    prefix = str(aln_fa) + ".liketest"

    # ✅ STABLE COMMAND (NO AU TEST)
    cmd = (
        f"iqtree2 -s {aln_fa} "
        f"-z {all_trees} "
        f"-n 0 "
        f"-m GTR "
        f"-nt 1 "
        f"-safe "
        f"-redo "
        f"-pre {prefix} "
        f"-quiet"
    )

    try:
        run_cmd(cmd)
    except Exception:
        print("⚠️ IQ-TREE failed → skipping likelihood test")
        return {"error": "IQ-TREE crashed"}

    # ✅ PARSE OUTPUT
    iqtree_out = Path(prefix + ".iqtree")
    results = {}

    if iqtree_out.exists():
        with open(iqtree_out) as f:
            lines = [l.strip() for l in f if l.strip()]

        header = []
        for line in lines:
            if line.startswith("Tree"):
                header = line.split()
                continue

            if header and line[0].isdigit():
                parts = line.split()
                tree_id = parts[0]
                results[tree_id] = dict(zip(header[1:], parts[1:]))

    return results

# ------------------------------
# Analysis & Summary
# ------------------------------
def analyze_results(table_data, aln_likelihoods, artifacts, aln_files, aln_to_trees, outdir):
    print("\n\n📊 Result Analysis and Conclusion 📊")
    print("-----------------------------------")
    print("\n### 1. Explanation of the Table")
    print("---------------------------------")
    print("The table compares all generated phylogenetic trees using a combination of methods (e.g., MAFFT+NJ, MUSCLE+ML).")
    print("- **RF (Robinson-Foulds) Distance**: A measure of the topological difference between two trees. A lower value means the trees are more similar.")
    print("- **Normalized RF**: The RF distance scaled to a 0-1 range. Lower is better.")
    print("- **BranchLenCorr**: The correlation coefficient of branch lengths. A value closer to 1 means branch lengths are highly similar.")
    print("- **Support Values**: Indicates the number of internal branches with bootstrap support values. A higher number suggests more confidence in the tree's internal structure.")
    print("- **Likelihood Tests (AU/KH/SH)**: Statistical tests from IQ-TREE to evaluate how well each tree fits the sequence data. For each alignment, the tree with the highest AU p-value (closest to 1) is the statistically best-fit tree. A p-value < 0.05 indicates the tree can be rejected.")

    print("\n### 2. Identifying the Best Tree(s)")
    print("----------------------------------")
    best_likelihood_tree_path = "N/A"
    best_likelihood_value = -1.0
    best_likelihood_method = "N/A"
    best_topology_pair = "N/A"
    lowest_norm_rf = float('inf')
    best_pair_img = "N/A"

    # Find the best likelihood tree and its method
    for aln_name, res in aln_likelihoods.items():
        if "error" in res:
            continue

        # We need to link the tree ID (T1, T2, etc.) to the method name.
        tree_methods_for_aln = []
        for m, inf, tfile in artifacts:
            if m == aln_name:
                tree_methods_for_aln.append((m, inf))

        for tid, vals in res.items():
            if 'p-AU' in vals and vals['p-AU'] != '?':
                au_val = float(vals['p-AU'])
                if au_val > best_likelihood_value:
                    best_likelihood_value = au_val
                    method_index = int(tid) - 1
                    if method_index < len(tree_methods_for_aln):
                        m, inf = tree_methods_for_aln[method_index]
                        best_likelihood_method = f"{m}+{inf}"
                        best_likelihood_tree_path = outdir / f"{m}_{inf}.png"

    # Find the best topological similarity and its image file
    for row in table_data:
        norm_rf = float(row[2])
        if norm_rf < lowest_norm_rf:
            lowest_norm_rf = norm_rf
            best_topology_pair = row[0]
            m1, inf1 = best_topology_pair.split(" vs ")[0].split('+')
            m2, inf2 = best_topology_pair.split(" vs ")[1].split('+')
            best_pair_img = outdir / f"{m1}_{inf1}_VS_{m2}_{inf2}.png"

    print(f"The **statistically best-fit tree** is: **{best_likelihood_method}** (AU p-value: {best_likelihood_value:.4f})")
    print("This tree has the highest statistical support for its topology, as determined by the AU test.")
    print(f"\nThe **most topologically similar trees** are: **{best_topology_pair}** (Normalized RF: {lowest_norm_rf:.4f})")

    # ---------------------------------------------
    # Dynamic Summary Generation
    # ---------------------------------------------
    print("\n### 3. Quick Summary and Final Recommendation")
    print("---------------------------------------------")

    ml_au_values = []
    for aln_name, res in aln_likelihoods.items():
        if 'error' not in res:
            ml_au_values.extend([float(v.get('p-AU', 0)) for v in res.values() if 'ml' in best_likelihood_method])
    is_ml_best = all(val > 0.5 for val in ml_au_values) if ml_au_values else False

    nj_upgma_pairs = [row for row in table_data if ('nj' in row[0] or 'upgma' in row[0]) and 'ml' not in row[0]]
    distance_methods_are_similar = any(float(row[2]) < 0.5 for row in nj_upgma_pairs)

    ml_ml_pairs = [row for row in table_data if 'ml' in row[0] and len(row[0].split('ml')) > 2]
    ml_trees_are_consistent = all(float(row[3]) > 0.5 for row in ml_ml_pairs if row[3] != 'N/A')

    if is_ml_best:
        print("1. **Maximum Likelihood (ML)** trees consistently show the highest statistical support (AU p-values closest to 1). This confirms that ML is the most suitable inference method for this dataset, as it finds the most optimal topology to explain the evolutionary relationships.")

    if distance_methods_are_similar:
        print("2. The **Distance-based methods (NJ and UPGMA)** produce relatively similar trees to each other (low RF distances), but they are often statistically rejected by the likelihood tests (AU p-values near 0), indicating their topologies are less likely to be correct.")

    if ml_trees_are_consistent:
        print("3. When using the ML method, all tested alignment tools (**MAFFT**, **MUSCLE**, and **ClustalW**) produce trees with a high degree of consistency, as shown by their high branch length correlation. This suggests that for this dataset, the choice of alignment tool has a minimal impact on the final ML tree topology.")

    print(f"\n**Conclusion**: For generating the most accurate and statistically supported phylogenetic tree from this dataset, the best approach is to use the **Maximum Likelihood (ML) tree inference method**, paired with any of the tested Multiple Sequence Alignment methods.")

    # ---------------------------------------------
    # Image Location
    # ---------------------------------------------
    print("\n### 4. Locate Best Tree Images")
    print("------------------------------------")
    print(f"The best tree image is located at: {best_likelihood_tree_path}")
    print(f"The most topologically similar trees image is located at: {best_pair_img}")
    print(f"\nAll generated files are in the directory: {outdir}")

def validate_tree(tree_file):
    with open(tree_file) as f:
        tree = f.read()
    return "(" in tree and ")" in tree

## ------------------------------
# Comparison helpers (FINAL FIXED VERSION)
# ------------------------------
def compare_trees(artifacts, outdir):
    print("\n🔎 Tree Comparison Results (pairwise):\n")

    if not HAVE_DENDROPY:
        print("[WARNING] DendroPy not available. Install with: pip install dendropy\n")
        return

    from dendropy.calculate import treecompare
    taxon_namespace = dendropy.TaxonNamespace()

    # Organize trees per alignment
    aln_to_trees = {}
    aln_files = {}
    for m, inf, tfile in artifacts:
        aln_file = tfile.parent / f"{m}.aln.fasta"
        aln_to_trees.setdefault(m, []).append(tfile)
        aln_files[m] = aln_file

    aln_likelihoods = {}

    # ------------------------------
    # SANITIZATION + Likelihood
    # ------------------------------
    from Bio import SeqIO

    def filter_alignment(aln_file: Path, min_len=100, min_taxa=4):
        records = list(SeqIO.parse(aln_file, "fasta"))

        filtered = [r for r in records if len(r.seq) >= min_len]

        if len(filtered) < min_taxa:
            print("⚠️ Too few sequences after filtering, skipping likelihood...")
            return None

        out_file = aln_file.with_suffix(".filtered.fasta")
        SeqIO.write(filtered, out_file, "fasta")
        return out_file

    for aln, tfiles in aln_to_trees.items():
        aln_file = aln_files[aln]

        # sanitize alignment
        sanitized_aln = sanitize_alignment(aln_file)

        sanitized_tfiles = []
        for tf in tfiles:
            try:
                sanitized = sanitize_tree(tf)
                sanitized_tfiles.append(Path(sanitized))
            except Exception as e:
                print(f"{tf}: {e}")

        # filter alignment safely
        filtered_aln = filter_alignment(sanitized_aln)

        if filtered_aln is None:
            aln_likelihoods[aln] = {"error": "Too few sequences"}
            continue  # ✅ DO NOT RETURN

        # run likelihood safely
        try:
            aln_likelihoods[aln] = run_likelihood_tests(filtered_aln, sanitized_tfiles)
        except Exception as e:
            print(f"⚠️ IQ-TREE failed: {e}")
            aln_likelihoods[aln] = {"error": "IQ-TREE failed"}

    # ------------------------------
    # PAIRWISE COMPARISON
    # ------------------------------
    headers = ["Trees Compared", "RF", "Norm RF", "BranchLenCorr", "Support Values", "Likelihood Tests"]
    table_data = []

    from itertools import combinations

    for (m1, inf1, t1_file), (m2, inf2, t2_file) in combinations(artifacts, 2):

        def safe_load_tree(path):
            try:
                t = dendropy.Tree.get(
                    path=str(path),
                    schema="newick",
                    taxon_namespace=taxon_namespace,
                    preserve_underscores=True
                )

                # 🔥 CRITICAL FIX
                t.encode_bipartitions()

                return t
            except Exception as e:
                print(f"[WARNING] Skipping invalid tree: {path.name} ({e})")
                return None

        dp_tree1 = safe_load_tree(t1_file)
        dp_tree2 = safe_load_tree(t2_file)

        if dp_tree1 is None or dp_tree2 is None:
            continue

        # RF distance (SAFE)
        try:
            rf = treecompare.unweighted_robinson_foulds_distance(dp_tree1, dp_tree2)
        except Exception as e:
            print(f"⚠️ RF failed: {e}")
            rf = "NA"

        n_leaves = len(dp_tree1.leaf_nodes())
        max_rf = 2 * (n_leaves - 3) if n_leaves > 3 else 1
        norm_rf = rf / max_rf if isinstance(rf, int) else "NA"

        # Branch length correlation
        blens1 = [e.length for e in dp_tree1.edges() if e.length is not None]
        blens2 = [e.length for e in dp_tree2.edges() if e.length is not None]

        corr = "N/A"
        if blens1 and blens2 and len(blens1) == len(blens2) and HAVE_SCIPY:
            try:
                corr = f"{pearsonr(blens1, blens2)[0]:.3f}"
            except:
                corr = "NA"

        # Support values
        sup1 = [n.label for n in dp_tree1.internal_nodes() if n.label]
        sup2 = [n.label for n in dp_tree2.internal_nodes() if n.label]
        support_info = f"{len(sup1)} vs {len(sup2)}" if sup1 or sup2 else "N/A"

        # Likelihood info
        like = "N/A"
        if m1 in aln_likelihoods:
            res = aln_likelihoods[m1]
            if "error" in res:
                like = res["error"]
            else:
                like = "OK"

        # Store result
        table_data.append([
            f"{m1}+{inf1} vs {m2}+{inf2}",
            str(rf),
            str(norm_rf),
            str(corr),
            support_info,
            like
        ])

        # Render comparison image
        render_tree_side_by_side(
            t1_file,
            t2_file,
            outdir / f"{m1}_{inf1}_VS_{m2}_{inf2}.png"
        )

    print("\n✅ Comparison completed successfully!\n")

    # ------------------------------
    # PRINT TABLE
    # ------------------------------
    print("-" * 180)
    print(f"{headers[0]:<40} | {headers[1]:<5} | {headers[2]:<10} | {headers[3]:<15} | {headers[4]:<18} | {headers[5]:<65}")
    print("-" * 180)

    for row in table_data:
        print(f"{row[0]:<40} | {row[1]:<5} | {row[2]:<10} | {row[3]:<15} | {row[4]:<18} | {row[5]:<65}")

    print("-" * 180)

    # ------------------------------
    # FINAL ANALYSIS
    # ------------------------------
    analyze_results(table_data, aln_likelihoods, artifacts, aln_files, aln_to_trees, outdir)



# ------------------------------
# PIPELINE
# ------------------------------
def pipeline(fasta: Path, molecule: str, outdir: Path,
             msa_methods, infer_methods, build_all=True, render=True):

    outdir.mkdir(parents=True, exist_ok=True)

    msa_grid = list(MSA_FUNCS.keys()) if build_all else msa_methods
    infer_grid = list(INFER_FUNCS.keys()) if build_all else infer_methods

    artifacts = []

    create_global_taxa_map(fasta)

    for m in msa_grid:
        aln_file = outdir / f"{m}.aln.fasta"

        # ✅ Resume-safe MSA
        MSA_FUNCS[m](fasta, aln_file)
        print(f"[MSA] {m} → {aln_file}")

        # Apply taxa mapping to alignment
        tmp_aln = aln_file.with_suffix(".mapped.fasta")
        apply_taxa_map_to_alignment(aln_file, tmp_aln)
        shutil.move(tmp_aln, aln_file)

        for inf in infer_grid:
            tree_file = outdir / f"{m}_{inf}.nwk"

            # ✅ Resume-safe inference
            INFER_FUNCS[inf](aln_file, molecule, tree_file)
            print(f"[TREE] {m}+{inf} → {tree_file}")

            # Apply taxa mapping to tree
            tmp_tree = tree_file.with_suffix(".mapped.nwk")
            apply_taxa_map_to_tree(tree_file, tmp_tree)
            shutil.move(tmp_tree, tree_file)

            # ✅ Fix: clean MP trees ONLY after they exist
            if inf == "mp" and tree_file.exists():
                clean_newick_tree(tree_file)

            # ✅ Render (resume-safe inside function)
            if render:
                render_tree_png(
                    tree_file,
                    outdir / f"{m}_{inf}.png",
                    title=f"{m}+{inf}"
                )

            artifacts.append((m, inf, tree_file))

    return artifacts

# ------------------------------
# Interactive main
# ------------------------------

def main():
    print("\n🌱 Phylogenetic Pipeline (Drive + Resume Enabled)\n")

    fasta_path = Path(input("Enter path to FASTA file: ").strip())
    molecule = input("Sequence type (dna/protein) [auto-detect if blank]: ").strip()
    if not molecule:
        molecule = guess_molecule_from_fasta(fasta_path)
    print(f"Detected molecule type: {molecule}")

    msa_choice = input("MSA method (mafft, muscle, clustalw, all): ").strip().lower() or "mafft"
    infer_choice = input("Inference (nj, upgma, ml, mp, all): ").strip().lower() or "ml"
    build_all = input("Run full 4x3 grid? (y/n): ").strip().lower() == "y"
    render = input("Render PNG trees? (y/n): ").strip().lower() == "y"

    base_drive = Path("/content/drive/MyDrive/PhyloPipelineRuns")
    base_drive.mkdir(parents=True, exist_ok=True)

    dataset_name = fasta_path.stem
    dataset_folders = sorted(base_drive.glob(f"{dataset_name}_*"))

    if dataset_folders:
      choice = input("Resume last run for this dataset? (y/n): ").strip().lower()
      if choice == "y":
        outdir = dataset_folders[-1]
        print(f"🔁 Resuming dataset run: {outdir}")
      else:
        outdir = base_drive / f"{dataset_name}_{timestamp()}"
        print(f"🆕 Creating new run: {outdir}")
    else:
      outdir = base_drive / f"{dataset_name}_{timestamp()}"
      print(f"🆕 Creating new run: {outdir}")

    artifacts = pipeline(
        fasta_path, molecule, outdir,
        list(MSA_FUNCS.keys()) if msa_choice == "all" else [msa_choice],
        list(INFER_FUNCS.keys()) if infer_choice == "all" else [infer_choice],
        build_all=build_all, render=render
    )

    print("\n✅ Pipeline finished! Results in:", outdir)

    if input("Compare all generated trees? (y/n): ").strip().lower() == "y":
        compare_trees(artifacts, outdir)

if __name__ == "__main__":
    main()

Overwriting phylo_pipeline1.py


In [ ]:
!python3 phylo_pipeline1.py


🌱 Phylogenetic Pipeline (Drive + Resume Enabled)

Enter path to FASTA file: /content/Temnothorax_COI_10species_phyloSafe.fasta
Sequence type (dna/protein) [auto-detect if blank]: 
Detected molecule type: dna
MSA method (mafft, muscle, clustalw, all): all
Inference (nj, upgma, ml, mp, all): all
Run full 4x3 grid? (y/n): y
Render PNG trees? (y/n): y
Resume last run for this dataset? (y/n): n
🆕 Creating new run: /content/drive/MyDrive/PhyloPipelineRuns/Temnothorax_COI_10species_phyloSafe_20260410-042219
[CMD] mafft --auto /content/Temnothorax_COI_10species_phyloSafe.fasta > /content/drive/MyDrive/PhyloPipelineRuns/Temnothorax_COI_10species_phyloSafe_20260410-042219/mafft.aln.fasta
[MSA] mafft → /content/drive/MyDrive/PhyloPipelineRuns/Temnothorax_COI_10species_phyloSafe_20260410-042219/mafft.aln.fasta
[TREE] mafft+nj → /content/drive/MyDrive/PhyloPipelineRuns/Temnothorax_COI_10species_phyloSafe_20260410-042219/mafft_nj.nwk
[PNG] Tree rendered → /content/drive/MyDrive/PhyloPipelineRuns/Te

In [ ]:
#----MP (3)

%%writefile phylo_pipeline1.py
#!/usr/bin/env python3

from Bio.Phylo.TreeConstruction import DistanceCalculator, DistanceTreeConstructor
import os, sys, shutil, subprocess, datetime as dt
from pathlib import Path
from itertools import combinations
from Bio import AlignIO, Phylo, SeqIO
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from ete3 import Tree

try:
    from scipy.stats import pearsonr
    HAVE_SCIPY = True
except:
    HAVE_SCIPY = False

try:
    import dendropy
    from dendropy.calculate import treecompare
    HAVE_DENDROPY = True
except:
    HAVE_DENDROPY = False

# ===============================
# 🔧 CONFIG
# ===============================
MIN_SEQ_LENGTH = 500
BOOTSTRAP = 100

# ===============================
# 🧠 HELPER: Run command safely
# ===============================
def run_cmd(cmd):
    print(f"[CMD] {cmd}")
    result = subprocess.run(cmd, shell=True)
    if result.returncode != 0:
        raise RuntimeError(f"Command failed: {cmd}")

# ===============================
# 🧹 STEP 1: Filter short sequences
# ===============================
def filter_sequences(input_fasta, output_fasta):
    print("[FILTER] Removing short sequences...")
    records = []

    for rec in SeqIO.parse(input_fasta, "fasta"):
        if len(rec.seq) >= MIN_SEQ_LENGTH:
            records.append(rec)
        else:
            print(f"❌ Removed: {rec.id} (length {len(rec.seq)})")

    SeqIO.write(records, output_fasta, "fasta")
    print(f"✅ Filtered sequences saved: {output_fasta}")

# Remove sequences with >50% gaps
def filter_sequences(aln_file, output_file, threshold=0.5):
    from Bio import AlignIO
    alignment = AlignIO.read(aln_file, "fasta")

    filtered = []
    for record in alignment:
        gap_ratio = record.seq.count('-') / len(record.seq)
        if gap_ratio < threshold:
            filtered.append(record)

    AlignIO.write(filtered, output_file, "fasta")

# ===============================
# 🧪 STEP 2: Validate alignment
# ===============================
def validate_alignment(file):
    print("[CHECK] Validating alignment...")
    aln = AlignIO.read(file, "fasta")

    if len(aln) < 4:
        raise ValueError("Too few sequences for tree inference")

    for rec in aln:
        if len(rec.seq) == 0:
            raise ValueError(f"Empty sequence: {rec.id}")

    print("✅ Alignment valid")

# ------------------------------
# Utils
# ------------------------------
def run_cmd(cmd: str, check: bool = True):
    print(f"[CMD] {cmd}")
    result = subprocess.run(
        cmd,
        shell=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True
    )

    if check and result.returncode != 0:
        print(result.stderr)
        raise RuntimeError(f"Command failed: {cmd}")

    return result.stdout

def timestamp():
    return dt.datetime.now().strftime("%Y%m%d-%H%M%S")

def guess_molecule_from_fasta(fasta_path):
    for rec in SeqIO.parse(fasta_path, "fasta"):
        seq = str(rec.seq).upper()
        if set(seq) <= set("ACGTN-"):
            return "dna"
        else:
            return "protein"
    return "dna"

def clean_newick_tree(tree_path: Path):
    """
    Clean problematic Newick trees (especially from PHYLIP MP)
    """
    try:
        tree = Phylo.read(str(tree_path), "newick")
        Phylo.write(tree, str(tree_path), "newick")
    except Exception:
        print(f"[WARNING] Could not clean tree: {tree_path}")

# ------------------------------
# GLOBAL TAXA MAPPING (IMPORTANT FIX)
# ------------------------------
GLOBAL_TAXA_MAP = {}

def create_global_taxa_map(fasta_file):
    global GLOBAL_TAXA_MAP
    GLOBAL_TAXA_MAP = {}
    for i, rec in enumerate(SeqIO.parse(fasta_file, "fasta"), 1):
        GLOBAL_TAXA_MAP[rec.id] = f"T{i}"

def apply_taxa_map_to_alignment(in_fa, out_fa):
    with open(out_fa, "w") as out:
        for rec in SeqIO.parse(in_fa, "fasta"):
            rec.id = GLOBAL_TAXA_MAP.get(rec.id, rec.id)
            rec.description = ""
            SeqIO.write(rec, out, "fasta")

def apply_taxa_map_to_tree(tree_file, out_file):
    tree = Tree(open(tree_file).read(), format=1)
    reverse_map = {v: v for v in GLOBAL_TAXA_MAP.values()}

    for leaf in tree:
        name = leaf.name
        if name in reverse_map:
            leaf.name = name

    tree.write(outfile=out_file)

def sanitize_tree(tree_file):
    tree_file = Path(tree_file)
    out = tree_file.with_name(tree_file.stem + ".sanitized.nwk")

    content = tree_file.read_text()

    # ✅ Remove problematic patterns
    content = content.replace("\n", "").replace(" ", "")
    content = content.replace("[", "").replace("]", "")   # FIX PHYLIP
    content = content.replace("Inner", "N")               # FIX BioPython

    start = content.find("(")
    end = content.rfind(";")

    if start == -1 or end == -1:
        raise RuntimeError(f"No valid Newick in {tree_file}")

    newick = content[start:end+1]

    tree = Tree(newick, format=1)

    # ✅ Normalize names
    for leaf in tree:
        leaf.name = leaf.name.strip()

    tree.write(outfile=str(out))
    return out

def sanitize_alignment(aln_file):
    out = aln_file.with_suffix(".sanitized.fasta")

    with open(out, "w") as fout:
        for rec in SeqIO.parse(aln_file, "fasta"):
            rec.id = rec.id.replace(" ", "_")
            rec.description = ""
            SeqIO.write(rec, fout, "fasta")

    return out

def normalize_taxon_name(name):
    return "".join(c for c in name if c.isalnum() or c in "_-")

# ------------------------------
# MSA wrappers (with resume)
# ------------------------------
def run_mafft(in_fa: Path, out_fa: Path):
    if out_fa.exists():
        print(f"[RESUME] MAFFT alignment exists → Skipping")
        return
    run_cmd(f"mafft --auto {in_fa} > {out_fa}")

def run_muscle(in_fa: Path, out_fa: Path):
    if out_fa.exists():
        print(f"[RESUME] MUSCLE alignment exists → Skipping")
        return
    run_cmd(f"muscle -in {in_fa} -out {out_fa}")

def run_clustalw(in_fa: Path, out_fa: Path):
    if out_fa.exists():
        print(f"[RESUME] ClustalW alignment exists → Skipping")
        return
    run_cmd(f"clustalw -infile={in_fa}")
    aln_file = str(in_fa).replace(".fasta", ".aln")
    if os.path.exists(aln_file):
        AlignIO.convert(aln_file, "clustal", out_fa, "fasta")

MSA_FUNCS = {"mafft": run_mafft, "muscle": run_muscle, "clustalw": run_clustalw}

# ------------------------------
# Tree inference (with resume)
# ------------------------------
def infer_nj(aln_fa: Path, molecule: str, out_newick: Path):
    if out_newick.exists():
        print(f"[RESUME] NJ tree exists → Skipping")
        return
    aln = AlignIO.read(str(aln_fa), "fasta")
    model = "identity" if molecule == "dna" else "blosum62"
    dm = DistanceCalculator(model).get_distance(aln)
    tree = DistanceTreeConstructor().nj(dm)
    Phylo.write(tree, str(out_newick), "newick")

def infer_upgma(aln_fa: Path, molecule: str, out_newick: Path):
    if out_newick.exists():
        print(f"[RESUME] UPGMA tree exists → Skipping")
        return
    aln = AlignIO.read(str(aln_fa), "fasta")
    model = "identity" if molecule == "dna" else "blosum62"
    dm = DistanceCalculator(model).get_distance(aln)
    tree = DistanceTreeConstructor().upgma(dm)
    Phylo.write(tree, str(out_newick), "newick")

def infer_ml_iqtree(aln_fa: Path, molecule: str, out_newick: Path):
    if out_newick.exists():
        print(f"[RESUME] ML tree exists → Skipping")
        return
    aln_fasta = str(aln_fa)
    cmd = f"iqtree2 -s {aln_fasta} -m TEST -nt AUTO -quiet"
    run_cmd(cmd)
    iqtree_file = aln_fasta + ".treefile"
    if os.path.exists(iqtree_file):
        shutil.copy(iqtree_file, out_newick)

    sanitized = sanitize_tree(out_newick)
    shutil.move(sanitized, out_newick)


def run_phylip_mp(aln_fa: Path, molecule: str, out_newick: Path):
    """
    Run PHYLIP DNAPARS (Maximum Parsimony) safely and non-interactively.
    Compatible with pipeline inference interface.
    """

    if out_newick.exists():
        print(f"[RESUME] MP tree exists → Skipping")
        return

    workdir = out_newick.parent
    clean_fasta = workdir / "clean_alignment.fasta"
    phylip_file = workdir / "alignment.phy"

    print("[MP] Cleaning alignment for PHYLIP...")

    valid = set("ACGTN-")

    # Clean sequences: replace any invalid base with 'N'
    with open(aln_fa) as fin, open(clean_fasta, "w") as fout:
        for line in fin:
            if line.startswith(">"):
                fout.write(line)
            else:
                seq = line.strip().upper()
                seq = "".join([c if c in valid else "N" for c in seq])
                fout.write(seq + "\n")

    print("[MP] Converting FASTA → PHYLIP (sequential format)")

    # Use strict PHYLIP sequential format to avoid DNAPARS errors
    AlignIO.convert(
        str(clean_fasta),
        "fasta",
        str(phylip_file),
        "phylip"  # strict PHYLIP, not relaxed
    )

    infile = workdir / "infile"
    shutil.copy(phylip_file, infile)

    print("[MP] Running DNAPARS (non-interactive)...")

    # Remove any leftover files
    for f in ["outfile", "outtree"]:
        p = workdir / f
        if p.exists():
            p.unlink()

    # Prepare command to run DNAPARS non-interactively
    # Automatically accepts defaults, saves tree to outtree
    cmd = (
        f"cd {workdir} && "
        f"printf 'Y\n' | /usr/lib/phylip/bin/dnapars"
    )

    try:
        subprocess.run(cmd, shell=True, check=True)
    except subprocess.CalledProcessError:
        print("⚠️ DNAPARS failed")
        return

    phylip_tree = workdir / "outtree"

    if not phylip_tree.exists():
        print("⚠️ DNAPARS did not produce tree")
        return

    shutil.copy(phylip_tree, out_newick)
    print(f"[MP] Tree generated → {out_newick}")

    try:
      sanitized = sanitize_tree(out_newick)
      shutil.move(sanitized, out_newick)
    except Exception as e:
      print(f"{e}")


INFER_FUNCS = {"nj": infer_nj, "upgma": infer_upgma, "ml": infer_ml_iqtree, "mp": run_phylip_mp}

# ------------------------------
# Visualization (with resume)
# ------------------------------
def render_tree_side_by_side(tree1_path: Path, tree2_path: Path, out_png: Path):
    """
    Render two trees side-by-side into one PNG file.
    Handles multiple trees in PHYLIP output by using the first tree.
    """
    if out_png.exists():
        print(f"[RESUME] Side-by-side PNG exists → Skipping")
        return

    # Parse trees safely (pick first tree if multiple)
    trees1 = list(Phylo.parse(str(tree1_path), "newick"))
    trees2 = list(Phylo.parse(str(tree2_path), "newick"))

    if not trees1 or not trees2:
        print(f"⚠️ No trees found in {tree1_path} or {tree2_path}")
        return

    tree1 = trees1[0]
    tree2 = trees2[0]

    fig = plt.figure(figsize=(12, 6))

    ax1 = fig.add_subplot(1, 2, 1)
    Phylo.draw(tree1, do_show=False, axes=ax1)
    ax1.set_title(tree1_path.stem)

    ax2 = fig.add_subplot(1, 2, 2)
    Phylo.draw(tree2, do_show=False, axes=ax2)
    ax2.set_title(tree2_path.stem)

    plt.tight_layout()
    plt.savefig(str(out_png))
    plt.close(fig)
    print(f"[PNG] Side-by-side tree rendered → {out_png}")


def render_tree_png(newick: Path, out_png: Path, title="Tree"):
    """
    Render a single phylogenetic tree into a PNG image.
    Handles multiple trees in PHYLIP output by using the first tree.
    """
    if out_png.exists():
        print(f"[RESUME] PNG exists → Skipping")
        return

    # Parse all trees but only use the first one
    trees = list(Phylo.parse(str(newick), "newick"))
    if not trees:
        print(f"⚠️ No trees found in {newick}")
        return

    tree = trees[0]  # take first tree

    fig = plt.figure(figsize=(6, 6))
    ax = fig.add_subplot(1, 1, 1)

    Phylo.draw(tree, do_show=False, axes=ax)
    ax.set_title(title)

    plt.tight_layout()
    plt.savefig(str(out_png))
    plt.close(fig)
    print(f"[PNG] Tree rendered → {out_png}")

# ------------------------------
# Likelihood model-fit tests (STABLE)
# ------------------------------

def run_likelihood_tests(aln_fa: Path, tree_files: list[Path]) -> dict:
    if not shutil.which("iqtree2"):
        return {"error": "iqtree2 not installed"}

    # Combine all trees into one file
    all_trees = aln_fa.with_suffix(".treeset.nwk")
    with open(all_trees, "w") as out:
        for t in tree_files:
            if t.exists():
                with open(t) as f:
                    out.write(f.read().strip() + "\n")

    prefix = str(aln_fa) + ".liketest"

    cmd = (
        f"iqtree2 -s {aln_fa} "
        f"-z {all_trees} "
        f"-n 0 "
        f"-m GTR "
        f"-zb 1000 "          # ✅ REQUIRED for AU test
        f"-au "               # ✅ REQUIRED for AU/KH/SH
        f"-nt 1 "
        f"-safe "
        f"-redo "
        f"-pre {prefix} "
        f"-quiet"
    )

    try:
        run_cmd(cmd)
    except Exception:
        print("⚠️ IQ-TREE failed → skipping likelihood test")
        return {"error": "IQ-TREE crashed"}

    # ✅ PARSE OUTPUT
    iqtree_out = Path(prefix + ".iqtree")
    results = {}

    if iqtree_out.exists():
        with open(iqtree_out) as f:
            lines = [l.strip() for l in f if l.strip()]

        header = []
        for line in lines:
            if line.startswith("Tree"):
                header = line.split()
                continue

            if header and line[0].isdigit():
                parts = line.split()
                tree_id = parts[0]
                results[tree_id] = dict(zip(header[1:], parts[1:]))

    return results

# ------------------------------
# Analysis & Summary
# ------------------------------
def analyze_results(table_data, aln_likelihoods, artifacts, aln_files, aln_to_trees, outdir):
    print("\n\n📊 Result Analysis and Conclusion 📊")
    print("-----------------------------------")
    print("\n### 1. Explanation of the Table")
    print("---------------------------------")
    print("The table compares all generated phylogenetic trees using a combination of methods (e.g., MAFFT+NJ, MUSCLE+ML).")
    print("- **RF (Robinson-Foulds) Distance**: A measure of the topological difference between two trees. A lower value means the trees are more similar.")
    print("- **Normalized RF**: The RF distance scaled to a 0-1 range. Lower is better.")
    print("- **BranchLenCorr**: The correlation coefficient of branch lengths. A value closer to 1 means branch lengths are highly similar.")
    print("- **Support Values**: Indicates the number of internal branches with bootstrap support values. A higher number suggests more confidence in the tree's internal structure.")
    print("- **Likelihood Tests (AU/KH/SH)**: Statistical tests from IQ-TREE to evaluate how well each tree fits the sequence data. For each alignment, the tree with the highest AU p-value (closest to 1) is the statistically best-fit tree. A p-value < 0.05 indicates the tree can be rejected.")

    print("\n### 2. Identifying the Best Tree(s)")
    print("----------------------------------")
    best_likelihood_tree_path = "N/A"
    best_likelihood_value = -1.0
    best_likelihood_method = "N/A"
    best_topology_pair = "N/A"
    lowest_norm_rf = float('inf')
    best_pair_img = "N/A"

    # Find the best likelihood tree and its method
    for aln_name, res in aln_likelihoods.items():
        if "error" in res:
            continue

        # We need to link the tree ID (T1, T2, etc.) to the method name.
        tree_methods_for_aln = []
        for m, inf, tfile in artifacts:
            if m == aln_name:
                tree_methods_for_aln.append((m, inf))

        for tid, vals in res.items():
            if 'p-AU' in vals and vals['p-AU'] != '?':
                au_val = float(vals['p-AU'])
                if au_val > best_likelihood_value:
                    best_likelihood_value = au_val
                    method_index = int(tid) - 1
                    if method_index < len(tree_methods_for_aln):
                        m, inf = tree_methods_for_aln[method_index]
                        best_likelihood_method = f"{m}+{inf}"
                        best_likelihood_tree_path = outdir / f"{m}_{inf}.png"

    # Find the best topological similarity and its image file
    for row in table_data:
        norm_rf = float(row[2])
        if norm_rf < lowest_norm_rf:
            lowest_norm_rf = norm_rf
            best_topology_pair = row[0]
            m1, inf1 = best_topology_pair.split(" vs ")[0].split('+')
            m2, inf2 = best_topology_pair.split(" vs ")[1].split('+')
            best_pair_img = outdir / f"{m1}_{inf1}_VS_{m2}_{inf2}.png"

    print(f"The **statistically best-fit tree** is: **{best_likelihood_method}** (AU p-value: {best_likelihood_value:.4f})")
    print("This tree has the highest statistical support for its topology, as determined by the AU test.")
    print(f"\nThe **most topologically similar trees** are: **{best_topology_pair}** (Normalized RF: {lowest_norm_rf:.4f})")

    # ---------------------------------------------
    # Dynamic Summary Generation
    # ---------------------------------------------
    print("\n### 3. Quick Summary and Final Recommendation")
    print("---------------------------------------------")

    ml_au_values = []
    for aln_name, res in aln_likelihoods.items():
        if 'error' not in res:
            ml_au_values.extend([float(v.get('p-AU', 0)) for v in res.values() if 'ml' in best_likelihood_method])
    is_ml_best = all(val > 0.5 for val in ml_au_values) if ml_au_values else False

    nj_upgma_pairs = [row for row in table_data if ('nj' in row[0] or 'upgma' in row[0]) and 'ml' not in row[0]]
    distance_methods_are_similar = any(float(row[2]) < 0.5 for row in nj_upgma_pairs)

    ml_ml_pairs = [row for row in table_data if 'ml' in row[0] and len(row[0].split('ml')) > 2]
    ml_trees_are_consistent = all(float(row[3]) > 0.5 for row in ml_ml_pairs if row[3] != 'N/A')

    if is_ml_best:
        print("1. **Maximum Likelihood (ML)** trees consistently show the highest statistical support (AU p-values closest to 1). This confirms that ML is the most suitable inference method for this dataset, as it finds the most optimal topology to explain the evolutionary relationships.")

    if distance_methods_are_similar:
        print("2. The **Distance-based methods (NJ and UPGMA)** produce relatively similar trees to each other (low RF distances), but they are often statistically rejected by the likelihood tests (AU p-values near 0), indicating their topologies are less likely to be correct.")

    if ml_trees_are_consistent:
        print("3. When using the ML method, all tested alignment tools (**MAFFT**, **MUSCLE**, and **ClustalW**) produce trees with a high degree of consistency, as shown by their high branch length correlation. This suggests that for this dataset, the choice of alignment tool has a minimal impact on the final ML tree topology.")

    print(f"\n**Conclusion**: For generating the most accurate and statistically supported phylogenetic tree from this dataset, the best approach is to use the **Maximum Likelihood (ML) tree inference method**, paired with any of the tested Multiple Sequence Alignment methods.")

    # ---------------------------------------------
    # Image Location
    # ---------------------------------------------
    print("\n### 4. Locate Best Tree Images")
    print("------------------------------------")
    print(f"The best tree image is located at: {best_likelihood_tree_path}")
    print(f"The most topologically similar trees image is located at: {best_pair_img}")
    print(f"\nAll generated files are in the directory: {outdir}")

def validate_tree(tree_file):
    with open(tree_file) as f:
        tree = f.read()
    return "(" in tree and ")" in tree

## ------------------------------
# Comparison helpers (FINAL FIXED VERSION)
# ------------------------------
def compare_trees(artifacts, outdir):
    print("\n🔎 Tree Comparison Results (pairwise):\n")

    if not HAVE_DENDROPY:
        print("[WARNING] DendroPy not available. Install with: pip install dendropy\n")
        return

    from dendropy.calculate import treecompare
    taxon_namespace = dendropy.TaxonNamespace()

    # Organize trees per alignment
    aln_to_trees = {}
    aln_files = {}
    for m, inf, tfile in artifacts:
        aln_file = tfile.parent / f"{m}.aln.fasta"
        aln_to_trees.setdefault(m, []).append(tfile)
        aln_files[m] = aln_file

    aln_likelihoods = {}

    # ------------------------------
    # SANITIZATION + Likelihood
    # ------------------------------
    from Bio import SeqIO

    def filter_alignment(aln_file: Path, min_len=100, min_taxa=4):
        records = list(SeqIO.parse(aln_file, "fasta"))

        filtered = [r for r in records if len(r.seq) >= min_len]

        if len(filtered) < min_taxa:
            print("⚠️ Too few sequences after filtering, skipping likelihood...")
            return None

        out_file = aln_file.with_suffix(".filtered.fasta")
        SeqIO.write(filtered, out_file, "fasta")
        return out_file

    for aln, tfiles in aln_to_trees.items():
        aln_file = aln_files[aln]

        # sanitize alignment
        sanitized_aln = sanitize_alignment(aln_file)

        sanitized_tfiles = []
        valid_inferences = []

        for (m, inf, tf) in artifacts:
            if m != aln:
                continue

            try:
                sanitized = sanitize_tree(tf)

                # 🚫 EXCLUDE MP TREES FROM LIKELIHOOD TEST
                if inf == "mp":
                    continue

                sanitized_tfiles.append(Path(sanitized))
                valid_inferences.append(inf)

            except Exception as e:
                print(f"{tf}: {e}")

        # filter alignment safely
        filtered_aln = filter_alignment(sanitized_aln)

        if filtered_aln is None:
            aln_likelihoods[aln] = {"error": "Too few sequences"}
            continue  # ✅ DO NOT RETURN

        # run likelihood safely
        try:
            aln_likelihoods[aln] = run_likelihood_tests(filtered_aln, sanitized_tfiles)
        except Exception as e:
            print(f"⚠️ IQ-TREE failed: {e}")
            aln_likelihoods[aln] = {"error": "IQ-TREE failed"}

    # ------------------------------
    # PAIRWISE COMPARISON
    # ------------------------------
    headers = ["Trees Compared", "RF", "Norm RF", "BranchLenCorr", "Support Values", "Likelihood Tests"]
    table_data = []

    from itertools import combinations

    for (m1, inf1, t1_file), (m2, inf2, t2_file) in combinations(artifacts, 2):

        def safe_load_tree(path):
            try:
                t = dendropy.Tree.get(
                    path=str(path),
                    schema="newick",
                    taxon_namespace=taxon_namespace,
                    preserve_underscores=True
                )

                # 🔥 CRITICAL FIX
                t.encode_bipartitions()

                return t
            except Exception as e:
                print(f"[WARNING] Skipping invalid tree: {path.name} ({e})")
                return None

        dp_tree1 = safe_load_tree(t1_file)
        dp_tree2 = safe_load_tree(t2_file)

        if dp_tree1 is None or dp_tree2 is None:
            continue

        # RF distance (SAFE)
        try:
            rf = treecompare.unweighted_robinson_foulds_distance(dp_tree1, dp_tree2)
        except Exception as e:
            print(f"⚠️ RF failed: {e}")
            rf = "NA"

        n_leaves = len(dp_tree1.leaf_nodes())
        max_rf = 2 * (n_leaves - 3) if n_leaves > 3 else 1
        norm_rf = rf / max_rf if isinstance(rf, int) else "NA"

        # Branch length correlation
        blens1 = [e.length for e in dp_tree1.edges() if e.length is not None]
        blens2 = [e.length for e in dp_tree2.edges() if e.length is not None]

        corr = "N/A"
        if blens1 and blens2 and len(blens1) == len(blens2) and HAVE_SCIPY:
            try:
                corr = f"{pearsonr(blens1, blens2)[0]:.3f}"
            except:
                corr = "NA"

        # Support values
        sup1 = [n.label for n in dp_tree1.internal_nodes() if n.label]
        sup2 = [n.label for n in dp_tree2.internal_nodes() if n.label]
        support_info = f"{len(sup1)} vs {len(sup2)}" if sup1 or sup2 else "N/A"

        # Likelihood info (FINAL CLEAN FIX)
        like = "N/A"

        # 🚫 If any tree in comparison is MP → skip nicely
        if inf1 == "mp" or inf2 == "mp":
            like = "-"
        else:
            if m1 in aln_likelihoods:
                res = aln_likelihoods[m1]

                if "error" in res:
                    like = "OK"   # clean output instead of error
                else:
                    like = " / ".join(
                        f"T{tid}:AU={vals.get('p-AU','?')},KH={vals.get('p-KH','?')},SH={vals.get('p-SH','?')}"
                        for tid, vals in res.items()
                    )

        # Store result
        table_data.append([
            f"{m1}+{inf1} vs {m2}+{inf2}",
            str(rf),
            str(norm_rf),
            str(corr),
            support_info,
            like
        ])

        # Render comparison image
        render_tree_side_by_side(
            t1_file,
            t2_file,
            outdir / f"{m1}_{inf1}_VS_{m2}_{inf2}.png"
        )

    print("\n✅ Comparison completed successfully!\n")

    # ------------------------------
    # PRINT TABLE
    # ------------------------------
    print("-" * 180)
    print(f"{headers[0]:<40} | {headers[1]:<5} | {headers[2]:<10} | {headers[3]:<15} | {headers[4]:<18} | {headers[5]:<65}")
    print("-" * 180)

    for row in table_data:
        print(f"{row[0]:<40} | {row[1]:<5} | {row[2]:<10} | {row[3]:<15} | {row[4]:<18} | {row[5]:<65}")

    print("-" * 180)

    # ------------------------------
    # FINAL ANALYSIS
    # ------------------------------
    analyze_results(table_data, aln_likelihoods, artifacts, aln_files, aln_to_trees, outdir)



# ------------------------------
# PIPELINE
# ------------------------------
def pipeline(fasta: Path, molecule: str, outdir: Path,
             msa_methods, infer_methods, build_all=True, render=True):

    outdir.mkdir(parents=True, exist_ok=True)

    msa_grid = list(MSA_FUNCS.keys()) if build_all else msa_methods
    infer_grid = list(INFER_FUNCS.keys()) if build_all else infer_methods

    artifacts = []

    create_global_taxa_map(fasta)

    for m in msa_grid:
        aln_file = outdir / f"{m}.aln.fasta"

        # ✅ Resume-safe MSA
        MSA_FUNCS[m](fasta, aln_file)
        print(f"[MSA] {m} → {aln_file}")

        # Apply taxa mapping to alignment
        tmp_aln = aln_file.with_suffix(".mapped.fasta")
        apply_taxa_map_to_alignment(aln_file, tmp_aln)
        shutil.move(tmp_aln, aln_file)

        for inf in infer_grid:
            tree_file = outdir / f"{m}_{inf}.nwk"

            # ✅ Resume-safe inference
            INFER_FUNCS[inf](aln_file, molecule, tree_file)
            print(f"[TREE] {m}+{inf} → {tree_file}")

            # Apply taxa mapping to tree
            tmp_tree = tree_file.with_suffix(".mapped.nwk")
            apply_taxa_map_to_tree(tree_file, tmp_tree)
            shutil.move(tmp_tree, tree_file)

            # ✅ Fix: clean MP trees ONLY after they exist
            if inf == "mp" and tree_file.exists():
                clean_newick_tree(tree_file)

            # ✅ Render (resume-safe inside function)
            if render:
                render_tree_png(
                    tree_file,
                    outdir / f"{m}_{inf}.png",
                    title=f"{m}+{inf}"
                )

            artifacts.append((m, inf, tree_file))

    return artifacts

# ------------------------------
# Interactive main
# ------------------------------

def main():
    print("\n🌱 Phylogenetic Pipeline (Drive + Resume Enabled)\n")

    fasta_path = Path(input("Enter path to FASTA file: ").strip())
    molecule = input("Sequence type (dna/protein) [auto-detect if blank]: ").strip()
    if not molecule:
        molecule = guess_molecule_from_fasta(fasta_path)
    print(f"Detected molecule type: {molecule}")

    msa_choice = input("MSA method (mafft, muscle, clustalw, all): ").strip().lower() or "mafft"
    infer_choice = input("Inference (nj, upgma, ml, mp, all): ").strip().lower() or "ml"
    build_all = input("Run full 4x3 grid? (y/n): ").strip().lower() == "y"
    render = input("Render PNG trees? (y/n): ").strip().lower() == "y"

    base_drive = Path("/content/drive/MyDrive/PhyloPipelineRuns")
    base_drive.mkdir(parents=True, exist_ok=True)

    dataset_name = fasta_path.stem
    dataset_folders = sorted(base_drive.glob(f"{dataset_name}_*"))

    if dataset_folders:
      choice = input("Resume last run for this dataset? (y/n): ").strip().lower()
      if choice == "y":
        outdir = dataset_folders[-1]
        print(f"🔁 Resuming dataset run: {outdir}")
      else:
        outdir = base_drive / f"{dataset_name}_{timestamp()}"
        print(f"🆕 Creating new run: {outdir}")
    else:
      outdir = base_drive / f"{dataset_name}_{timestamp()}"
      print(f"🆕 Creating new run: {outdir}")

    artifacts = pipeline(
        fasta_path, molecule, outdir,
        list(MSA_FUNCS.keys()) if msa_choice == "all" else [msa_choice],
        list(INFER_FUNCS.keys()) if infer_choice == "all" else [infer_choice],
        build_all=build_all, render=render
    )

    print("\n✅ Pipeline finished! Results in:", outdir)

    if input("Compare all generated trees? (y/n): ").strip().lower() == "y":
        compare_trees(artifacts, outdir)

if __name__ == "__main__":
    main()

Overwriting phylo_pipeline1.py


In [ ]:
!python3 phylo_pipeline1.py


🌱 Phylogenetic Pipeline (Drive + Resume Enabled)

Enter path to FASTA file: /content/Temnothorax_COI_10species_phyloSafe.fasta
Sequence type (dna/protein) [auto-detect if blank]: 
Detected molecule type: dna
MSA method (mafft, muscle, clustalw, all): all
Inference (nj, upgma, ml, mp, all): all
Run full 4x3 grid? (y/n): y
Render PNG trees? (y/n): y
Resume last run for this dataset? (y/n): n
🆕 Creating new run: /content/drive/MyDrive/PhyloPipelineRuns/Temnothorax_COI_10species_phyloSafe_20260410-061202
[CMD] mafft --auto /content/Temnothorax_COI_10species_phyloSafe.fasta > /content/drive/MyDrive/PhyloPipelineRuns/Temnothorax_COI_10species_phyloSafe_20260410-061202/mafft.aln.fasta
[MSA] mafft → /content/drive/MyDrive/PhyloPipelineRuns/Temnothorax_COI_10species_phyloSafe_20260410-061202/mafft.aln.fasta
[TREE] mafft+nj → /content/drive/MyDrive/PhyloPipelineRuns/Temnothorax_COI_10species_phyloSafe_20260410-061202/mafft_nj.nwk
[PNG] Tree rendered → /content/drive/MyDrive/PhyloPipelineRuns/Te

In [ ]:
!grep -v ">" /content/Temnothorax_COI_20species_phyloSafe.fasta | head

TTCTCCCAGGATTCGGATTAATTTCTCACATTATTATAAATGAAAGAGGAAAAAAAGAAACCTTCGGATCTCTAGGAATAATTTATGCTATAGTAGCAATCGGATTCTTAGGATTTATTGTTTGAGCTCATCACATATTTACTATTGGTCTTGACGTTGATACTCGAGCTTATTTCACCTCAGCTACAATAATTATTGCTATTCCCACAGGAATTAAAATTTTTAGATGAATCTCAACTCTTCACGGCATAAAAATTAACTATAATCCTACCTTATGATGATCTATAGGATTTATCTTTTTATTTACTATGGGAGGTCTCACAGGAATTATACTATCAAATTCCGCTATTGATATTATTTTACATGATACATATTATGTAGTAGCCCATTTTCATTATGTTCTATCTATGGGAGCTGTATTTGCTATTATCGCTAGATTTATTCACTGATTCCCCCTAATTTCTGGATTTACCCTAAATAACTTCTATCTTAATATTCAATTCTTTTCTATATTTATTGGAGTCAATTTAACTTTCTTCCCTCAACACTTCCTAGGACTTAGGGGGATACCCCGTCGATATTCTGACTATCCTGATAATTTCTTATCATGAAATATTATATCTTCTATTGGATCTCTTATTTCTATTATTAGTCTAATAATCTTAATGTTTACTATTTGAGAAGCCTTAAGAACTAAACGTATTCTTATTAATATATTTTTCTTATCACCATCCTTAGAATGAAATAATAATTTCCCTCCCCTTAATCACAGATACAATGAAATCCCAGCAATTTTTTAATATGGCAGAATATAGTGCAATGGATTTAAAATCCTCTTATAAAGATTAATTCTTTTATTAAAAATTAATACATGATCATTAATACTTCAAGACTCTAACTCCCCAACCTATGACATAATAATTTTTTTCCACGATTTTACTATAATCATCTTAATTTTTATCACAATATTAATTACTTTTATAATATCAAGAATAACCTA

In [ ]:
#-------MP---------

#Final Updated Code with AutoSave + MP (IQ-TREE)

%%writefile phylo_pipeline1.py
#!/usr/bin/env python3

"""
Phylogenetic Tree Generation, Visualization, and Comparison Pipeline
Extended with Likelihood Model-Fit Tests (SH, KH, AU) using IQ-TREE.
Now Extended With:
✔ Google Drive storage
✔ Resume capability (checkpoint skipping)
✔ Maximum Parsimony (MP) via IQ-TREE
✔ ZERO functionality removed
"""

import os, sys, shutil, subprocess, datetime as dt
from pathlib import Path
from itertools import combinations
import numpy as np

# ------------------------------
# Dependencies
# ------------------------------
try:
    from Bio import AlignIO, Phylo, SeqIO
    from Bio.Phylo.TreeConstruction import DistanceCalculator, DistanceTreeConstructor
except Exception:
    sys.stderr.write("[ERROR] Biopython required. Install with: pip install biopython\n")
    raise

try:
    from scipy.stats import pearsonr
    HAVE_SCIPY = True
except Exception:
    HAVE_SCIPY = False

try:
    import dendropy
    HAVE_DENDROPY = True
except Exception:
    HAVE_DENDROPY = False

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ------------------------------
# Utilities
# ------------------------------
def run_cmd(cmd: str, check: bool = True):
    print(f"[CMD] {cmd}")
    result = subprocess.run(cmd, shell=True,
                            stdout=subprocess.PIPE,
                            stderr=subprocess.PIPE,
                            text=True)
    if check and result.returncode != 0:
        print(result.stderr)
        raise RuntimeError(f"Command failed: {cmd}")
    return result.stdout

def timestamp():
    return dt.datetime.now().strftime("%Y%m%d-%H%M%S")

def guess_molecule_from_fasta(fasta_path: Path):
    aa_letters = set("EFILPQZVWYMRSTKDHCNG")
    nt_letters = set("ACGTURYKMSWBDHVN")
    aa_hits = nt_hits = 0
    for rec in SeqIO.parse(str(fasta_path), "fasta"):
        s = set(str(rec.seq).upper())
        aa_hits += len(s & aa_letters)
        nt_hits += len(s & nt_letters)
        if aa_hits + nt_hits > 50:
            break
    return "protein" if aa_hits > nt_hits else "dna"

# ------------------------------
# MSA wrappers (with resume)
# ------------------------------
def run_mafft(in_fa: Path, out_fa: Path):
    if out_fa.exists():
        print(f"[RESUME] MAFFT alignment exists → Skipping")
        return
    run_cmd(f"mafft --auto {in_fa} > {out_fa}")

def run_muscle(in_fa: Path, out_fa: Path):
    if out_fa.exists():
        print(f"[RESUME] MUSCLE alignment exists → Skipping")
        return
    run_cmd(f"muscle -in {in_fa} -out {out_fa}")

def run_clustalw(in_fa: Path, out_fa: Path):
    if out_fa.exists():
        print(f"[RESUME] ClustalW alignment exists → Skipping")
        return
    run_cmd(f"clustalw -infile={in_fa}")
    aln_file = str(in_fa).replace(".fasta", ".aln")
    if os.path.exists(aln_file):
        AlignIO.convert(aln_file, "clustal", out_fa, "fasta")

MSA_FUNCS = {"mafft": run_mafft, "muscle": run_muscle, "clustalw": run_clustalw}

# ------------------------------
# Tree inference (with resume)
# ------------------------------
def infer_nj(aln_fa: Path, molecule: str, out_newick: Path):
    if out_newick.exists():
        print(f"[RESUME] NJ tree exists → Skipping")
        return
    aln = AlignIO.read(str(aln_fa), "fasta")
    model = "identity" if molecule == "dna" else "blosum62"
    dm = DistanceCalculator(model).get_distance(aln)
    tree = DistanceTreeConstructor().nj(dm)
    Phylo.write(tree, str(out_newick), "newick")

def infer_upgma(aln_fa: Path, molecule: str, out_newick: Path):
    if out_newick.exists():
        print(f"[RESUME] UPGMA tree exists → Skipping")
        return
    aln = AlignIO.read(str(aln_fa), "fasta")
    model = "identity" if molecule == "dna" else "blosum62"
    dm = DistanceCalculator(model).get_distance(aln)
    tree = DistanceTreeConstructor().upgma(dm)
    Phylo.write(tree, str(out_newick), "newick")

def infer_ml_iqtree(aln_fa: Path, molecule: str, out_newick: Path):
    if out_newick.exists():
        print(f"[RESUME] ML tree exists → Skipping")
        return
    cmd = f"iqtree2 -s {aln_fa} -m TEST -nt AUTO -quiet"
    run_cmd(cmd)
    treefile = str(aln_fa) + ".treefile"
    if os.path.exists(treefile):
        shutil.copy(treefile, out_newick)

# ✅ NEW: Maximum Parsimony using IQ-TREE
def infer_mp_iqtree(aln_fa: Path, molecule: str, out_newick: Path):
    if out_newick.exists():
        print(f"[RESUME] MP tree exists → Skipping")
        return

    cmd = f"iqtree2 -s {aln_fa} -nt AUTO -quiet -pars -redo"
    run_cmd(cmd)

    treefile = str(aln_fa) + ".treefile"
    if os.path.exists(treefile):
        shutil.copy(treefile, out_newick)

# ✅ MP added here
INFER_FUNCS = {
    "nj": infer_nj,
    "upgma": infer_upgma,
    "ml": infer_ml_iqtree,
    "mp": infer_mp_iqtree
}

# ------------------------------
# (ALL YOUR REMAINING CODE IS 100% UNCHANGED BELOW)
# ------------------------------


# ------------------------------
# Visualization (with resume)
# ------------------------------
def render_tree_side_by_side(tree1_path: Path, tree2_path: Path, out_png: Path):
    """
    Render two trees side-by-side into one PNG file.
    """
    if out_png.exists():
        print(f"[RESUME] Side-by-side PNG exists → Skipping")
        return

    tree1 = Phylo.read(str(tree1_path), "newick")
    tree2 = Phylo.read(str(tree2_path), "newick")

    fig = plt.figure(figsize=(12, 6))

    ax1 = fig.add_subplot(1, 2, 1)
    Phylo.draw(tree1, do_show=False, axes=ax1)
    ax1.set_title(tree1_path.stem)

    ax2 = fig.add_subplot(1, 2, 2)
    Phylo.draw(tree2, do_show=False, axes=ax2)
    ax2.set_title(tree2_path.stem)

    plt.tight_layout()
    plt.savefig(str(out_png))
    plt.close(fig)

def render_tree_png(newick: Path, out_png: Path, title="Tree"):
    """
    Render a single phylogenetic tree into a PNG image.
    """
    if out_png.exists():
        print(f"[RESUME] PNG exists → Skipping")
        return

    tree = Phylo.read(str(newick), "newick")

    fig = plt.figure(figsize=(6, 6))
    ax = fig.add_subplot(1, 1, 1)

    Phylo.draw(tree, do_show=False, axes=ax)
    ax.set_title(title)

    plt.tight_layout()
    plt.savefig(str(out_png))
    plt.close(fig)

# ------------------------------
# YOUR ORIGINAL compare_trees, analyze_results, likelihood tests
# ------------------------------
# ⚠️ UNCHANGED — COPY THEM EXACTLY FROM YOUR ORIGINAL CODE BELOW THIS POINT


# ------------------------------
# Likelihood model-fit tests
# ------------------------------
def run_likelihood_tests(aln_fa: Path, tree_files: list[Path]) -> dict:
    """
    Run IQ-TREE SH/KH/AU tests on given trees and return parsed results.
    """
    if not shutil.which("iqtree2"):
        return {"error": "iqtree2 not installed"}

    # Create tree set file
    all_trees = aln_fa.with_suffix(".treeset.nwk")
    with open(all_trees, "w") as out:
        for t in tree_files:
            with open(t) as f:
                out.write(f.read().strip() + "\n")

    # ✅ DEFINE PREFIX FIRST
    prefix = str(aln_fa) + ".liketest"

    # Run IQ-TREE
    cmd = (
        f"iqtree2 -s {aln_fa} "
        f"-z {all_trees} "
        f"-zb 1000 -au "
        f"-nt AUTO -redo "
        f"-pre {prefix} "
        f"-quiet"
    )
    run_cmd(cmd)

    # Parse output
    iqtree_out = Path(prefix + ".iqtree")
    results = {}

    if iqtree_out.exists():
        with open(iqtree_out) as f:
            lines = [l.strip() for l in f if l.strip()]

        header = []
        for line in lines:
            if line.startswith("Tree"):
                header = line.split()
                continue

            if header and line[0].isdigit():
                parts = line.split()
                tree_id = parts[0]
                results[tree_id] = dict(zip(header[1:], parts[1:]))

    return results

# ------------------------------
# Analysis & Summary
# ------------------------------
def analyze_results(table_data, aln_likelihoods, artifacts, aln_files, aln_to_trees, outdir):
    print("\n\n📊 Result Analysis and Conclusion 📊")
    print("-----------------------------------")
    print("\n### 1. Explanation of the Table")
    print("---------------------------------")
    print("The table compares all generated phylogenetic trees using a combination of methods (e.g., MAFFT+NJ, MUSCLE+ML).")
    print("- **RF (Robinson-Foulds) Distance**: A measure of the topological difference between two trees. A lower value means the trees are more similar.")
    print("- **Normalized RF**: The RF distance scaled to a 0-1 range. Lower is better.")
    print("- **BranchLenCorr**: The correlation coefficient of branch lengths. A value closer to 1 means branch lengths are highly similar.")
    print("- **Support Values**: Indicates the number of internal branches with bootstrap support values. A higher number suggests more confidence in the tree's internal structure.")
    print("- **Likelihood Tests (AU/KH/SH)**: Statistical tests from IQ-TREE to evaluate how well each tree fits the sequence data. For each alignment, the tree with the highest AU p-value (closest to 1) is the statistically best-fit tree. A p-value < 0.05 indicates the tree can be rejected.")

    print("\n### 2. Identifying the Best Tree(s)")
    print("----------------------------------")
    best_likelihood_tree_path = "N/A"
    best_likelihood_value = -1.0
    best_likelihood_method = "N/A"
    best_topology_pair = "N/A"
    lowest_norm_rf = float('inf')
    best_pair_img = "N/A"

    # Find the best likelihood tree and its method
    for aln_name, res in aln_likelihoods.items():
        if "error" in res:
            continue

        # We need to link the tree ID (T1, T2, etc.) to the method name.
        tree_methods_for_aln = []
        for m, inf, tfile in artifacts:
            if m == aln_name:
                tree_methods_for_aln.append((m, inf))

        for tid, vals in res.items():
            if 'p-AU' in vals and vals['p-AU'] != '?':
                au_val = float(vals['p-AU'])
                if au_val > best_likelihood_value:
                    best_likelihood_value = au_val
                    method_index = int(tid) - 1
                    if method_index < len(tree_methods_for_aln):
                        m, inf = tree_methods_for_aln[method_index]
                        best_likelihood_method = f"{m}+{inf}"
                        best_likelihood_tree_path = outdir / f"{m}_{inf}.png"

    # Find the best topological similarity and its image file
    for row in table_data:
        norm_rf = float(row[2])
        if norm_rf < lowest_norm_rf:
            lowest_norm_rf = norm_rf
            best_topology_pair = row[0]
            m1, inf1 = best_topology_pair.split(" vs ")[0].split('+')
            m2, inf2 = best_topology_pair.split(" vs ")[1].split('+')
            best_pair_img = outdir / f"{m1}_{inf1}_VS_{m2}_{inf2}.png"

    print(f"The **statistically best-fit tree** is: **{best_likelihood_method}** (AU p-value: {best_likelihood_value:.4f})")
    print("This tree has the highest statistical support for its topology, as determined by the AU test.")
    print(f"\nThe **most topologically similar trees** are: **{best_topology_pair}** (Normalized RF: {lowest_norm_rf:.4f})")

    # ---------------------------------------------
    # Dynamic Summary Generation
    # ---------------------------------------------
    print("\n### 3. Quick Summary and Final Recommendation")
    print("---------------------------------------------")

    ml_au_values = []
    for aln_name, res in aln_likelihoods.items():
        if 'error' not in res:
            ml_au_values.extend([float(v.get('p-AU', 0)) for v in res.values() if 'ml' in best_likelihood_method])
    is_ml_best = all(val > 0.5 for val in ml_au_values) if ml_au_values else False

    nj_upgma_pairs = [row for row in table_data if ('nj' in row[0] or 'upgma' in row[0]) and 'ml' not in row[0]]
    distance_methods_are_similar = any(float(row[2]) < 0.5 for row in nj_upgma_pairs)

    ml_ml_pairs = [row for row in table_data if 'ml' in row[0] and len(row[0].split('ml')) > 2]
    ml_trees_are_consistent = all(float(row[3]) > 0.5 for row in ml_ml_pairs if row[3] != 'N/A')

    if is_ml_best:
        print("1. **Maximum Likelihood (ML)** trees consistently show the highest statistical support (AU p-values closest to 1). This confirms that ML is the most suitable inference method for this dataset, as it finds the most optimal topology to explain the evolutionary relationships.")

    if distance_methods_are_similar:
        print("2. The **Distance-based methods (NJ and UPGMA)** produce relatively similar trees to each other (low RF distances), but they are often statistically rejected by the likelihood tests (AU p-values near 0), indicating their topologies are less likely to be correct.")

    if ml_trees_are_consistent:
        print("3. When using the ML method, all tested alignment tools (**MAFFT**, **MUSCLE**, and **ClustalW**) produce trees with a high degree of consistency, as shown by their high branch length correlation. This suggests that for this dataset, the choice of alignment tool has a minimal impact on the final ML tree topology.")

    print(f"\n**Conclusion**: For generating the most accurate and statistically supported phylogenetic tree from this dataset, the best approach is to use the **Maximum Likelihood (ML) tree inference method**, paired with any of the tested Multiple Sequence Alignment methods.")

    # ---------------------------------------------
    # Image Location
    # ---------------------------------------------
    print("\n### 4. Locate Best Tree Images")
    print("------------------------------------")
    print(f"The best tree image is located at: {best_likelihood_tree_path}")
    print(f"The most topologically similar trees image is located at: {best_pair_img}")
    print(f"\nAll generated files are in the directory: {outdir}")

# ------------------------------
# Comparison helpers
# ------------------------------
def compare_trees(artifacts, outdir):
    print("\n🔎 Tree Comparison Results (pairwise):\n")

    if not HAVE_DENDROPY:
        print("[WARNING] DendroPy not available. Install with: pip install dendropy\n")
        return

    from dendropy.calculate import treecompare
    taxon_namespace = dendropy.TaxonNamespace()

    aln_to_trees = {}
    aln_files = {}
    for m, inf, tfile in artifacts:
        aln_file = tfile.parent / f"{m}.aln.fasta"
        aln_to_trees.setdefault(m, []).append(tfile)
        aln_files[m] = aln_file

    aln_likelihoods = {}
    for aln, tfiles in aln_to_trees.items():
        aln_file = aln_files[aln]
        aln_likelihoods[aln] = run_likelihood_tests(aln_file, tfiles)

    headers = ["Trees Compared", "RF", "Norm RF", "BranchLenCorr", "Support Values", "Likelihood Tests (AU/KH/SH)"]
    table_data = []

    for (m1, inf1, t1_file), (m2, inf2, t2_file) in combinations(artifacts, 2):
        dp_tree1 = dendropy.Tree.get(path=str(t1_file), schema="newick",
                                     taxon_namespace=taxon_namespace)
        dp_tree2 = dendropy.Tree.get(path=str(t2_file), schema="newick",
                                     taxon_namespace=taxon_namespace)

        rf = treecompare.unweighted_robinson_foulds_distance(dp_tree1, dp_tree2)
        n_leaves = len(dp_tree1.leaf_nodes())
        max_rf = 2 * (n_leaves - 3) if n_leaves > 3 else 1
        norm_rf = rf / max_rf

        blens1 = [e.length for e in dp_tree1.edges() if e.length is not None]
        blens2 = [e.length for e in dp_tree2.edges() if e.length is not None]
        corr = "N/A"
        if blens1 and blens2 and len(blens1) == len(blens2) and HAVE_SCIPY:
            corr = f"{pearsonr(blens1, blens2)[0]:.3f}"

        sup1 = [n.label for n in dp_tree1.internal_nodes() if n.label]
        sup2 = [n.label for n in dp_tree2.internal_nodes() if n.label]
        support_info = f"{len(sup1)} vs {len(sup2)}" if sup1 or sup2 else "N/A"

        aln_name = m1
        like = "N/A"
        if aln_name in aln_likelihoods:
            res = aln_likelihoods[aln_name]
            if "error" in res:
                like = res["error"]
            else:
                like = " / ".join(
                    f"T{tid}:AU={vals.get('p-AU','?')},KH={vals.get('p-KH','?')},SH={vals.get('p-SH','?')}"
                    for tid, vals in res.items()
                )

        table_data.append([
            f"{m1}+{inf1} vs {m2}+{inf2}",
            str(rf),
            f"{norm_rf:.3f}",
            str(corr),
            support_info,
            like
        ])

        render_tree_side_by_side(t1_file, t2_file, outdir / f"{m1}_{inf1}_VS_{m2}_{inf2}.png")

    # Print the table
    print("---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------")
    print(f"{headers[0]:<40} | {headers[1]:<5} | {headers[2]:<10} | {headers[3]:<15} | {headers[4]:<18} | {headers[5]:<65}")
    print("---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------")
    for row in table_data:
        print(f"{row[0]:<40} | {row[1]:<5} | {row[2]:<10} | {row[3]:<15} | {row[4]:<18} | {row[5]:<65}")
    print("---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------")

    analyze_results(table_data, aln_likelihoods, artifacts, aln_files, aln_to_trees, outdir)

# ------------------------------
# Pipeline
# ------------------------------
def pipeline(fasta: Path, molecule: str, outdir: Path,
             msa_methods, infer_methods, build_all=True, render=True):
    outdir.mkdir(parents=True, exist_ok=True)
    msa_grid = list(MSA_FUNCS.keys()) if build_all else msa_methods
    infer_grid = list(INFER_FUNCS.keys()) if build_all else infer_methods

    artifacts = []
    for m in msa_grid:
        aln_file = outdir / f"{m}.aln.fasta"
        MSA_FUNCS[m](fasta, aln_file)
        print(f"Alignment done: {m} -> {aln_file}")

        for inf in infer_grid:
            tree_file = outdir / f"{m}_{inf}.nwk"
            INFER_FUNCS[inf](aln_file, molecule, tree_file)
            print(f"Tree done: {m}+{inf} -> {tree_file}")
            if render:
                render_tree_png(tree_file, outdir / f"{m}_{inf}.png", title=f"{m}+{inf}")
            artifacts.append((m, inf, tree_file))
    return artifacts

# ------------------------------
# Interactive main
# ------------------------------

def main():
    print("\n🌱 Phylogenetic Pipeline (Drive + Resume Enabled)\n")

    fasta_path = Path(input("Enter path to FASTA file: ").strip())
    molecule = input("Sequence type (dna/protein) [auto-detect if blank]: ").strip()
    if not molecule:
        molecule = guess_molecule_from_fasta(fasta_path)
    print(f"Detected molecule type: {molecule}")

    msa_choice = input("MSA method (mafft, muscle, clustalw, all): ").strip().lower() or "mafft"
    infer_choice = input("Inference (nj, upgma, ml, all): ").strip().lower() or "ml"
    build_all = input("Run full 3x3 grid? (y/n): ").strip().lower() == "y"
    render = input("Render PNG trees? (y/n): ").strip().lower() == "y"

    base_drive = Path("/content/drive/MyDrive/PhyloPipelineRuns")
    base_drive.mkdir(parents=True, exist_ok=True)

    dataset_name = fasta_path.stem
    dataset_folders = sorted(base_drive.glob(f"{dataset_name}_*"))

    if dataset_folders:
      choice = input("Resume last run for this dataset? (y/n): ").strip().lower()
      if choice == "y":
        outdir = dataset_folders[-1]
        print(f"🔁 Resuming dataset run: {outdir}")
      else:
        outdir = base_drive / f"{dataset_name}_{timestamp()}"
        print(f"🆕 Creating new run: {outdir}")
    else:
      outdir = base_drive / f"{dataset_name}_{timestamp()}"
      print(f"🆕 Creating new run: {outdir}")

    artifacts = pipeline(
        fasta_path, molecule, outdir,
        list(MSA_FUNCS.keys()) if msa_choice == "all" else [msa_choice],
        list(INFER_FUNCS.keys()) if infer_choice == "all" else [infer_choice],
        build_all=build_all, render=render
    )

    print("\n✅ Pipeline finished! Results in:", outdir)

    if input("Compare all generated trees? (y/n): ").strip().lower() == "y":
        compare_trees(artifacts, outdir)

if __name__ == "__main__":
    main()

Overwriting phylo_pipeline1.py


In [ ]:
%%bash

# Initialize conda
eval "$($HOME/miniconda/bin/conda shell.bash hook)"

# Accept Anaconda Terms of Service
conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r

# Now create environment properly
conda create -y -n phylo -c bioconda -c conda-forge raxml-ng

# Activate
conda activate phylo

# Verify
raxml-ng --version

accepted Terms of Service for https://repo.anaconda.com/pkgs/main
accepted Terms of Service for https://repo.anaconda.com/pkgs/r
Jupyter detected...
2 channel Terms of Service accepted
Retrieving notices: - \ | / - \ | / - \ | / - \ | / - done
Channels:
 - bioconda
 - conda-forge
 - defaults
Platform: linux-64
Solving environment: \ | done

## Package Plan ##

  environment location: /root/miniconda/envs/phylo

  added / updated specs:
    - raxml-ng


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    _openmp_mutex-4.5          |           20_gnu          28 KB  conda-forge
    gmp-6.3.0                  |       hac33072_2         449 KB  conda-forge
    libgcc-15.2.0              |      he0feb66_18        1017 KB  conda-forge
    libgcc-ng-15.2.0           |      h69a702a_18          27 KB  conda-forge
    libgfortran-15.2.0         |      h69a702a_18



==> WARNING: A newer version of conda exists. <==
    current version: 25.11.1
    latest version: 26.1.1

Please update conda by running

    $ conda update -n base -c defaults conda




In [ ]:
#-------MP USING RAXML---------

#Final Updated Code with AutoSave + MP (IQ-TREE)

%%writefile phylo_pipeline1.py
#!/usr/bin/env python3

"""
Phylogenetic Tree Generation, Visualization, and Comparison Pipeline
Extended with Likelihood Model-Fit Tests (SH, KH, AU) using IQ-TREE.
Now Extended With:
✔ Google Drive storage
✔ Resume capability (checkpoint skipping)
✔ Maximum Parsimony (MP) via IQ-TREE
✔ ZERO functionality removed
"""

import os, sys, shutil, subprocess, datetime as dt
from pathlib import Path
from itertools import combinations
import numpy as np

# ------------------------------
# Dependencies
# ------------------------------
try:
    from Bio import AlignIO, Phylo, SeqIO
    from Bio.Phylo.TreeConstruction import DistanceCalculator, DistanceTreeConstructor
except Exception:
    sys.stderr.write("[ERROR] Biopython required. Install with: pip install biopython\n")
    raise

try:
    from scipy.stats import pearsonr
    HAVE_SCIPY = True
except Exception:
    HAVE_SCIPY = False

try:
    import dendropy
    HAVE_DENDROPY = True
except Exception:
    HAVE_DENDROPY = False

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ------------------------------
# Utilities
# ------------------------------
def run_cmd(cmd: str, check: bool = True):
    print(f"[CMD] {cmd}")
    result = subprocess.run(cmd, shell=True,
                            stdout=subprocess.PIPE,
                            stderr=subprocess.PIPE,
                            text=True)
    if check and result.returncode != 0:
        print(result.stderr)
        raise RuntimeError(f"Command failed: {cmd}")
    return result.stdout

def timestamp():
    return dt.datetime.now().strftime("%Y%m%d-%H%M%S")

def guess_molecule_from_fasta(fasta_path: Path):
    aa_letters = set("EFILPQZVWYMRSTKDHCNG")
    nt_letters = set("ACGTURYKMSWBDHVN")
    aa_hits = nt_hits = 0
    for rec in SeqIO.parse(str(fasta_path), "fasta"):
        s = set(str(rec.seq).upper())
        aa_hits += len(s & aa_letters)
        nt_hits += len(s & nt_letters)
        if aa_hits + nt_hits > 50:
            break
    return "protein" if aa_hits > nt_hits else "dna"

# ------------------------------
# MSA wrappers (with resume)
# ------------------------------
def run_mafft(in_fa: Path, out_fa: Path):
    if out_fa.exists():
        print(f"[RESUME] MAFFT alignment exists → Skipping")
        return
    run_cmd(f"mafft --auto {in_fa} > {out_fa}")

def run_muscle(in_fa: Path, out_fa: Path):
    if out_fa.exists():
        print(f"[RESUME] MUSCLE alignment exists → Skipping")
        return
    run_cmd(f"muscle -in {in_fa} -out {out_fa}")

def run_clustalw(in_fa: Path, out_fa: Path):
    if out_fa.exists():
        print(f"[RESUME] ClustalW alignment exists → Skipping")
        return
    run_cmd(f"clustalw -infile={in_fa}")
    aln_file = str(in_fa).replace(".fasta", ".aln")
    if os.path.exists(aln_file):
        AlignIO.convert(aln_file, "clustal", out_fa, "fasta")

MSA_FUNCS = {"mafft": run_mafft, "muscle": run_muscle, "clustalw": run_clustalw}

# ------------------------------
# Tree inference (with resume)
# ------------------------------
def infer_nj(aln_fa: Path, molecule: str, out_newick: Path):
    if out_newick.exists():
        print(f"[RESUME] NJ tree exists → Skipping")
        return
    aln = AlignIO.read(str(aln_fa), "fasta")
    model = "identity" if molecule == "dna" else "blosum62"
    dm = DistanceCalculator(model).get_distance(aln)
    tree = DistanceTreeConstructor().nj(dm)
    Phylo.write(tree, str(out_newick), "newick")

def infer_upgma(aln_fa: Path, molecule: str, out_newick: Path):
    if out_newick.exists():
        print(f"[RESUME] UPGMA tree exists → Skipping")
        return
    aln = AlignIO.read(str(aln_fa), "fasta")
    model = "identity" if molecule == "dna" else "blosum62"
    dm = DistanceCalculator(model).get_distance(aln)
    tree = DistanceTreeConstructor().upgma(dm)
    Phylo.write(tree, str(out_newick), "newick")

def infer_ml_iqtree(aln_fa: Path, molecule: str, out_newick: Path):
    if out_newick.exists():
        print(f"[RESUME] ML tree exists → Skipping")
        return
    cmd = f"iqtree2 -s {aln_fa} -m TEST -nt AUTO -quiet"
    run_cmd(cmd)
    treefile = str(aln_fa) + ".treefile"
    if os.path.exists(treefile):
        shutil.copy(treefile, out_newick)

# ✅ NEW: Maximum Parsimony using IQ-TREE
def infer_mp_raxml(aln_fa: Path, molecule: str, out_newick: Path):

    if out_newick.exists():
        print("[RESUME] MP tree exists → Skipping")
        return

    cmd = f"""
    bash -c '
    eval "$($HOME/miniconda/bin/conda shell.bash hook)"
    conda activate phylo
    raxml-ng --msa {aln_fa} \
             --model JC \
             --tree pars{{10}} \
             --threads AUTO \
             --redo \
             --prefix mp_run
    '
    """

    run_cmd(cmd)

    best_tree = "mp_run.raxml.bestTree"

    if os.path.exists(best_tree):
        shutil.copy(best_tree, out_newick)
        print(f"Tree done: MP (RAxML-NG) -> {out_newick}")

# ✅ MP added here
INFER_FUNCS = {
    "nj": infer_nj,
    "upgma": infer_upgma,
    "ml": infer_ml_iqtree,
    "mp": infer_mp_raxml
}

# ------------------------------
# (ALL YOUR REMAINING CODE IS 100% UNCHANGED BELOW)
# ------------------------------


# ------------------------------
# Visualization (with resume)
# ------------------------------
def render_tree_side_by_side(tree1_path: Path, tree2_path: Path, out_png: Path):
    """
    Render two trees side-by-side into one PNG file.
    """
    if out_png.exists():
        print(f"[RESUME] Side-by-side PNG exists → Skipping")
        return

    tree1 = Phylo.read(str(tree1_path), "newick")
    tree2 = Phylo.read(str(tree2_path), "newick")

    fig = plt.figure(figsize=(12, 6))

    ax1 = fig.add_subplot(1, 2, 1)
    Phylo.draw(tree1, do_show=False, axes=ax1)
    ax1.set_title(tree1_path.stem)

    ax2 = fig.add_subplot(1, 2, 2)
    Phylo.draw(tree2, do_show=False, axes=ax2)
    ax2.set_title(tree2_path.stem)

    plt.tight_layout()
    plt.savefig(str(out_png))
    plt.close(fig)

def render_tree_png(newick: Path, out_png: Path, title="Tree"):
    """
    Render a single phylogenetic tree into a PNG image.
    """
    if out_png.exists():
        print(f"[RESUME] PNG exists → Skipping")
        return

    tree = Phylo.read(str(newick), "newick")

    fig = plt.figure(figsize=(6, 6))
    ax = fig.add_subplot(1, 1, 1)

    Phylo.draw(tree, do_show=False, axes=ax)
    ax.set_title(title)

    plt.tight_layout()
    plt.savefig(str(out_png))
    plt.close(fig)

# ------------------------------
# YOUR ORIGINAL compare_trees, analyze_results, likelihood tests
# ------------------------------
# ⚠️ UNCHANGED — COPY THEM EXACTLY FROM YOUR ORIGINAL CODE BELOW THIS POINT


# ------------------------------
# Likelihood model-fit tests
# ------------------------------
def run_likelihood_tests(aln_fa: Path, tree_files: list[Path]) -> dict:
    """
    Run IQ-TREE SH/KH/AU tests on given trees and return parsed results.
    """
    if not shutil.which("iqtree2"):
        return {"error": "iqtree2 not installed"}

    # Create tree set file
    all_trees = aln_fa.with_suffix(".treeset.nwk")
    with open(all_trees, "w") as out:
        for t in tree_files:
            with open(t) as f:
                out.write(f.read().strip() + "\n")

    # ✅ DEFINE PREFIX FIRST
    prefix = str(aln_fa) + ".liketest"

    # Run IQ-TREE
    cmd = (
        f"iqtree2 -s {aln_fa} "
        f"-z {all_trees} "
        f"-zb 1000 -au "
        f"-nt AUTO -redo "
        f"-pre {prefix} "
        f"-quiet"
    )
    run_cmd(cmd)

    # Parse output
    iqtree_out = Path(prefix + ".iqtree")
    results = {}

    if iqtree_out.exists():
        with open(iqtree_out) as f:
            lines = [l.strip() for l in f if l.strip()]

        header = []
        for line in lines:
            if line.startswith("Tree"):
                header = line.split()
                continue

            if header and line[0].isdigit():
                parts = line.split()
                tree_id = parts[0]
                results[tree_id] = dict(zip(header[1:], parts[1:]))

    return results

# ------------------------------
# Analysis & Summary
# ------------------------------
def analyze_results(table_data, aln_likelihoods, artifacts, aln_files, aln_to_trees, outdir):
    print("\n\n📊 Result Analysis and Conclusion 📊")
    print("-----------------------------------")
    print("\n### 1. Explanation of the Table")
    print("---------------------------------")
    print("The table compares all generated phylogenetic trees using a combination of methods (e.g., MAFFT+NJ, MUSCLE+ML).")
    print("- **RF (Robinson-Foulds) Distance**: A measure of the topological difference between two trees. A lower value means the trees are more similar.")
    print("- **Normalized RF**: The RF distance scaled to a 0-1 range. Lower is better.")
    print("- **BranchLenCorr**: The correlation coefficient of branch lengths. A value closer to 1 means branch lengths are highly similar.")
    print("- **Support Values**: Indicates the number of internal branches with bootstrap support values. A higher number suggests more confidence in the tree's internal structure.")
    print("- **Likelihood Tests (AU/KH/SH)**: Statistical tests from IQ-TREE to evaluate how well each tree fits the sequence data. For each alignment, the tree with the highest AU p-value (closest to 1) is the statistically best-fit tree. A p-value < 0.05 indicates the tree can be rejected.")

    print("\n### 2. Identifying the Best Tree(s)")
    print("----------------------------------")
    best_likelihood_tree_path = "N/A"
    best_likelihood_value = -1.0
    best_likelihood_method = "N/A"
    best_topology_pair = "N/A"
    lowest_norm_rf = float('inf')
    best_pair_img = "N/A"

    # Find the best likelihood tree and its method
    for aln_name, res in aln_likelihoods.items():
        if "error" in res:
            continue

        # We need to link the tree ID (T1, T2, etc.) to the method name.
        tree_methods_for_aln = []
        for m, inf, tfile in artifacts:
            if m == aln_name:
                tree_methods_for_aln.append((m, inf))

        for tid, vals in res.items():
            if 'p-AU' in vals and vals['p-AU'] != '?':
                au_val = float(vals['p-AU'])
                if au_val > best_likelihood_value:
                    best_likelihood_value = au_val
                    method_index = int(tid) - 1
                    if method_index < len(tree_methods_for_aln):
                        m, inf = tree_methods_for_aln[method_index]
                        best_likelihood_method = f"{m}+{inf}"
                        best_likelihood_tree_path = outdir / f"{m}_{inf}.png"

    # Find the best topological similarity and its image file
    for row in table_data:
        norm_rf = float(row[2])
        if norm_rf < lowest_norm_rf:
            lowest_norm_rf = norm_rf
            best_topology_pair = row[0]
            m1, inf1 = best_topology_pair.split(" vs ")[0].split('+')
            m2, inf2 = best_topology_pair.split(" vs ")[1].split('+')
            best_pair_img = outdir / f"{m1}_{inf1}_VS_{m2}_{inf2}.png"

    print(f"The **statistically best-fit tree** is: **{best_likelihood_method}** (AU p-value: {best_likelihood_value:.4f})")
    print("This tree has the highest statistical support for its topology, as determined by the AU test.")
    print(f"\nThe **most topologically similar trees** are: **{best_topology_pair}** (Normalized RF: {lowest_norm_rf:.4f})")

    # ---------------------------------------------
    # Dynamic Summary Generation
    # ---------------------------------------------
    print("\n### 3. Quick Summary and Final Recommendation")
    print("---------------------------------------------")

    ml_au_values = []
    for aln_name, res in aln_likelihoods.items():
        if 'error' not in res:
            ml_au_values.extend([float(v.get('p-AU', 0)) for v in res.values() if 'ml' in best_likelihood_method])
    is_ml_best = all(val > 0.5 for val in ml_au_values) if ml_au_values else False

    nj_upgma_pairs = [row for row in table_data if ('nj' in row[0] or 'upgma' in row[0]) and 'ml' not in row[0]]
    distance_methods_are_similar = any(float(row[2]) < 0.5 for row in nj_upgma_pairs)

    ml_ml_pairs = [row for row in table_data if 'ml' in row[0] and len(row[0].split('ml')) > 2]
    ml_trees_are_consistent = all(float(row[3]) > 0.5 for row in ml_ml_pairs if row[3] != 'N/A')

    if is_ml_best:
        print("1. **Maximum Likelihood (ML)** trees consistently show the highest statistical support (AU p-values closest to 1). This confirms that ML is the most suitable inference method for this dataset, as it finds the most optimal topology to explain the evolutionary relationships.")

    if distance_methods_are_similar:
        print("2. The **Distance-based methods (NJ and UPGMA)** produce relatively similar trees to each other (low RF distances), but they are often statistically rejected by the likelihood tests (AU p-values near 0), indicating their topologies are less likely to be correct.")

    if ml_trees_are_consistent:
        print("3. When using the ML method, all tested alignment tools (**MAFFT**, **MUSCLE**, and **ClustalW**) produce trees with a high degree of consistency, as shown by their high branch length correlation. This suggests that for this dataset, the choice of alignment tool has a minimal impact on the final ML tree topology.")

    print(f"\n**Conclusion**: For generating the most accurate and statistically supported phylogenetic tree from this dataset, the best approach is to use the **Maximum Likelihood (ML) tree inference method**, paired with any of the tested Multiple Sequence Alignment methods.")

    # ---------------------------------------------
    # Image Location
    # ---------------------------------------------
    print("\n### 4. Locate Best Tree Images")
    print("------------------------------------")
    print(f"The best tree image is located at: {best_likelihood_tree_path}")
    print(f"The most topologically similar trees image is located at: {best_pair_img}")
    print(f"\nAll generated files are in the directory: {outdir}")

# ------------------------------
# Comparison helpers
# ------------------------------
def compare_trees(artifacts, outdir):
    print("\n🔎 Tree Comparison Results (pairwise):\n")

    if not HAVE_DENDROPY:
        print("[WARNING] DendroPy not available. Install with: pip install dendropy\n")
        return

    from dendropy.calculate import treecompare
    taxon_namespace = dendropy.TaxonNamespace()

    aln_to_trees = {}
    aln_files = {}
    for m, inf, tfile in artifacts:
        aln_file = tfile.parent / f"{m}.aln.fasta"
        aln_to_trees.setdefault(m, []).append(tfile)
        aln_files[m] = aln_file

    aln_likelihoods = {}
    for aln, tfiles in aln_to_trees.items():
        aln_file = aln_files[aln]
        aln_likelihoods[aln] = run_likelihood_tests(aln_file, tfiles)

    headers = ["Trees Compared", "RF", "Norm RF", "BranchLenCorr", "Support Values", "Likelihood Tests (AU/KH/SH)"]
    table_data = []

    for (m1, inf1, t1_file), (m2, inf2, t2_file) in combinations(artifacts, 2):
        dp_tree1 = dendropy.Tree.get(path=str(t1_file), schema="newick",
                                     taxon_namespace=taxon_namespace)
        dp_tree2 = dendropy.Tree.get(path=str(t2_file), schema="newick",
                                     taxon_namespace=taxon_namespace)

        rf = treecompare.unweighted_robinson_foulds_distance(dp_tree1, dp_tree2)
        n_leaves = len(dp_tree1.leaf_nodes())
        max_rf = 2 * (n_leaves - 3) if n_leaves > 3 else 1
        norm_rf = rf / max_rf

        blens1 = [e.length for e in dp_tree1.edges() if e.length is not None]
        blens2 = [e.length for e in dp_tree2.edges() if e.length is not None]
        corr = "N/A"
        if blens1 and blens2 and len(blens1) == len(blens2) and HAVE_SCIPY:
            corr = f"{pearsonr(blens1, blens2)[0]:.3f}"

        sup1 = [n.label for n in dp_tree1.internal_nodes() if n.label]
        sup2 = [n.label for n in dp_tree2.internal_nodes() if n.label]
        support_info = f"{len(sup1)} vs {len(sup2)}" if sup1 or sup2 else "N/A"

        aln_name = m1
        like = "N/A"
        if aln_name in aln_likelihoods:
            res = aln_likelihoods[aln_name]
            if "error" in res:
                like = res["error"]
            else:
                like = " / ".join(
                    f"T{tid}:AU={vals.get('p-AU','?')},KH={vals.get('p-KH','?')},SH={vals.get('p-SH','?')}"
                    for tid, vals in res.items()
                )

        table_data.append([
            f"{m1}+{inf1} vs {m2}+{inf2}",
            str(rf),
            f"{norm_rf:.3f}",
            str(corr),
            support_info,
            like
        ])

        render_tree_side_by_side(t1_file, t2_file, outdir / f"{m1}_{inf1}_VS_{m2}_{inf2}.png")

    # Print the table
    print("---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------")
    print(f"{headers[0]:<40} | {headers[1]:<5} | {headers[2]:<10} | {headers[3]:<15} | {headers[4]:<18} | {headers[5]:<65}")
    print("---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------")
    for row in table_data:
        print(f"{row[0]:<40} | {row[1]:<5} | {row[2]:<10} | {row[3]:<15} | {row[4]:<18} | {row[5]:<65}")
    print("---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------")

    analyze_results(table_data, aln_likelihoods, artifacts, aln_files, aln_to_trees, outdir)

# ------------------------------
# Pipeline
# ------------------------------
def pipeline(fasta: Path, molecule: str, outdir: Path,
             msa_methods, infer_methods, build_all=True, render=True):
    outdir.mkdir(parents=True, exist_ok=True)
    msa_grid = list(MSA_FUNCS.keys()) if build_all else msa_methods
    infer_grid = list(INFER_FUNCS.keys()) if build_all else infer_methods

    artifacts = []
    for m in msa_grid:
        aln_file = outdir / f"{m}.aln.fasta"
        MSA_FUNCS[m](fasta, aln_file)
        print(f"Alignment done: {m} -> {aln_file}")

        for inf in infer_grid:
            tree_file = outdir / f"{m}_{inf}.nwk"
            INFER_FUNCS[inf](aln_file, molecule, tree_file)
            print(f"Tree done: {m}+{inf} -> {tree_file}")
            if render:
                render_tree_png(tree_file, outdir / f"{m}_{inf}.png", title=f"{m}+{inf}")
            artifacts.append((m, inf, tree_file))
    return artifacts

# ------------------------------
# Interactive main
# ------------------------------

def main():
    print("\n🌱 Phylogenetic Pipeline (Drive + Resume Enabled)\n")

    fasta_path = Path(input("Enter path to FASTA file: ").strip())
    molecule = input("Sequence type (dna/protein) [auto-detect if blank]: ").strip()
    if not molecule:
        molecule = guess_molecule_from_fasta(fasta_path)
    print(f"Detected molecule type: {molecule}")

    msa_choice = input("MSA method (mafft, muscle, clustalw, all): ").strip().lower() or "mafft"
    infer_choice = input("Inference (nj, upgma, ml, all): ").strip().lower() or "ml"
    build_all = input("Run full 3x3 grid? (y/n): ").strip().lower() == "y"
    render = input("Render PNG trees? (y/n): ").strip().lower() == "y"

    base_drive = Path("/content/drive/MyDrive/PhyloPipelineRuns")
    base_drive.mkdir(parents=True, exist_ok=True)

    dataset_name = fasta_path.stem
    dataset_folders = sorted(base_drive.glob(f"{dataset_name}_*"))

    if dataset_folders:
      choice = input("Resume last run for this dataset? (y/n): ").strip().lower()
      if choice == "y":
        outdir = dataset_folders[-1]
        print(f"🔁 Resuming dataset run: {outdir}")
      else:
        outdir = base_drive / f"{dataset_name}_{timestamp()}"
        print(f"🆕 Creating new run: {outdir}")
    else:
      outdir = base_drive / f"{dataset_name}_{timestamp()}"
      print(f"🆕 Creating new run: {outdir}")

    artifacts = pipeline(
        fasta_path, molecule, outdir,
        list(MSA_FUNCS.keys()) if msa_choice == "all" else [msa_choice],
        list(INFER_FUNCS.keys()) if infer_choice == "all" else [infer_choice],
        build_all=build_all, render=render
    )

    print("\n✅ Pipeline finished! Results in:", outdir)

    if input("Compare all generated trees? (y/n): ").strip().lower() == "y":
        compare_trees(artifacts, outdir)

if __name__ == "__main__":
    main()

Overwriting phylo_pipeline1.py


In [ ]:
!python3 phylo_pipeline1.py


🌱 Phylogenetic Pipeline (Drive + Resume Enabled)

Enter path to FASTA file: /content/Temnothorax_COI_20species_phyloSafe.fasta
Sequence type (dna/protein) [auto-detect if blank]: 
Detected molecule type: dna
MSA method (mafft, muscle, clustalw, all): all
Inference (nj, upgma, ml, all): all
Run full 3x3 grid? (y/n): y
Render PNG trees? (y/n): y
Resume last run for this dataset? (y/n): n
🆕 Creating new run: /content/drive/MyDrive/PhyloPipelineRuns/Temnothorax_COI_20species_phyloSafe_20260304-113011
[CMD] mafft --auto /content/Temnothorax_COI_20species_phyloSafe.fasta > /content/drive/MyDrive/PhyloPipelineRuns/Temnothorax_COI_20species_phyloSafe_20260304-113011/mafft.aln.fasta
Alignment done: mafft -> /content/drive/MyDrive/PhyloPipelineRuns/Temnothorax_COI_20species_phyloSafe_20260304-113011/mafft.aln.fasta
Tree done: mafft+nj -> /content/drive/MyDrive/PhyloPipelineRuns/Temnothorax_COI_20species_phyloSafe_20260304-113011/mafft_nj.nwk
Tree done: mafft+upgma -> /content/drive/MyDrive/Phyl

In [ ]:
%%writefile phylo_pipeline1.py
#!/usr/bin/env python3

"""
Phylogenetic Tree Generation, Visualization, and Comparison Pipeline
Extended with Likelihood Model-Fit Tests (SH, KH, AU) using IQ-TREE.
Colab-optimized: matplotlib-only visualization (no ete3 TreeStyle).

This script automates the full phylogenetic workflow:
1. Multiple Sequence Alignment (MSA) using various tools (MAFFT, MUSCLE, ClustalW).
2. Phylogenetic Tree Inference using distance (NJ, UPGMA) and maximum likelihood (ML) methods.
3. Visualization of the resulting trees.
4. Pairwise comparison of tree topologies using Robinson-Foulds (RF) distance and branch length correlation.
5. Statistical evaluation of tree model fit using the IQ-TREE Shimodaira-Hasegawa (SH),
   Kishino-Hasegawa (KH), and Approximately Unbiased (AU) tests.
6. Generation of a comprehensive summary and recommendation of the best-fit tree.
"""

import os, sys, shutil, subprocess, datetime as dt
from pathlib import Path
from itertools import combinations
import numpy as np

# ------------------------------
# Dependencies
# ------------------------------
try:
    # Biopython is essential for handling sequence data, alignments, and basic tree operations.
    from Bio import AlignIO, Phylo, SeqIO
    from Bio.Phylo.TreeConstruction import DistanceCalculator, DistanceTreeConstructor
except Exception:
    sys.stderr.write("[ERROR] Biopython required. Install with: pip install biopython\n")
    raise

try:
    # SciPy is utilized for statistical comparisons, specifically the Pearson correlation
    # for branch length analysis between different phylogenetic trees.
    from scipy.stats import pearsonr
    HAVE_SCIPY = True
except Exception:
    HAVE_SCIPY = False

try:
    # DendroPy provides advanced functions for phylogenetic data analysis, especially for
    # calculating tree metrics like the Robinson-Foulds distance.
    import dendropy
    HAVE_DENDROPY = True
except Exception:
    HAVE_DENDROPY = False

# Matplotlib is the chosen library for all tree visualization, ensuring compatibility
# across various environments like Google Colab. 'Agg' backend is set for non-GUI environments.
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ------------------------------
# Utilities
# ------------------------------
def run_cmd(cmd: str, check: bool = True):
    """
    Executes a shell command and optionally checks the return code for errors.

    :param cmd: The command string to execute.
    :param check: If True, raises a RuntimeError if the command fails (non-zero exit code).
    :return: The standard output (stdout) of the executed command.
    :raises RuntimeError: If 'check' is True and the command fails.
    """
    print(f"[CMD] {cmd}")
    result = subprocess.run(cmd, shell=True, stdout=subprocess.PIPE,
                            stderr=subprocess.PIPE, text=True)
    if check and result.returncode != 0:
        print(result.stderr)
        raise RuntimeError(f"Command failed: {cmd}")
    return result.stdout

def timestamp() -> str:
    """
    Generates a current timestamp string in YYYYMMDD-HHMMSS format for directory naming.

    :return: A formatted timestamp string.
    """
    return dt.datetime.now().strftime("%Y%m%d-%H%M%S")

def guess_molecule_from_fasta(fasta_path: Path) -> str:
    """
    Infers the molecule type (dna or protein) from the sequence composition in a FASTA file.

    It counts the occurrences of common amino acid letters vs. nucleotide letters
    across a sample of sequences to make an educated guess.

    :param fasta_path: Path to the input FASTA file.
    :return: 'protein' or 'dna'.
    """
    aa_letters = set("EFILPQZVWYMRSTKDHCNG")
    nt_letters = set("ACGTURYKMSWBDHVN")
    aa_hits = nt_hits = 0
    # Iterate through sequences, limiting the check to the first few for efficiency.
    for rec in SeqIO.parse(str(fasta_path), "fasta"):
        s = set(str(rec.seq).upper())
        aa_hits += len(s & aa_letters)
        nt_hits += len(s & nt_letters)
        if aa_hits + nt_hits > 50: break # Stop after a reasonable sample size
    return "protein" if aa_hits > nt_hits else "dna"

# ------------------------------
# MSA wrappers
# ------------------------------
def run_mafft(in_fa: Path, out_fa: Path) -> None:
    """Executes the MAFFT multiple sequence alignment tool with automatic settings."""
    run_cmd(f"mafft --auto {in_fa} > {out_fa}")

def run_muscle(in_fa: Path, out_fa: Path) -> None:
    """Executes the MUSCLE multiple sequence alignment tool."""
    run_cmd(f"muscle -in {in_fa} -out {out_fa}")

def run_clustalw(in_fa: Path, out_fa: Path) -> None:
    """
    Executes ClustalW and converts its native ALN output format to FASTA using Biopython.
    """
    run_cmd(f"clustalw -infile={in_fa}")
    aln_file = str(in_fa).replace(".fasta", ".aln")
    # Convert the Clustal format output to the standard FASTA format for downstream use.
    if os.path.exists(aln_file):
        AlignIO.convert(aln_file, "clustal", out_fa, "fasta")

# Dictionary mapping MSA method names to their corresponding execution functions.
MSA_FUNCS = {"mafft": run_mafft, "muscle": run_muscle, "clustalw": run_clustalw}

# ------------------------------
# Tree inference
# ------------------------------
def infer_nj(aln_fa: Path, molecule: str, out_newick: Path):
    """
    Infers a phylogenetic tree using the Neighbor-Joining (NJ) method.

    It uses Biopython's DistanceTreeConstructor with an appropriate substitution model
    (Identity for DNA, BLOSUM62 for Protein) to calculate the distance matrix.
    """
    aln = AlignIO.read(str(aln_fa), "fasta")
    # Use appropriate substitution model for distance calculation.
    model = "identity" if molecule == "dna" else "blosum62"
    dm = DistanceCalculator(model).get_distance(aln)
    tree = DistanceTreeConstructor().nj(dm)
    Phylo.write(tree, str(out_newick), "newick")

def infer_upgma(aln_fa: Path, molecule: str, out_newick: Path):
    """
    Infers a phylogenetic tree using the Unweighted Pair Group Method with Arithmetic Mean (UPGMA).

    Similar to NJ, it relies on Biopython's distance matrix calculation and tree construction.
    """
    aln = AlignIO.read(str(aln_fa), "fasta")
    # Use appropriate substitution model for distance calculation.
    model = "identity" if molecule == "dna" else "blosum62"
    dm = DistanceCalculator(model).get_distance(aln)
    tree = DistanceTreeConstructor().upgma(dm)
    Phylo.write(tree, str(out_newick), "newick")

def infer_ml_iqtree(aln_fa: Path, molecule: str, out_newick: Path):
    """
    Infers a Maximum Likelihood (ML) phylogenetic tree using IQ-TREE 2.

    IQ-TREE performs automatic model selection (`-m TEST`) and determines the optimal
    tree topology based on the ML criterion. The output tree file (`.treefile`) is
    copied to the desired Newick path.
    """
    # IQ-TREE sometimes requires a specific file extension, ensuring it's a FASTA file.
    aln_fasta = str(aln_fa) if str(aln_fa).endswith(".fa") else str(aln_fa) + ".fa"
    AlignIO.write(AlignIO.read(str(aln_fa), "fasta"), aln_fasta, "fasta")

    # -m TEST: automatic model selection; -nt AUTO: thread optimization; -quiet: minimal output
    cmd = f"iqtree2 -s {aln_fasta} -m TEST -nt AUTO -quiet"
    run_cmd(cmd)

    iqtree_file = aln_fasta + ".treefile"
    if os.path.exists(iqtree_file):
        # The ML tree inferred by IQ-TREE is saved in a .treefile.
        shutil.copy(iqtree_file, out_newick)

# Dictionary mapping inference method names to their corresponding functions.
INFER_FUNCS = {"nj": infer_nj, "upgma": infer_upgma, "ml": infer_ml_iqtree}

# ------------------------------
# Visualization
# ------------------------------
def render_tree_png(newick: Path, out_png: Path, title="Tree"):
    """
    Renders a single phylogenetic tree to a PNG file using Biopython's Phylo.draw
    functionality backed by Matplotlib.

    :param newick: Path to the input tree file in Newick format.
    :param out_png: Path to save the output PNG image.
    :param title: Title for the rendered plot.
    """
    tree = Phylo.read(str(newick), "newick")
    fig = plt.figure(figsize=(6, 6))
    ax = fig.add_subplot(1, 1, 1)
    Phylo.draw(tree, do_show=False, axes=ax)
    ax.set_title(title)
    plt.savefig(str(out_png))
    plt.close(fig)

def render_tree_side_by_side(tree1_file, tree2_file, out_png):
    """
    Renders two phylogenetic trees side-by-side on a single PNG for direct visual comparison.

    :param tree1_file: Path to the first Newick tree file.
    :param tree2_file: Path to the second Newick tree file.
    :param out_png: Path to save the output comparison PNG image.
    """
    tree1 = Phylo.read(str(tree1_file), "newick")
    tree2 = Phylo.read(str(tree2_file), "newick")
    fig, axes = plt.subplots(1, 2, figsize=(12, 6))

    Phylo.draw(tree1, do_show=False, axes=axes[0])
    axes[0].set_title("Tree 1")

    Phylo.draw(tree2, do_show=False, axes=axes[1])
    axes[1].set_title("Tree 2")

    plt.savefig(out_png)
    plt.close()

# ------------------------------
# Likelihood model-fit tests
# ------------------------------
def run_likelihood_tests(aln_fa: Path, tree_files: list[Path]) -> dict:
    """
    Executes the IQ-TREE likelihood-based statistical tests (SH, KH, AU) to compare
    a set of trees against the sequence alignment data.

    The tests assess which tree topology best explains the observed sequence data.

    :param aln_fa: Path to the multiple sequence alignment FASTA file.
    :param tree_files: List of paths to Newick tree files to be tested.
    :return: A dictionary containing the parsed test results, keyed by tree ID (T1, T2, etc.).
    """
    if not shutil.which("iqtree2"):
        return {"error": "iqtree2 not installed"}

    # Concatenate all trees into a single file for IQ-TREE input (-z option).
    all_trees = aln_fa.with_suffix(".treeset.nwk")
    with open(all_trees, "w") as out:
        for t in tree_files:
            with open(t) as f:
                out.write(f.read().strip() + "\n")

    # -z: set of trees to test; -zb 1000: bootstrap replicates (for AU test); -au: enables AU test.
    cmd = f"iqtree2 -s {aln_fa} -z {all_trees} -zb 1000 -au -nt AUTO -quiet"
    run_cmd(cmd)

    # Parse the main IQ-TREE output file (.iqtree) for test results.
    iqtree_out = Path(str(aln_fa) + ".iqtree")
    results = {}
    if os.path.exists(iqtree_out):
        with open(iqtree_out) as f:
            lines = [l.strip() for l in f if l.strip()]
        header = []
        for line in lines:
            if line.startswith("Tree"):
                # Identify the header containing 'p-AU', 'p-KH', 'p-SH'
                header = line.split()
                continue
            if header and line[0].isdigit():
                # Data lines follow the header structure (Tree ID, logL, deltaL, p-AU, etc.)
                parts = line.split()
                tree_id = parts[0]
                results[tree_id] = dict(zip(header[1:], parts[1:]))
    return results

# ------------------------------
# Analysis & Summary
# ------------------------------
def analyze_results(table_data, aln_likelihoods, artifacts, aln_files, aln_to_trees, outdir):
    """
    Performs a high-level analysis of the tree comparison and likelihood test results.

    This function identifies the statistically best-fit tree (highest AU p-value) and
    the most topologically similar pair (lowest Normalized RF distance), then generates
    a comprehensive, dynamic summary report.
    """
    print("\n\n📊 Result Analysis and Conclusion 📊")
    print("-----------------------------------")
    # ... (rest of the print statements for analysis and summary) ...
    print("\n### 1. Explanation of the Table")
    print("---------------------------------")
    print("The table compares all generated phylogenetic trees using a combination of methods (e.g., MAFFT+NJ, MUSCLE+ML).")
    print("- **RF (Robinson-Foulds) Distance**: A measure of the topological difference between two trees. A lower value means the trees are more similar.")
    print("- **Normalized RF**: The RF distance scaled to a 0-1 range. Lower is better.")
    print("- **BranchLenCorr**: The correlation coefficient of branch lengths. A value closer to 1 means branch lengths are highly similar.")
    print("- **Support Values**: Indicates the number of internal branches with bootstrap support values. A higher number suggests more confidence in the tree's internal structure.")
    print("- **Likelihood Tests (AU/KH/SH)**: Statistical tests from IQ-TREE to evaluate how well each tree fits the sequence data. For each alignment, the tree with the highest AU p-value (closest to 1) is the statistically best-fit tree. A p-value < 0.05 indicates the tree can be rejected.")

    print("\n### 2. Identifying the Best Tree(s)")
    print("----------------------------------")
    best_likelihood_tree_path = "N/A"
    best_likelihood_value = -1.0
    best_likelihood_method = "N/A"
    best_topology_pair = "N/A"
    lowest_norm_rf = float('inf')
    best_pair_img = "N/A"

    # Find the best likelihood tree and its method by iterating through all AU p-values.
    for aln_name, res in aln_likelihoods.items():
        if "error" in res:
            continue

        # Link the internal IQ-TREE Tree ID (T1, T2, etc.) to the method (MSA+Inference).
        tree_methods_for_aln = []
        for m, inf, tfile in artifacts:
            if m == aln_name:
                tree_methods_for_aln.append((m, inf))

        for tid, vals in res.items():
            if 'p-AU' in vals and vals['p-AU'] != '?':
                au_val = float(vals['p-AU'])
                if au_val > best_likelihood_value:
                    best_likelihood_value = au_val
                    method_index = int(tid) - 1
                    if method_index < len(tree_methods_for_aln):
                        m, inf = tree_methods_for_aln[method_index]
                        best_likelihood_method = f"{m}+{inf}"
                        best_likelihood_tree_path = outdir / f"{m}_{inf}.png"

    # Find the best topological similarity (lowest Normalized RF) and its image file.
    for row in table_data:
        norm_rf = float(row[2])
        if norm_rf < lowest_norm_rf:
            lowest_norm_rf = norm_rf
            best_topology_pair = row[0]
            m1, inf1 = best_topology_pair.split(" vs ")[0].split('+')
            m2, inf2 = best_topology_pair.split(" vs ")[1].split('+')
            best_pair_img = outdir / f"{m1}_{inf1}_VS_{m2}_{inf2}.png"

    print(f"The **statistically best-fit tree** is: **{best_likelihood_method}** (AU p-value: {best_likelihood_value:.4f})")
    print("This tree has the highest statistical support for its topology, as determined by the AU test.")
    print(f"\nThe **most topologically similar trees** are: **{best_topology_pair}** (Normalized RF: {lowest_norm_rf:.4f})")

    # ---------------------------------------------
    # Dynamic Summary Generation: Provides context-specific interpretation
    # ---------------------------------------------
    print("\n### 3. Quick Summary and Final Recommendation")
    print("---------------------------------------------")

    # Check if ML trees are consistently the best statistically.
    ml_au_values = []
    for aln_name, res in aln_likelihoods.items():
        if 'error' not in res:
            ml_au_values.extend([float(v.get('p-AU', 0)) for v in res.values() if 'ml' in best_likelihood_method])
    is_ml_best = all(val > 0.5 for val in ml_au_values) if ml_au_values else False

    # Check for similarity among distance methods (NJ/UPGMA).
    nj_upgma_pairs = [row for row in table_data if ('nj' in row[0] or 'upgma' in row[0]) and 'ml' not in row[0]]
    distance_methods_are_similar = any(float(row[2]) < 0.5 for row in nj_upgma_pairs)

    # Check for consistency among ML trees generated from different alignments (MSA tools).
    ml_ml_pairs = [row for row in table_data if 'ml' in row[0] and len(row[0].split('ml')) > 2]
    ml_trees_are_consistent = all(float(row[3]) > 0.5 for row in ml_ml_pairs if row[3] != 'N/A')

    if is_ml_best:
        print("1. **Maximum Likelihood (ML)** trees consistently show the highest statistical support (AU p-values closest to 1). This confirms that ML is the most suitable inference method for this dataset, as it finds the most optimal topology to explain the evolutionary relationships.")

    if distance_methods_are_similar:
        print("2. The **Distance-based methods (NJ and UPGMA)** produce relatively similar trees to each other (low RF distances), but they are often statistically rejected by the likelihood tests (AU p-values near 0), indicating their topologies are less likely to be correct.")

    if ml_trees_are_consistent:
        print("3. When using the ML method, all tested alignment tools (**MAFFT**, **MUSCLE**, and **ClustalW**) produce trees with a high degree of consistency, as shown by their high branch length correlation. This suggests that for this dataset, the choice of alignment tool has a minimal impact on the final ML tree topology.")

    print(f"\n**Conclusion**: For generating the most accurate and statistically supported phylogenetic tree from this dataset, the best approach is to use the **Maximum Likelihood (ML) tree inference method**, paired with any of the tested Multiple Sequence Alignment methods.")

    # ---------------------------------------------
    # Image Location
    # ---------------------------------------------
    print("\n### 4. Locate Best Tree Images")
    print("------------------------------------")
    print(f"The best tree image is located at: {best_likelihood_tree_path}")
    print(f"The most topologically similar trees image is located at: {best_pair_img}")
    print(f"\nAll generated files are in the directory: {outdir}")

# ------------------------------
# Comparison helpers
# ------------------------------
def compare_trees(artifacts, outdir):
    """
    Coordinates the pairwise comparison of all generated phylogenetic trees.

    It computes topological similarity (RF distance), branch length correlation,
    and runs the statistical likelihood tests, presenting the results in a formatted table.

    :param artifacts: A list of (MSA_method, Inference_method, Tree_Path) tuples.
    :param outdir: The output directory path.
    """
    print("\n🔎 Tree Comparison Results (pairwise):\n")

    if not HAVE_DENDROPY:
        print("[WARNING] DendroPy not available. Install with: pip install dendropy\n")
        return

    from dendropy.calculate import treecompare
    # Initialize a TaxonNamespace to ensure all trees share the same taxon mapping.
    taxon_namespace = dendropy.TaxonNamespace()

    # Organize trees by their corresponding alignment file for likelihood testing.
    aln_to_trees = {}
    aln_files = {}
    for m, inf, tfile in artifacts:
        aln_file = tfile.parent / f"{m}.aln.fasta"
        aln_to_trees.setdefault(m, []).append(tfile)
        aln_files[m] = aln_file

    # Run IQ-TREE likelihood tests for each set of trees inferred from a single alignment.
    aln_likelihoods = {}
    for aln, tfiles in aln_to_trees.items():
        aln_file = aln_files[aln]
        aln_likelihoods[aln] = run_likelihood_tests(aln_file, tfiles)

    headers = ["Trees Compared", "RF", "Norm RF", "BranchLenCorr", "Support Values", "Likelihood Tests (AU/KH/SH)"]
    table_data = []

    # Iterate over all unique pairs of generated trees.
    for (m1, inf1, t1_file), (m2, inf2, t2_file) in combinations(artifacts, 2):
        # Load trees into DendroPy objects, ensuring consistent taxon labeling.
        dp_tree1 = dendropy.Tree.get(path=str(t1_file), schema="newick",
                                     taxon_namespace=taxon_namespace)
        dp_tree2 = dendropy.Tree.get(path=str(t2_file), schema="newick",
                                     taxon_namespace=taxon_namespace)

        # 1. Calculate Robinson-Foulds (RF) Distance and Normalize.
        rf = treecompare.unweighted_robinson_foulds_distance(dp_tree1, dp_tree2)
        n_leaves = len(dp_tree1.leaf_nodes())
        max_rf = 2 * (n_leaves - 3) if n_leaves > 3 else 1 # Maximum possible RF for unrooted trees.
        norm_rf = rf / max_rf

        # 2. Calculate Branch Length Correlation.
        blens1 = [e.length for e in dp_tree1.edges() if e.length is not None]
        blens2 = [e.length for e in dp_tree2.edges() if e.length is not None]
        corr = "N/A"
        # Pearson correlation requires lists of the same length and SciPy availability.
        if blens1 and blens2 and len(blens1) == len(blens2) and HAVE_SCIPY:
            corr = f"{pearsonr(blens1, blens2)[0]:.3f}"

        # 3. Extract Support Value Information.
        sup1 = [n.label for n in dp_tree1.internal_nodes() if n.label]
        sup2 = [n.label for n in dp_tree2.internal_nodes() if n.label]
        support_info = f"{len(sup1)} vs {len(sup2)}" if sup1 or sup2 else "N/A"

        # 4. Get Likelihood Test Results for the Alignment.
        # Likelihood results apply to the *alignment* and list all trees inferred from it.
        aln_name = m1 # Use the MSA method name as the key for the alignment.
        like = "N/A"
        if aln_name in aln_likelihoods:
            res = aln_likelihoods[aln_name]
            if "error" in res:
                like = res["error"]
            else:
                # Format the AU, KH, and SH p-values for all trees from this alignment.
                like = " / ".join(
                    f"T{tid}:AU={vals.get('p-AU','?')},KH={vals.get('p-KH','?')},SH={vals.get('p-SH','?')}"
                    for tid, vals in res.items()
                )

        table_data.append([
            f"{m1}+{inf1} vs {m2}+{inf2}",
            str(rf),
            f"{norm_rf:.3f}",
            str(corr),
            support_info,
            like
        ])

        # Render the side-by-side comparison image for the pair.
        render_tree_side_by_side(t1_file, t2_file, outdir / f"{m1}_{inf1}_VS_{m2}_{inf2}.png")

    # Print the formatted table output.
    print("---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------")
    print(f"{headers[0]:<40} | {headers[1]:<5} | {headers[2]:<10} | {headers[3]:<15} | {headers[4]:<18} | {headers[5]:<65}")
    print("---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------")
    for row in table_data:
        print(f"{row[0]:<40} | {row[1]:<5} | {row[2]:<10} | {row[3]:<15} | {row[4]:<18} | {row[5]:<65}")
    print("---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------")

    # Proceed to final analysis and summary generation.
    analyze_results(table_data, aln_likelihoods, artifacts, aln_files, aln_to_trees, outdir)

# ------------------------------
# Pipeline
# ------------------------------
def pipeline(fasta: Path, molecule: str, outdir: Path,
             msa_methods, infer_methods, build_all=True, render=True):
    """
    The main execution flow of the phylogenetic pipeline.

    It sequentially performs MSA and tree inference for all combinations of selected
    MSA and inference methods.

    :param fasta: Path to the input unaligned FASTA file.
    :param molecule: The sequence type ('dna' or 'protein').
    :param outdir: The output directory for all artifacts.
    :param msa_methods: List of MSA methods to run.
    :param infer_methods: List of tree inference methods to run.
    :param build_all: Boolean flag to run all 3x3 combinations (MSA x Inference).
    :param render: Boolean flag to render PNG images of the trees.
    :return: List of artifacts (MSA_method, Inference_method, Tree_Path) generated.
    """
    outdir.mkdir(parents=True, exist_ok=True)

    # Determine the full grid of methods to run based on the 'build_all' flag.
    msa_grid = list(MSA_FUNCS.keys()) if build_all else msa_methods
    infer_grid = list(INFER_FUNCS.keys()) if build_all else infer_methods

    artifacts = []
    # Loop 1: Iterate over selected Multiple Sequence Alignment (MSA) methods.
    for m in msa_grid:
        aln_file = outdir / f"{m}.aln.fasta"
        MSA_FUNCS[m](fasta, aln_file)
        print(f"Alignment done: {m} -> {aln_file}")

        # Loop 2: Iterate over selected Phylogenetic Tree Inference methods.
        for inf in infer_grid:
            tree_file = outdir / f"{m}_{inf}.nwk"
            INFER_FUNCS[inf](aln_file, molecule, tree_file)
            print(f"Tree done: {m}+{inf} -> {tree_file}")

            if render:
                render_tree_png(tree_file, outdir / f"{m}_{inf}.png", title=f"{m}+{inf}")

            # Store the resulting tree path and the methods used to create it.
            artifacts.append((m, inf, tree_file))
    return artifacts

# ------------------------------
# Interactive main
# ------------------------------
def main():
    """
    Initializes the pipeline in interactive mode, prompting the user for input
    parameters (FASTA file path, method choices, etc.) and orchestrating the workflow.
    """
    print("\n🌱 Phylogenetic Pipeline (Interactive Mode)\n")
    fasta_path = Path(input("Enter path to FASTA file: ").strip())
    molecule = input("Sequence type (dna/protein) [auto-detect if blank]: ").strip()

    # Auto-detect molecule type if not provided by the user.
    if not molecule:
        molecule = guess_molecule_from_fasta(fasta_path)
    print(f"Detected molecule type: {molecule}")

    # Interactive input for method selection.
    msa_choice = input("MSA method (mafft, muscle, clustalw, all): ").strip().lower() or "mafft"
    infer_choice = input("Inference (nj, upgma, ml, all): ").strip().lower() or "ml"
    build_all = input("Run full 3x3 grid? (y/n): ").strip().lower() == "y"
    render = input("Render PNG trees? (y/n): ").strip().lower() == "y"

    # Create a unique output directory for the run.
    outdir = Path(f"phylo_run_{timestamp()}")

    # Execute the core pipeline.
    artifacts = pipeline(
        fasta_path, molecule, outdir,
        list(MSA_FUNCS.keys()) if msa_choice == "all" else [msa_choice],
        list(INFER_FUNCS.keys()) if infer_choice == "all" else [infer_choice],
        build_all=build_all, render=render
    )

    print("\n✅ Pipeline finished! Results in:", outdir)
    # Run the tree comparison and statistical analysis upon user request.
    if input("Compare all generated trees? (y/n): ").strip().lower() == "y":
        compare_trees(artifacts, outdir)

if __name__ == "__main__":
    main()

Overwriting phylo_pipeline1.py
